# Data Loaders
> Unified data loading pipelines integrating fastai DataBlocks with MONAI Datasets.

The `bonsai.data.load` module provides a highly flexible, registry-based architecture for building complex biomedical imaging pipelines. It seamlessly bridges the gap between MONAI's powerful, memory-efficient datasets (like `CacheDataset` or `PersistentDataset`) and fastai's intuitive training ecosystem (`DataLoaders`, `DataBlock`), offering automated source detection, prefix-based configuration routing, and smart dataset splitting.

In [ ]:
#| default_exp data.load

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *
import sys
import tempfile

from unittest.mock import patch as mockpatch, MagicMock
from torch.utils.data import TensorDataset
from torch import zeros as torchzeros, ones as torchones, Tensor as torchTensor, long as torchlong 
from types import SimpleNamespace



In [ ]:
#| export

# =================================
# Standard library
# =================================
import os
import types
from inspect import signature, _empty
from pathlib import Path
import pandas as pd
import numpy as np
import re
# import matplotlib.pyplot as plt

# for pipeline context
from dataclasses import dataclass, field
from typing import Any, Optional

# =================================
# PyTorch
# =================================
from torch.utils.data import (
    DataLoader as torchDataLoader, 
    Dataset as torchDataset, 
    default_collate as torch_default_collate
)

# =================================
# fastai
# =================================
from fastai.data.all import (
    DataLoaders, delegates, RegexLabeller, is_listy,
    ColReader, CategoryMap
)

from fastai.vision.all import (
    DataBlock, CategoryBlock, MultiCategoryBlock, RegressionBlock,
    TfmdDL, TransformBlock, Pipeline,
    get_image_files, 
    parent_label, 
    partial, show_results, store_attr
)

# =================================
# MONAI
# =================================
from monai.data import (
    Dataset as MonaiDataset, CacheDataset, 
    PersistentDataset, SmartCacheDataset,
    DataLoader as MonaiDataLoader
)
# from monai.data.utils import pickle_hashing
from monai.transforms import Compose
from monai.transforms.transform import Randomizable

# =================================
# bonsai
# =================================
from bonsai.utils import *
from bonsai.data.core import *
from bonsai.backend import get_backend, set_backend
# from bonsai.transforms import RandMonaiTransform, RandTransform

# =================================
# fasttransform patch
# =================================
import fasttransform

_original_repr = fasttransform.transform.Transform.__repr__

def _safe_repr(self):
    try:
        return _original_repr(self)
    except AttributeError as e:
        if "'_BoundFunction' object has no attribute 'methods'" in str(e):
            return f"{self.__class__.__name__}(...)"
        raise

fasttransform.transform.Transform.__repr__ = _safe_repr

# =================================
# MonaiDataLoader patch
# =================================
import inspect

MonaiDataLoader.__init__.__signature__ = inspect.signature(
    torchDataLoader.__init__
)

## BioDataBlock
> `BioDataBlock` is built on top of fastai's `DataBlock`, serving as a tailored, high-level container used to build datasets and dataloaders efficiently for biomedical data. 

* **Seamless Integration:** Integrates item and batch transformations, specialized input getters, and splitting logic into a single cohesive pipeline.
* **Biomedical Defaults:** Defaults to using `BioImage` blocks, streamlining the setup of data pipelines for training and validation in medical imaging tasks.
* **Flexibility:** Retains all the powerful customization capabilities of the standard fastai `DataBlock` while providing a familiar interface for bonsai workflows.

In [ ]:
#| export
class BioDataBlock(DataBlock):
    """ 
    The `BioDataBlock` class serves as a generic container to build `Datasets` and `DataLoaders` efficiently. It integrates item and batch transformations, getters, and splitters, simplifying the setup of data pipelines for training and validation.
    """
    def __init__(self, 
            blocks:list=(BioImage.get_datablock(), BioImage.get_datablock()),       # One or more `TransformBlock`s
            dl_type:TfmdDL=None,                                                    # Task specific `TfmdDL`, defaults to `block`'s dl_type or`TfmdDL`
            get_items=get_image_files,
            get_y=None,
            get_x=None,
            getters:list=None,                                                      # Getter functions applied to results of `get_items`
            n_inp:int=None,                                                         # Number of inputs
            item_tfms:list=None,                                                    # `ItemTransform`s, applied on an item 
            batch_tfms:list=None,                                                   # `Transform`s or `RandTransform`s, applied by batch
            splitter=None, 
        ):
        super().__init__(
            blocks=blocks, 
            dl_type=dl_type, 
            get_items=get_items,
            get_y=get_y,
            get_x=get_x,
            getters=getters, 
            n_inp=n_inp, 
            item_tfms=item_tfms, 
            batch_tfms=batch_tfms,
            splitter=splitter,
            )
        

In [ ]:
show_doc(BioDataBlock)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L106){target="_blank" style="float:right; font-size:smaller"}

### BioDataBlock

```python

def BioDataBlock(
    blocks:list=(<fastai.data.block.TransformBlock object>, <fastai.data.block.TransformBlock object>), # One or more `TransformBlock`s
    dl_type:TfmdDL=None, # Task specific `TfmdDL`, defaults to `block`'s dl_type or`TfmdDL`
    get_items:function=get_image_files, get_y:NoneType=None, get_x:NoneType=None,
    getters:list=None, # Getter functions applied to results of `get_items`
    n_inp:int=None, # Number of inputs
    item_tfms:list=None, # `ItemTransform`s, applied on an item
    batch_tfms:list=None, # `Transform`s or `RandTransform`s, applied by batch
    splitter:NoneType=None
):


```

*The `BioDataBlock` class serves as a generic container to build `Datasets` and `DataLoaders` efficiently. It integrates item and batch transformations, getters, and splitters, simplifying the setup of data pipelines for training and validation.*

In [ ]:
# Example: Initializing a BioDataBlock
# Creates a preconfigured dataset builder tailored for biomedical imaging
bio_block = BioDataBlock(n_inp=1)

print(f"Number of inputs: {bio_block.n_inp}")
print(f"Default blocks assigned: {len(bio_block.blocks)}")

Number of inputs: 1
Default blocks assigned: 2


In [ ]:
#| hide

def test_biodatablock():
    # Init
    builder = BioDataBlock(n_inp=1)
    test_eq(isinstance(builder, DataBlock), True)
    test_eq(builder.n_inp, 1)

    # Verify default blocks
    default_builder = BioDataBlock()
    test_eq(len(default_builder.blocks), 2)
    
    return "BioDataBlock tests passed"

test_eq(test_biodatablock(), "BioDataBlock tests passed")

#### `BioDataBlock` Class Attributes & Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`blocks`** | `list` | `(BioImage.get_datablock(), ...)` | One or more `TransformBlock`s defining the types of your inputs and targets. |
| **`dl_type`** | `TfmdDL` | `None` | Task-specific dataloader type. Defaults to the block's `dl_type` or `TfmdDL`. |
| **`get_items`** | `callable` | `get_image_files` | Function to gather items (e.g., file paths) from the source. |
| **`get_x`** | `callable` | `None` | Getter function applied to the results of `get_items` to extract inputs. |
| **`get_y`** | `callable` | `None` | Getter function applied to the results of `get_items` to extract targets. |
| **`getters`** | `list` | `None` | List of getter functions if using multiple inputs/targets. |
| **`n_inp`** | `int` | `None` | Explicit number of inputs. |
| **`item_tfms`** | `list` | `None` | `ItemTransform`s applied on individual items before batching. |
| **`batch_tfms`** | `list` | `None` | `Transform`s applied on batched tensors (usually on the GPU). |
| **`splitter`** | `callable` | `None` | Function to split items into training and validation sets. |

## BioDataloaders: unified method

The module offers classes to construct data loaders for `fastTrainer`, seamlessly supporting different dataset backends.

### Component Registries
> Dynamic registries allow you to easily extend the pipeline by adding new custom components without modifying the core library code.

* **Extensibility:** Easily add custom data sources, specialized datasets, loaders, or task handlers.
* **Decorator-Based:** Utilizes simple decorators to cleanly inject user-defined classes into the global configuration ecosystem.
* **Backend Tracking:** The dataset decorator specifically tracks the underlying backend (like `"pytorch"` or `"monai"`) to ensure compatibility during pipeline execution.

Included Registries & Methods:
* **`SOURCE_REGISTRY` & `register_source`**: Global registry and decorator for injecting custom data sources.
* **`DATASET_REGISTRY` & `register_dataset`**: Global registry and decorator for handling datasets and their specific backends.
* **`LOADER_REGISTRY` & `register_loader`**: Global registry and decorator for custom dataloaders.
* **`TASK_REGISTRY` & `register_task`**: Global registry and decorator for specialized task handlers.

In [ ]:
#| export
SOURCE_REGISTRY = {}
DATASET_REGISTRY = {}
LOADER_REGISTRY = {}
TASK_REGISTRY = {}

def register_source(name):
    def wrapper(cls):
        SOURCE_REGISTRY[name] = cls
        return cls
    return wrapper


def register_dataset(name, dataset_backend):
    def wrapper(cls):
        DATASET_REGISTRY[name] = (cls, dataset_backend)
        return cls
    return wrapper


def register_loader(name):
    def wrapper(cls):
        LOADER_REGISTRY[name] = cls
        return cls
    return wrapper

def register_task(name):
    def wrapper(cls):
        TASK_REGISTRY[name] = cls
        return cls
    return wrapper

In [ ]:
# Example: Dynamically registering custom components

@register_source("my_custom_source")
class MyCustomSource:
    pass

@register_dataset("my_custom_dataset", dataset_backend="pytorch")
class MyCustomDataset:
    pass

# Verify components are in the global registries
print(f"Available sources: {list(SOURCE_REGISTRY.keys())}")
print(f"Available datasets: {list(DATASET_REGISTRY.keys())}")

Available sources: ['my_custom_source']
Available datasets: ['my_custom_dataset']


In [ ]:
#| hide

def test_registries():
    # Register dummy components
    @register_source("dummy_src")
    class DummySrc: pass

    @register_dataset("dummy_ds", dataset_backend="dummy")
    class DummyDs: pass

    @register_loader("dummy_ld")
    class DummyLd: pass

    @register_task("dummy_tk")
    class DummyTk: pass

    # Verify registration
    test_eq(SOURCE_REGISTRY["dummy_src"], DummySrc)
    test_eq(DATASET_REGISTRY["dummy_ds"], (DummyDs, "dummy"))
    test_eq(LOADER_REGISTRY["dummy_ld"], DummyLd)
    test_eq(TASK_REGISTRY["dummy_tk"], DummyTk)

    # Cleanup
    del SOURCE_REGISTRY["dummy_src"]
    del DATASET_REGISTRY["dummy_ds"]
    del LOADER_REGISTRY["dummy_ld"]
    del TASK_REGISTRY["dummy_tk"]

    if "my_custom_source" in SOURCE_REGISTRY:
        del SOURCE_REGISTRY["my_custom_source"]
    if "my_custom_dataset" in DATASET_REGISTRY:
        del DATASET_REGISTRY["my_custom_dataset"]
        
    return "Registry tests passed"

test_eq(test_registries(), "Registry tests passed")

#### `register_source` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`name`** | `str` | *Required* | The string identifier used to register the custom data source in `SOURCE_REGISTRY`. |

#### `register_dataset` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`name`** | `str` | *Required* | The string identifier used to register the custom dataset in `DATASET_REGISTRY`. |
| **`dataset_backend`** | `str` | *Required* | The underlying backend framework (e.g., `"pytorch"`, `"monai"`) associated with the dataset. |

#### `register_loader` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`name`** | `str` | *Required* | The string identifier used to register the custom dataloader in `LOADER_REGISTRY`. |

#### `register_task` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`name`** | `str` | *Required* | The string identifier used to register the custom task handler in `TASK_REGISTRY`. |

### Utility Functions
> This section provides specialized helpers to bridge configurations and data formats seamlessly across the pipeline.

* **Configuration Management:** `split_prefixed_kwargs` cleanly separates a single dictionary of keyword arguments into multiple configuration groups based on string prefixes (e.g., `train_`, `val_`). Non-prefixed arguments automatically serve as fallback defaults across all groups.
* **Dataset Bridging:** `TupleDataset` acts as a crucial bridge between MONAI's dictionary-based datasets and standard PyTorch/fastai pipelines, wrapping a dictionary dataset and flattening specified input (`x_keys`) and target (`y_keys`) dictionaries into a single ordered tuple upon item retrieval.

Included Classes & Methods:
* **`split_prefixed_kwargs`** is a utility designed to cleanly separate a single dictionary of keyword arguments into multiple configuration groups based on string prefixes (such as `train_` and `val_`). Arguments with a matching prefix are routed to their respective group, while non-prefixed arguments are automatically applied to all groups as fallback defaults. This is highly useful for managing distinct parameters for Training and Validation DataLoaders from a single unified config.

* **`TupleDataset`** acts as a crucial bridge between MONAI's dictionary-based datasets and standard PyTorch/fastai pipelines, which expect tuple-based inputs. It wraps a dictionary dataset and flattens the specified input (`x_keys`) and target (`y_keys`) dictionaries into a single, ordered tuple upon item retrieval.

In [ ]:
#| export
def split_prefixed_kwargs(kwargs, prefixes=("train_", "val_")):
    """
    Split a dictionary of kwargs into multiple groups based on prefixes.

    Example:
        kwargs = {
            "batch_size": 32,
            "train_cache_rate": 1.0,
            "val_cache_rate": 0.5
        }

        split_prefixed_kwargs(kwargs)
        -> {
             "train": {"cache_rate": 1.0, "batch_size": 32},
             "val": {"cache_rate": 0.5, "batch_size": 32}
           }

    Rules:
    - Keys starting with prefix go to that group (prefix removed)
    - All keys also appear in each group as defaults if no prefix exists
    """
    groups = {p.rstrip("_"): {} for p in prefixes}

    for k, v in kwargs.items():
        matched = False
        for p in prefixes:
            if k.startswith(p):
                groups[p.rstrip("_")][k[len(p):]] = v
                matched = True
                break
        if not matched:
            # No prefix → default to all groups
            for g in groups:
                groups[g][k] = v

    return groups

In [ ]:
show_doc(split_prefixed_kwargs)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L169){target="_blank" style="float:right; font-size:smaller"}

### split_prefixed_kwargs

```python

def split_prefixed_kwargs(
    kwargs, prefixes:tuple=('train_', 'val_')
):


```

*Split a dictionary of kwargs into multiple groups based on prefixes.*

Example:
    kwargs = {
        "batch_size": 32,
        "train_cache_rate": 1.0,
        "val_cache_rate": 0.5
    }

    split_prefixed_kwargs(kwargs)
    -> {
         "train": {"cache_rate": 1.0, "batch_size": 32},
         "val": {"cache_rate": 0.5, "batch_size": 32}
       }

Rules:
- Keys starting with prefix go to that group (prefix removed)
- All keys also appear in each group as defaults if no prefix exists

#### Example: Splitting mixed keyword arguments into specific groups


In [ ]:
sample_kwargs = {
    "batch_size": 32,
    "train_cache_rate": 1.0,
    "val_cache_rate": 0.5,
    "shuffle": True,
    "val_shuffle": False
}

# Split based on prefixes
split_results = split_prefixed_kwargs(sample_kwargs, prefixes=("train_", "val_"))

print("Train parameters:", split_results["train"])
print("Val parameters:  ", split_results["val"])

Train parameters: {'batch_size': 32, 'cache_rate': 1.0, 'shuffle': True}
Val parameters:   {'batch_size': 32, 'cache_rate': 0.5, 'shuffle': False}


In [ ]:
#| hide

def test_split_prefixed_kwargs():
    test_kwargs = {
        "batch_size": 32,
        "train_cache_rate": 1.0,
        "val_cache_rate": 0.5,
        "shuffle": True,
        "val_shuffle": False
    }

    # Split
    res = split_prefixed_kwargs(test_kwargs, prefixes=("train_", "val_"))

    # Verify train group
    test_eq(res["train"], {"batch_size": 32, "shuffle": True, "cache_rate": 1.0})
    
    # Verify val group
    test_eq(res["val"], {"batch_size": 32, "shuffle": False, "cache_rate": 0.5})

    return "split_prefixed_kwargs tests passed"

test_eq(test_split_prefixed_kwargs(), "split_prefixed_kwargs tests passed")

#### `split_prefixed_kwargs` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`kwargs`** | `dict` | *Required* | The dictionary of keyword arguments to split. |
| **`prefixes`** | `tuple` | `("train_", "val_")` | A tuple of string prefixes used to route arguments to their respective configuration groups. |

In [ ]:
#| export
class TupleDataset(torchDataset):
    def __init__(self, ds, x_keys="image", y_keys="label"):
        """
        ds: MONAI dataset (or any dict-like dataset)
        x_keys: single key (str) or list/tuple of keys for inputs
        y_keys: single key (str) or list/tuple of keys for outputs
        """
        self.ds = ds
        # Normalize to lists
        self.x_keys = [x_keys] if isinstance(x_keys, str) else list(x_keys)
        self.y_keys = [y_keys] if isinstance(y_keys, str) else list(y_keys)

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        # Flatten all keys into a single tuple
        output = tuple(item[k] for k in self.x_keys + self.y_keys)
        return output

    def __getattr__(self, name):
        # Forward any unknown attribute to the underlying dataset
        return getattr(self.ds, name)

In [ ]:
show_doc(TupleDataset)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L207){target="_blank" style="float:right; font-size:smaller"}

### TupleDataset

```python

def TupleDataset(
    ds, x_keys:str='image', y_keys:str='label'
):


```

*An abstract class representing a :class:`Dataset`.*

All datasets that represent a map from keys to data samples should subclass
it. All subclasses should overwrite :meth:`__getitem__`, supporting fetching a
data sample for a given key. Subclasses could also optionally overwrite
:meth:`__len__`, which is expected to return the size of the dataset by many
:class:`~torch.utils.data.Sampler` implementations and the default options
of :class:`~torch.utils.data.DataLoader`. Subclasses could also
optionally implement :meth:`__getitems__`, for speedup batched samples
loading. This method accepts list of indices of samples of batch and returns
list of samples.

.. note::
  :class:`~torch.utils.data.DataLoader` by default constructs an index
  sampler that yields integral indices.  To make it work with a map-style
  dataset with non-integral indices/keys, a custom sampler must be provided.

#### Example: Converting a dictionary dataset into a tuple-based dataset


In [ ]:
# 1. Mock dictionary dataset
mock_dict_dataset = [
    {"image": "scan_01.nii.gz", "mask": "seg_01.nii.gz", "meta": "patient_A"},
    {"image": "scan_02.nii.gz", "mask": "seg_02.nii.gz", "meta": "patient_B"}
]

# 2. Wrap to flatten specific keys into a tuple
tuple_dataset = TupleDataset(
    ds=mock_dict_dataset, 
    x_keys=["image", "meta"], 
    y_keys="mask"
)

# 3. Retrieve item
print(f"Original: {mock_dict_dataset[0]}")
print(f"Wrapped:  {tuple_dataset[0]}")

Original: {'image': 'scan_01.nii.gz', 'mask': 'seg_01.nii.gz', 'meta': 'patient_A'}
Wrapped:  ('scan_01.nii.gz', 'patient_A', 'seg_01.nii.gz')


In [ ]:
#| hide

def test_TupleDataset():
    # Dummy dict dataset
    class DummyDS:
        def __init__(self):
            self.data = [{"img": "i1.png", "lbl": 0, "ext": "a"}, {"img": "i2.png", "lbl": 1, "ext": "b"}]
            self.attr = "ok"
        def __len__(self): return len(self.data)
        def __getitem__(self, i): return self.data[i]

    dummy_ds = DummyDS()

    # Single keys
    ds_single = TupleDataset(dummy_ds, x_keys="img", y_keys="lbl")
    test_eq(len(ds_single), 2)
    test_eq(ds_single[0], ("i1.png", 0))

    # Multiple keys
    ds_multi = TupleDataset(dummy_ds, x_keys=["img", "ext"], y_keys=["lbl"])
    test_eq(ds_multi[1], ("i2.png", "b", 1))

    # Delegate attributes
    test_eq(ds_single.attr, "ok")

    return "TupleDataset tests passed"

test_eq(test_TupleDataset(), "TupleDataset tests passed")

#### `TupleDataset` Class Attributes & Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`ds`** | `Dataset` | *Required* | The underlying dictionary-based dataset (e.g., a MONAI dataset). |
| **`x_keys`** | `str \| list \| tuple` | `"image"` | The key or list of keys to extract as inputs (independent variables). |
| **`y_keys`** | `str \| list \| tuple` | `"label"` | The key or list of keys to extract as targets (dependent variables). |

In [ ]:
#| export
def _patch_dataset(ds, **attrs):
    """
    Patch a dataset instance with arbitrary attributes.

    This allows adding fastai-style attributes (like `vocab`) or other
    metadata to a dataset instance without modifying the class.

    Parameters
    ----------
    ds : Dataset
        Dataset instance to patch.
    **attrs : dict
        Arbitrary attributes to attach to the dataset.

    Returns
    -------
    Dataset
        The patched dataset.
    """
    for k, v in attrs.items():
        setattr(ds, k, v)
    return ds

In [ ]:
#| hide

def test_patch_dataset():
    class DummyDS: pass
    test_ds = DummyDS()

    # Patch attributes
    patched = _patch_dataset(test_ds, vocab=["A", "B"], num_classes=2, is_patched=True)

    # Verify
    test_eq(patched.vocab, ["A", "B"])
    test_eq(patched.num_classes, 2)
    test_eq(patched.is_patched, True)

    return "_patch_dataset tests passed"

test_eq(test_patch_dataset(), "_patch_dataset tests passed")

In [ ]:
#| export
def _patch_dataloader(dl):
    """
    Patch a PyTorch DataLoader for minimal fastai compatibility.

    Adds:
    - .one_batch(): returns a single batch
    - .new(**kwargs): clone dataloader with overrides
    - .show_results(): show batch results using BioImage/MetaTensor
    - .show_batch(): show a batch using BioImage/MetaTensor
    - .vocab: inferred from underlying dataset if present
    """

    # ---- helpers ----
    def _attach_method(obj, fn):
        setattr(obj, fn.__name__, types.MethodType(fn, obj))
    
    def _get_batch_items(b, max_n):
        "Return first `max_n` items from batch `b`, handling single tensors."
        # unpack batch
        if isinstance(b, (tuple, list)) and len(b) == 2:
            x, y = b
        else:
            x = b
            y = None

        # ensure x_items is a list
        if isinstance(x, MetaTensor):
            if x.dim() == 0:          # scalar
                x_items = [x]
            elif x.dim() == 3:        # single image (C,H,W)
                x_items = [x]
            else:                     # batched (B,C,H,W)
                x_items = [x[i] for i in range(min(max_n, x.shape[0]))]
        elif isinstance(x, list):
            x_items = x[:max_n]
        else:
            x_items = list(x)[:max_n]

        # same for y
        if y is None:
            y_items = [None]*len(x_items)
        elif isinstance(y, MetaTensor):
            if y.dim() == 0:
                y_items = [y.item()]
            elif y.dim() == 1:
                y_items = [y[i].item() for i in range(min(max_n, len(y)))]
            else:
                y_items = [y[i] for i in range(min(max_n, y.shape[0]))]
        else:
            y_items = list(y)[:len(x_items)]

        return x_items, y_items

    # ---- methods ----

    def do_item(self, i):
        """
        Return a single item from the dataset after minimal processing.
        Mimics fastai DataLoader.do_item behavior.
        """
        item = self.dataset[i]

        # if collate_fn exists, apply it to make batch-like
        if getattr(self, "collate_fn", None) is not None:
            try:
                item = self.collate_fn([item])
            except:
                pass

        return item

    def one_batch(self): return next(iter(self))

    def new(self, **kwargs):
        params = dict(
            dataset=self.dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            collate_fn=self.collate_fn,
            pin_memory=self.pin_memory,
            drop_last=self.drop_last,
        )
        params.update(kwargs)
        new_dl = type(self)(**params)
        _patch_dataloader(new_dl)
        return new_dl

    # alias the typedispatch function
    from bonsai.data import show_batch as _show_batch

    def show_batch(
        self, b=None, max_n=9, ctxs=None, show=True, unique=False, **kwargs
    ):
        "Show `max_n` input(s) and target(s) from the batch."
        if unique:
            old_get_idxs = getattr(self, "get_idxs", lambda: None)
            self.get_idxs = lambda: [0]

        if b is None: b = self.one_batch()
        x_items, y_items = _get_batch_items(b, max_n)

        if show:
            _show_batch(
                x_items,
                y_items,
                samples=None,
                ctxs=ctxs,
                max_n=max_n,
                vocab=getattr(self, "vocab", None),
                **kwargs
            )
        else:
            return x_items, y_items

        if unique: self.get_idxs = old_get_idxs

    # ---- attach methods ----
    for fn in (one_batch, new, do_item, show_results, show_batch):
        _attach_method(dl, fn)

    # ---- attach vocab if available ----
    if hasattr(dl.dataset, "vocab"):
        vocab = dl.dataset.vocab
        if isinstance(vocab, list):
            from fastai.data.transforms import CategoryMap
            vocab = CategoryMap(vocab)
        dl.vocab = vocab

    # ---- attach x/y keys if available ----
    if hasattr(dl.dataset, "x_keys"):
        dl.x_keys = dl.dataset.x_keys

    if hasattr(dl.dataset, "y_keys"):
        dl.y_keys = dl.dataset.y_keys

    return dl

In [ ]:
#| hide

def test_patch_dataloader():
    class DummyDS:
        def __init__(self):
            self.data = [([1, 2], 0), ([3, 4], 1)]
            self.vocab = ["cat", "dog"]
            self.x_keys = ["image"]
            self.y_keys = ["label"]
        def __len__(self): return len(self.data)
        def __getitem__(self, i): return self.data[i]

    dl = torchDataLoader(DummyDS(), batch_size=2)
    patched = _patch_dataloader(dl)

    # Verify methods attached
    test_eq(hasattr(patched, 'one_batch'), True)
    test_eq(hasattr(patched, 'new'), True)
    test_eq(hasattr(patched, 'show_batch'), True)

    # Verify attributes attached
    test_eq(list(patched.vocab), ["cat", "dog"])
    test_eq(patched.x_keys, ["image"])
    
    # Test method functionality
    batch = patched.one_batch()
    test_eq(len(batch), 2)
    
    # Test clone
    cloned = patched.new(batch_size=1)
    test_eq(cloned.batch_size, 1)
    test_eq(hasattr(cloned, 'one_batch'), True)

    return "_patch_dataloader tests passed"

test_eq(test_patch_dataloader(), "_patch_dataloader tests passed")

ImportError: cannot import name 'show_batch' from 'bonsai.data' (/home/bm/Documents/bonsai/bonsai/data/__init__.py)

In [ ]:
# #| export
# def _show_summary(train_dl, val_dl=None):
#     """
#     Display a fastai-like summary of a DataLoader pipeline.

#     Reports:
#       - dataset size and split information
#       - DataLoader configuration
#       - pipeline configuration
#       - x/y item pipelines
#       - item transforms
#       - one transformed sample
#       - collate function
#       - final batch structure
#       - sample metadata, including MONAI MetaTensor metadata

#     The summary is backend-agnostic and supports NumPy arrays,
#     PyTorch tensors and MONAI MetaTensors.
#     """

#     # ------------------------------------------------------------------
#     # Helpers
#     # ------------------------------------------------------------------

#     def _unwrap_dataset(ds):
#         """Unwrap dataset wrappers such as TupleDataset."""
#         while hasattr(ds, "ds"):
#             ds = ds.ds
#         return ds

#     def _type_name(x):
#         if isinstance(x, MetaTensor):
#             return "MetaTensor"
#         if isinstance(x, torchTensor):
#             return "torch.Tensor"
#         if isinstance(x, np.ndarray):
#             return "numpy.ndarray"
#         return type(x).__name__

#     def _shape(x):
#         if hasattr(x, "shape"):
#             return tuple(x.shape)
#         return None

#     def _dtype(x):
#         return getattr(x, "dtype", None)

#     def _memory_mb(x):
#         if isinstance(x, (MetaTensor, torchTensor)):
#             return x.numel() * x.element_size() / (1024 ** 2)

#         if isinstance(x, np.ndarray):
#             return x.nbytes / (1024 ** 2)

#         return None

#     def _describe(x, indent="    "):
#         """
#         Compact representation of an object.

#         Deliberately does not print the contents of large tensors/arrays.
#         """
#         typ = _type_name(x)
#         shape = _shape(x)
#         dtype = _dtype(x)

#         parts = [typ]

#         if shape is not None:
#             parts.append(f"shape={shape}")

#         if dtype is not None:
#             parts.append(f"dtype={dtype}")

#         mem = _memory_mb(x)

#         if mem is not None:
#             parts.append(f"~{mem:.2f} MB")

#         if isinstance(x, MetaTensor):
#             parts.append("metadata=yes")

#         print(indent + " ".join(parts))

#     def _iter_transforms(transform):
#         """Return transforms from MONAI/fastai-like Compose objects."""
#         if transform is None:
#             return []

#         transforms = getattr(transform, "transforms", None)

#         if transforms is not None:
#             return list(transforms)

#         return [transform]

#     def _transform_name(t):
#         """Compact transform name + parameters."""
#         name = type(t).__name__

#         try:
#             text = repr(t)
#             text = text.replace("\n", " ")

#             if len(text) > 180:
#                 text = text[:177] + "..."

#             return f"{name} -- {text}"

#         except Exception:
#             return name

#     def _show_pipeline(pipeline, prefix=""):
#         transforms = _iter_transforms(pipeline)

#         if not transforms:
#             print(prefix + "Pipeline: None")
#             return

#         print(prefix + "Pipeline:")

#         for i, t in enumerate(transforms, 1):
#             print(f"{prefix}  {i}. {_transform_name(t)}")

#     def _get_pipeline(ds, name):
#         """
#         Try the common locations where the item pipeline may live.
#         """
#         pipeline = getattr(ds, name, None)

#         if pipeline is not None:
#             return pipeline

#         inner = getattr(ds, "ds", None)

#         if inner is not None:
#             return getattr(inner, name, None)

#         return None

#     # ------------------------------------------------------------------
#     # Sample metadata
#     # ------------------------------------------------------------------

#     def _find_metadata(x):
#         """
#         Find metadata recursively.

#         MONAI MetaTensors expose metadata through ``.meta``.
#         Dictionary-based datasets may also contain an explicit ``meta``
#         dictionary.
#         """

#         if isinstance(x, MetaTensor):
#             return x.meta

#         if isinstance(x, dict):
#             if "meta" in x and isinstance(x["meta"], dict):
#                 return x["meta"]

#             for v in x.values():
#                 meta = _find_metadata(v)

#                 if meta is not None:
#                     return meta

#         if isinstance(x, (tuple, list)):
#             for v in x:
#                 meta = _find_metadata(v)

#                 if meta is not None:
#                     return meta

#         return None

#     def _format_metadata_value(value):
#         """Compact representation of metadata values."""

#         if isinstance(value, np.ndarray):
#             return (
#                 f"array(shape={value.shape}, "
#                 f"dtype={value.dtype})"
#             )

#         if isinstance(value, torchTensor):
#             return (
#                 f"tensor(shape={tuple(value.shape)}, "
#                 f"dtype={value.dtype})"
#             )

#         if isinstance(value, MetaTensor):
#             return (
#                 f"MetaTensor(shape={tuple(value.shape)}, "
#                 f"dtype={value.dtype})"
#             )

#         return value

#     def _show_sample_metadata(ds):
#         """
#         Display metadata attached to the first dataset sample.

#         This includes metadata stored in MONAI MetaTensors.
#         """

#         source_ds = _unwrap_dataset(ds)

#         try:
#             item = source_ds[0]

#         except Exception:
#             return

#         meta = _find_metadata(item)

#         if meta is None:
#             return

#         print("\nSample metadata:")
#         print("  available : yes")

#         # Show the most useful spatial/image metadata first.
#         fields = (
#             "filename",
#             "filename_or_obj",
#             "original_shape",
#             "spatial_shape",
#             "pixdim",
#             "spacing",
#             "space",
#             "affine",
#             "original_affine",
#         )

#         shown = set()

#         for key in fields:
#             if key not in meta:
#                 continue

#             value = _format_metadata_value(meta[key])

#             print(f"  {key:<15}: {value}")
#             shown.add(key)

#         # Show any additional metadata fields afterwards.
#         remaining = [
#             key for key in meta.keys()
#             if key not in shown
#         ]

#         for key in remaining:
#             value = _format_metadata_value(meta[key])
#             print(f"  {key:<15}: {value}")

#     # ------------------------------------------------------------------
#     # Pipeline configuration
#     # ------------------------------------------------------------------

#     def _show_config(dl):
#         """
#         Display the configuration used to construct the DataLoader.

#         ``dl.config`` contains the pipeline configuration together with
#         high-level information such as task, dataset name, backend and
#         mode.
#         """

#         config = getattr(dl, "config", None)

#         if not config:
#             return

#         print("\nPipeline configuration:")

#         for key, value in config.items():
#             print(f"  {key:<15}: {value}")

#     # ------------------------------------------------------------------
#     # Sample tracing
#     # ------------------------------------------------------------------

#     def _show_sample_pipeline(ds, x_keys, y_keys):
#         """
#         Build and display one sample through the dataset transform.
#         """

#         print("\nBuilding one sample")

#         try:
#             source_ds = _unwrap_dataset(ds)

#             # Raw item, before the MONAI transform pipeline.
#             raw_item = (
#                 source_ds.data[0]
#                 if hasattr(source_ds, "data")
#                 else None
#             )

#             if raw_item is None:
#                 print("  Could not access raw dataset item.")
#                 return

#             print("  starting from")
#             print(f"    {raw_item}")

#             transform = getattr(source_ds, "transform", None)

#             if transform is None:
#                 print("  No item transform to apply")
#                 return

#             transforms = _iter_transforms(transform)

#             current = raw_item

#             for t in transforms:
#                 print(f"\n  applying {_transform_name(t)}")

#                 try:
#                     current = t(current)

#                 except Exception as e:
#                     print(f"    ERROR: {e}")
#                     break

#                 print("    gives")
#                 _describe_sample(current)

#             print("\nFinal sample:")
#             _describe_sample(current)

#         except Exception as e:
#             print(f"  Could not build sample: {e}")

#     def _describe_sample(x, indent="    "):
#         """
#         Describe a sample without printing its full contents.
#         """

#         if isinstance(x, dict):
#             for k, v in x.items():
#                 print(f"{indent}{k}:")
#                 _describe_sample(v, indent + "  ")
#             return

#         if isinstance(x, (tuple, list)):
#             for i, v in enumerate(x):
#                 print(f"{indent}[{i}]:")
#                 _describe_sample(v, indent + "  ")
#             return

#         _describe(x, indent)

#     # ------------------------------------------------------------------
#     # Batch summary
#     # ------------------------------------------------------------------

#     def _show_batch(dl):
#         print("\nBuilding one batch")

#         try:
#             batch = next(iter(dl))

#         except Exception as e:
#             print(f"  Could not build batch: {e}")
#             return

#         collate = getattr(dl, "collate_fn", None)

#         print("\nCollating items in a batch")

#         if collate is None:
#             print("  collate_fn: default")

#         else:
#             print(
#                 f"  collate_fn: "
#                 f"{getattr(collate, '__name__', type(collate).__name__)}"
#             )

#         # --------------------------------------------------------------
#         # Final batch
#         # --------------------------------------------------------------

#         print("\nFinal batch:")

#         if isinstance(batch, (tuple, list)):

#             if len(batch) == 2:
#                 print("  x:")
#                 _describe(batch[0], "    ")

#                 print("  y:")
#                 _describe(batch[1], "    ")

#             else:
#                 for i, item in enumerate(batch):
#                     print(f"  [{i}]:")
#                     _describe(item, "    ")

#         elif isinstance(batch, dict):

#             for k, v in batch.items():
#                 print(f"  {k}:")
#                 _describe(v, "    ")

#         else:
#             _describe(batch, "    ")

#     # ------------------------------------------------------------------
#     # DataLoader
#     # ------------------------------------------------------------------

#     def _describe_dl(dl, name):

#         print(f"\n{name} DataLoader")
#         print("=" * (len(name) + 12))

#         ds = dl.dataset
#         source_ds = _unwrap_dataset(ds)

#         try:
#             ds_len = len(ds)

#         except Exception:
#             ds_len = "unknown"

#         print(f"Dataset size : {ds_len}")
#         print(f"Batch size   : {dl.batch_size}")
#         print(f"Batches      : {len(dl)}")

#         if hasattr(dl, "shuffle"):
#             print(f"Shuffle      : {dl.shuffle}")

#         print(f"Workers      : {dl.num_workers}")

#         # --------------------------------------------------------------
#         # Vocabulary
#         # --------------------------------------------------------------

#         vocab = getattr(dl, "vocab", None)

#         if vocab is not None:
#             print(f"Classes      : {vocab}")

#         # --------------------------------------------------------------
#         # Pipeline configuration
#         # --------------------------------------------------------------

#         _show_config(dl)

#         # --------------------------------------------------------------
#         # Dataset / item pipeline
#         # --------------------------------------------------------------

#         x_keys = getattr(ds, "x_keys", None)
#         y_keys = getattr(ds, "y_keys", None)

#         if x_keys is not None:
#             print(f"x_keys       : {x_keys}")

#         if y_keys is not None:
#             print(f"y_keys       : {y_keys}")

#         transform = getattr(source_ds, "transform", None)

#         if transform is not None:
#             print("\nItem transforms:")
#             _show_pipeline(transform, prefix="  ")

#         # --------------------------------------------------------------
#         # Sample
#         # --------------------------------------------------------------

#         _show_sample_pipeline(ds, x_keys, y_keys)

#         # --------------------------------------------------------------
#         # Collation / batch
#         # --------------------------------------------------------------

#         print("\nBatch transforms:")

#         collate = getattr(dl, "collate_fn", None)

#         if collate is None:
#             print("  collate_fn: default")

#         else:
#             print(
#                 f"  collate_fn: "
#                 f"{getattr(collate, '__name__', type(collate).__name__)}"
#             )

#         # --------------------------------------------------------------
#         # Final batch
#         # --------------------------------------------------------------

#         _show_batch(dl)

#         # --------------------------------------------------------------
#         # Sample metadata
#         # --------------------------------------------------------------

#         _show_sample_metadata(ds)

#     # ------------------------------------------------------------------
#     # Train / validation
#     # ------------------------------------------------------------------

#     _describe_dl(train_dl, "Train")

#     if val_dl is not None:
#         _describe_dl(val_dl, "Valid")

In [ ]:
#| export
# import copy

In [ ]:
#| export
def _show_summary(
    train_dl,
    val_dl=None,
    show_batch=True,
    show_stats=True,
    max_n=4,
    bins=50,
):
    """
    Display a fastai-like summary of a DataLoader pipeline.

    Reports:
      - dataset size and split information
      - DataLoader configuration
      - pipeline configuration
      - x/y item pipelines
      - item transforms
      - one transformed sample
      - collate function
      - final batch structure
      - sample metadata, including MONAI MetaTensor metadata
      - images before and after item transforms
      - intensity statistics and histograms before and after transforms

    Parameters
    ----------
    train_dl : DataLoader
        Training DataLoader.
    val_dl : DataLoader, optional
        Validation DataLoader.
    show_batch : bool, default=True
        Display image batches before and after item transforms.
    show_stats : bool, default=True
        Display numerical statistics and intensity histograms.
    max_n : int, default=4
        Maximum number of samples used when visualizing batches.
    bins : int, default=50
        Number of bins used for intensity histograms.

    Notes
    -----
    ``show_batch`` is not modified. The summary creates a temporary
    DataLoader for the untransformed data when necessary and uses the
    existing ``show_batch`` implementation for visualization.
    """

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------

    def _unwrap_dataset(ds):
        """Unwrap dataset wrappers such as TupleDataset."""
        while hasattr(ds, "ds"):
            ds = ds.ds
        return ds

    def _type_name(x):
        if isinstance(x, MetaTensor):
            return "MetaTensor"
        if isinstance(x, torchTensor):
            return "torch.Tensor"
        if isinstance(x, np.ndarray):
            return "numpy.ndarray"
        return type(x).__name__

    def _shape(x):
        if hasattr(x, "shape"):
            return tuple(x.shape)
        return None

    def _dtype(x):
        return getattr(x, "dtype", None)

    def _memory_mb(x):
        if isinstance(x, (MetaTensor, torchTensor)):
            return x.numel() * x.element_size() / (1024 ** 2)

        if isinstance(x, np.ndarray):
            return x.nbytes / (1024 ** 2)

        return None

    def _describe(x, indent="    "):
        """Compact representation of an object."""

        typ = _type_name(x)
        shape = _shape(x)
        dtype = _dtype(x)

        parts = [typ]

        if shape is not None:
            parts.append(f"shape={shape}")

        if dtype is not None:
            parts.append(f"dtype={dtype}")

        mem = _memory_mb(x)

        if mem is not None:
            parts.append(f"~{mem:.2f} MB")

        if isinstance(x, MetaTensor):
            parts.append("metadata=yes")

        print(indent + " ".join(parts))

    def _iter_transforms(transform):
        """Return transforms from MONAI/fastai-like Compose objects."""

        if transform is None:
            return []

        transforms = getattr(transform, "transforms", None)

        if transforms is not None:
            return list(transforms)

        return [transform]

    def _transform_name(t):
        """Compact transform name + parameters."""

        name = type(t).__name__

        try:
            text = repr(t).replace("\n", " ")

            if len(text) > 180:
                text = text[:177] + "..."

            return f"{name} -- {text}"

        except Exception:
            return name

    def _show_pipeline(pipeline, prefix=""):
        transforms = _iter_transforms(pipeline)

        if not transforms:
            print(prefix + "Pipeline: None")
            return

        print(prefix + "Pipeline:")

        for i, t in enumerate(transforms, 1):
            print(f"{prefix}  {i}. {_transform_name(t)}")

    def _get_pipeline(ds, name):
        """
        Try the common locations where the item pipeline may live.
        """

        pipeline = getattr(ds, name, None)

        if pipeline is not None:
            return pipeline

        inner = getattr(ds, "ds", None)

        if inner is not None:
            return getattr(inner, name, None)

        return None

    # ------------------------------------------------------------------
    # Sample metadata
    # ------------------------------------------------------------------

    def _find_metadata(x):
        """Find metadata recursively."""

        if isinstance(x, MetaTensor):
            return x.meta

        if isinstance(x, dict):
            if "meta" in x and isinstance(x["meta"], dict):
                return x["meta"]

            for v in x.values():
                meta = _find_metadata(v)

                if meta is not None:
                    return meta

        if isinstance(x, (tuple, list)):
            for v in x:
                meta = _find_metadata(v)

                if meta is not None:
                    return meta

        return None

    def _format_metadata_value(value):
        """Compact representation of metadata values."""

        if isinstance(value, np.ndarray):
            return (
                f"array(shape={value.shape}, "
                f"dtype={value.dtype})"
            )

        if isinstance(value, torchTensor):
            return (
                f"tensor(shape={tuple(value.shape)}, "
                f"dtype={value.dtype})"
            )

        if isinstance(value, MetaTensor):
            return (
                f"MetaTensor(shape={tuple(value.shape)}, "
                f"dtype={value.dtype})"
            )

        return value

    def _show_sample_metadata(ds):
        """Display metadata attached to the first dataset sample."""

        source_ds = _unwrap_dataset(ds)

        try:
            item = source_ds[0]
        except Exception:
            return

        meta = _find_metadata(item)

        if meta is None:
            return

        print("\nSample metadata:")
        print("  available : yes")

        fields = (
            "filename",
            "filename_or_obj",
            "original_shape",
            "spatial_shape",
            "pixdim",
            "spacing",
            "space",
            "affine",
            "original_affine",
        )

        shown = set()

        for key in fields:
            if key not in meta:
                continue

            value = _format_metadata_value(meta[key])

            print(f"  {key:<15}: {value}")
            shown.add(key)

        remaining = [
            key for key in meta.keys()
            if key not in shown
        ]

        for key in remaining:
            value = _format_metadata_value(meta[key])
            print(f"  {key:<15}: {value}")

    # ------------------------------------------------------------------
    # Pipeline configuration
    # ------------------------------------------------------------------

    def _show_config(dl):
        """Display the configuration used to construct the DataLoader."""

        config = getattr(dl, "config", None)

        if not config:
            return

        print("\nPipeline configuration:")

        for key, value in config.items():
            print(f"  {key:<15}: {value}")

    # ------------------------------------------------------------------
    # Sample tracing
    # ------------------------------------------------------------------

    def _describe_sample(x, indent="    "):
        """Describe a sample without printing its full contents."""

        if isinstance(x, dict):
            for k, v in x.items():
                print(f"{indent}{k}:")
                _describe_sample(v, indent + "  ")
            return

        if isinstance(x, (tuple, list)):
            for i, v in enumerate(x):
                print(f"{indent}[{i}]:")
                _describe_sample(v, indent + "  ")
            return

        _describe(x, indent)

    def _show_sample_pipeline(ds, x_keys, y_keys):
        """Build and display one sample through the dataset transform."""

        print("\nBuilding one sample")

        try:
            source_ds = _unwrap_dataset(ds)

            raw_item = (
                source_ds.data[0]
                if hasattr(source_ds, "data")
                else None
            )

            if raw_item is None:
                print("  Could not access raw dataset item.")
                return

            print("  starting from")
            print(f"    {raw_item}")

            transform = getattr(source_ds, "transform", None)

            if transform is None:
                print("  No item transform to apply")
                return

            transforms = _iter_transforms(transform)

            current = raw_item

            for t in transforms:
                print(f"\n  applying {_transform_name(t)}")

                try:
                    current = t(current)

                except Exception as e:
                    print(f"    ERROR: {e}")
                    break

                print("    gives")
                _describe_sample(current)

            print("\nFinal sample:")
            _describe_sample(current)

        except Exception as e:
            print(f"  Could not build sample: {e}")

    # ------------------------------------------------------------------
    # Raw dataset / DataLoader
    # ------------------------------------------------------------------

    def _get_stage_dataset(ds, n_transforms):
        """
        Create a temporary dataset containing only the first ``n_transforms``
        of the item pipeline.

        The original dataset is never modified.

        Parameters
        ----------
        ds : DataLoader dataset
            Dataset whose transform pipeline should be truncated.
        n_transforms : int
            Number of transforms to keep.

        Returns
        -------
        Dataset or None
            A shallow copy with the truncated transform pipeline.
        """

        source_ds = _unwrap_dataset(ds)

        transform = getattr(source_ds, "transform", None)

        if transform is None:
            return None

        transforms = _iter_transforms(transform)

        if not transforms:
            return None

        try:
            stage_ds = copy.copy(source_ds)

            if n_transforms == 0:
                stage_ds.transform = None

            elif n_transforms == 1:
                stage_ds.transform = transforms[0]

            else:
                # Preserve the same Compose type where possible.
                compose_cls = type(transform)

                try:
                    stage_ds.transform = compose_cls(
                        transforms[:n_transforms]
                    )
                except Exception:
                    stage_ds.transform = Compose(
                        transforms[:n_transforms]
                    )

            return stage_ds

        except Exception:
            return None


    def _get_stage_dl(dl, n_transforms):
        """
        Create a temporary DataLoader corresponding to a transform stage.

        The original DataLoader is never modified.
        """

        stage_ds = _get_stage_dataset(dl.dataset, n_transforms)

        if stage_ds is None:
            return None

        try:
            stage_dl = copy.copy(dl)
            stage_dl.dataset = stage_ds
            return stage_dl

        except Exception:
            return None

    # ------------------------------------------------------------------
    # Batch utilities
    # ------------------------------------------------------------------

    def _to_numpy(x):
        """Convert tensor-like image data to NumPy."""

        if isinstance(x, MetaTensor):
            return x.detach().cpu().numpy()

        if isinstance(x, torchTensor):
            return x.detach().cpu().numpy()

        if isinstance(x, np.ndarray):
            return x

        return None

    def _iter_named_values(x, name=""):
        """
        Recursively yield numerical arrays with their corresponding key.
        """

        if isinstance(x, dict):
            for key, value in x.items():
                child_name = f"{name}.{key}" if name else str(key)
                yield from _iter_named_values(value, child_name)
            return

        if isinstance(x, (tuple, list)):
            for i, value in enumerate(x):
                child_name = f"{name}[{i}]" if name else f"[{i}]"
                yield from _iter_named_values(value, child_name)
            return

        array = _to_numpy(x)

        if array is not None and np.issubdtype(array.dtype, np.number):
            yield name or "value", array

    def _stats(array):
        """Calculate compact numerical statistics."""

        array = np.asarray(array)

        if array.size == 0:
            return None

        values = array.astype(np.float64, copy=False)

        finite = values[np.isfinite(values)]

        if finite.size == 0:
            return None

        return {
            "shape": array.shape,
            "dtype": array.dtype,
            "min": float(finite.min()),
            "max": float(finite.max()),
            "mean": float(finite.mean()),
            "std": float(finite.std()),
        }

    def _show_stats(batch, title, bins=50):
        """
        Display statistics and intensity histograms for a batch.

        One histogram is produced for each numerical field in the batch.
        """

        named_values = list(_iter_named_values(batch))

        if not named_values:
            print(f"\n{title}: no numerical data found.")
            return

        print(f"\n{title}")

        for name, array in named_values:
            stats = _stats(array)

            if stats is None:
                continue

            print(
                f"  {name}: "
                f"shape={stats['shape']}  "
                f"dtype={stats['dtype']}  "
                f"range=[{stats['min']:.4g}, {stats['max']:.4g}]  "
                f"mean={stats['mean']:.4g}  "
                f"std={stats['std']:.4g}"
            )

        # --------------------------------------------------------------
        # Histograms
        # --------------------------------------------------------------

        for name, array in named_values:
            values = np.asarray(array).astype(
                np.float64,
                copy=False,
            )

            values = values[np.isfinite(values)]

            if values.size == 0:
                continue

            # Avoid huge memory use for very large volumes.
            if values.size > 1_000_000:
                rng = np.random.default_rng(0)
                values = rng.choice(
                    values,
                    size=1_000_000,
                    replace=False,
                )

            plt.figure(figsize=(6, 4))

            plt.hist(
                values.ravel(),
                bins=bins,
            )

            plt.xlabel("Intensity")
            plt.ylabel("Frequency")
            plt.title(f"{title}: {name}")
            plt.tight_layout()
            plt.show()

    # ------------------------------------------------------------------
    # Batch visualization
    # ------------------------------------------------------------------

    def _show_batch_comparison(dl):
        """
        Show images after loading and after all item transforms.

        ``show_batch`` itself is not modified.
        """

        transforms = _iter_transforms(
            getattr(_unwrap_dataset(dl.dataset), "transform", None)
        )

        if not transforms:
            print("\nNo item transforms available for comparison.")
            return

        # --------------------------------------------------------------
        # After first transform = after loading
        # --------------------------------------------------------------

        if len(transforms) >= 1:

            loaded_dl = _get_stage_dl(dl, 1)
            print(loaded_dl)

            if loaded_dl is not None:
                print("\nImages after loading:")

                try:
                    loaded_dl.show_batch(max_n=max_n)
                except TypeError:
                    loaded_dl.show_batch()

        # --------------------------------------------------------------
        # Final transformed images
        # --------------------------------------------------------------

        print("\nImages after transforms:")

        try:
            dl.show_batch(max_n=max_n)
        except TypeError:
            dl.show_batch()

    def _get_batch(dl):
        """Get one batch without modifying the DataLoader."""

        try:
            return next(iter(dl))
        except Exception as e:
            print(f"  Could not build batch: {e}")
            return None

    # ------------------------------------------------------------------
    # DataLoader
    # ------------------------------------------------------------------

    def _describe_dl(dl, name):

        print(f"\n{name} DataLoader")
        print("=" * (len(name) + 12))

        ds = dl.dataset
        source_ds = _unwrap_dataset(ds)

        try:
            ds_len = len(ds)
        except Exception:
            ds_len = "unknown"

        print(f"Dataset size : {ds_len}")
        print(f"Batch size   : {dl.batch_size}")
        print(f"Batches      : {len(dl)}")

        if hasattr(dl, "shuffle"):
            print(f"Shuffle      : {dl.shuffle}")

        print(f"Workers      : {dl.num_workers}")

        # --------------------------------------------------------------
        # Vocabulary
        # --------------------------------------------------------------

        vocab = getattr(dl, "vocab", None)

        if vocab is not None:
            print(f"Classes      : {vocab}")

        # --------------------------------------------------------------
        # Pipeline configuration
        # --------------------------------------------------------------

        _show_config(dl)

        # --------------------------------------------------------------
        # Dataset / item pipeline
        # --------------------------------------------------------------

        x_keys = getattr(ds, "x_keys", None)
        y_keys = getattr(ds, "y_keys", None)

        if x_keys is not None:
            print(f"x_keys       : {x_keys}")

        if y_keys is not None:
            print(f"y_keys       : {y_keys}")

        transform = getattr(source_ds, "transform", None)

        if transform is not None:
            print("\nItem transforms:")
            _show_pipeline(transform, prefix="  ")

        # --------------------------------------------------------------
        # Sample
        # --------------------------------------------------------------

        _show_sample_pipeline(ds, x_keys, y_keys)

        # --------------------------------------------------------------
        # Collation
        # --------------------------------------------------------------

        print("\nBatch transforms:")

        collate = getattr(dl, "collate_fn", None)

        if collate is None:
            print("  collate_fn: default")

        else:
            print(
                f"  collate_fn: "
                f"{getattr(collate, '__name__', type(collate).__name__)}"
            )

        # --------------------------------------------------------------
        # Final batch
        # --------------------------------------------------------------

        print("\nFinal batch:")

        batch = _get_batch(dl)

        if batch is not None:

            if isinstance(batch, (tuple, list)):

                if len(batch) == 2:
                    print("  x:")
                    _describe(batch[0], "    ")

                    print("  y:")
                    _describe(batch[1], "    ")

                else:
                    for i, item in enumerate(batch):
                        print(f"  [{i}]:")
                        _describe(item, "    ")

            elif isinstance(batch, dict):

                for k, v in batch.items():
                    print(f"  {k}:")
                    _describe(v, "    ")

            else:
                _describe(batch, "    ")

        # --------------------------------------------------------------
        # Visual comparison
        # --------------------------------------------------------------

        if show_batch:
            _show_batch_comparison(dl)

        # --------------------------------------------------------------
        # Numerical statistics
        # --------------------------------------------------------------

        if show_stats:

            # --------------------------------------------------------------
            # Statistics after loading
            # --------------------------------------------------------------

            loaded_dl = _get_stage_dl(dl, 1)

            if loaded_dl is not None:

                loaded_batch = _get_batch(loaded_dl)

                if loaded_batch is not None:
                    _show_stats(
                        loaded_batch,
                        "Statistics after loading",
                        bins=bins,
                    )

            # --------------------------------------------------------------
            # Statistics after all transforms
            # --------------------------------------------------------------

            if batch is not None:
                _show_stats(
                    batch,
                    "Statistics after transforms",
                    bins=bins,
                )

        # --------------------------------------------------------------
        # Sample metadata
        # --------------------------------------------------------------

        _show_sample_metadata(ds)

    # ------------------------------------------------------------------
    # Train / validation
    # ------------------------------------------------------------------

    _describe_dl(train_dl, "Train")

    if val_dl is not None:
        _describe_dl(val_dl, "Valid")

In [ ]:
#| hide
def test_show_summary():
    # Dummy dataloaders
    dummy_x = torchzeros(4, 1, 10, 10)
    dummy_y = torchones(4, dtype=torchlong)
    dummy_ds = TensorDataset(dummy_x, dummy_y)
    train_dl = torchDataLoader(dummy_ds, batch_size=2)
    val_dl = torchDataLoader(dummy_ds, batch_size=2)

    # Capture print output
    with mockpatch('builtins.print') as mock_print:
        _show_summary(train_dl, val_dl, show_batch=False, show_stats=False)
        output = "\n".join(str(call.args[0]) for call in mock_print.call_args_list if call.args)

    # Verify output contents
    test_eq("Train DataLoader" in output, True)
    test_eq("Valid DataLoader" in output, True)
    test_eq("Batch size   : 2" in output, True)
    test_eq("Batches      : 2" in output, True)

    return "_show_summary tests passed"

test_eq(test_show_summary(), "_show_summary tests passed")

#### Collate Functions

In [ ]:
#| export

def _to_numpy(x):
    if isinstance(x, torchTensor):
        return x.detach().cpu().numpy()
    if isinstance(x, tuple):
        return tuple(_to_numpy(v) for v in x)
    if isinstance(x, list):
        return [_to_numpy(v) for v in x]
    if isinstance(x, dict):
        return {k: _to_numpy(v) for k, v in x.items()}
    return x

def numpy_collate(batch):
    xs = []
    ys = []

    for sample in batch:
        x, y = sample

        if (
            isinstance(x, (tuple, list))
            and len(x) == 2
            and isinstance(x[1], dict)
        ):
            x = x[0]

        xs.append(x)
        ys.append(y)

    return np.stack(xs), np.asarray(ys)


MONAI_COLLATE = {
    "fastai": None,
    "monai": None,
    "torch": None,
    "ignite": None,
    "keras": None,
    "numpy": numpy_collate,
    # "tensorflow": numpy_collate,
    # "jax": numpy_collate,
}

### Pipeline Context
> `PipelineContext` is a central data structure (implemented as a standard Python `@dataclass`) designed to hold the entire state of the data loading pipeline as it progresses.

* **Unified State:** Efficiently stores raw inputs, task configurations, intermediate dataset objects, and the final constructed DataLoaders (`dls`).
* **Cross-Stage Communication:** Makes it easy to pass context across different stages of the bonsai architecture, ensuring data consistency.
* **Dynamic Metadata:** Built-in dictionaries for configuration (`config`) and runtime metadata (`metadata`) allow for flexible state updates.


In [ ]:
#| export
@dataclass
class PipelineContext:
    """Store the state and configuration of a data-loading pipeline.

    Attributes:
        data: Raw training input data.
        val_data: Raw validation input data.
        task: Pipeline task, such as ``"classification"`` or ``"segmentation"``.
        dataset_name: Name of the selected dataset component.
        mode: Current pipeline execution mode.
        records: Loaded or intermediate data records.
        train_ds: Training dataset.
        valid_ds: Validation dataset.
        dls: Constructed data loaders.
        config: Pipeline configuration settings.
        metadata: Runtime metadata collected during execution.
    """

    # raw inputs
    data: Any = None
    val_data: Any = None

    # task/config
    task: Optional[str] = None
    dataset_name: Optional[str] = None
    mode: str = "train"

    # loaded/intermediate state
    records: Optional[list[dict]] = None

    train_ds: Any = None
    valid_ds: Any = None

    dls: Any = None

    # unified config
    config: dict = field(default_factory=dict)

    # runtime metadata
    metadata: dict = field(default_factory=dict)

In [ ]:
show_doc(PipelineContext)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L1242){target="_blank" style="float:right; font-size:smaller"}

### PipelineContext

```python

def PipelineContext(
    data:Any=None, val_data:Any=None, task:Optional=None, dataset_name:Optional=None, mode:str='train',
    records:Optional=None, train_ds:Any=None, valid_ds:Any=None, dls:Any=None, config:dict=<factory>,
    metadata:dict=<factory>
)->None:


```

*Store the state and configuration of a data-loading pipeline.*

Attributes:
    data: Raw training input data.
    val_data: Raw validation input data.
    task: Pipeline task, such as ``"classification"`` or ``"segmentation"``.
    dataset_name: Name of the selected dataset component.
    mode: Current pipeline execution mode.
    records: Loaded or intermediate data records.
    train_ds: Training dataset.
    valid_ds: Validation dataset.
    dls: Constructed data loaders.
    config: Pipeline configuration settings.
    metadata: Runtime metadata collected during execution.

#### Example: Initializing and updating the state within a PipelineContext


In [ ]:
# 1. Create a context with raw data and a task configuration
ctx = PipelineContext(
    data=["scan1.nii.gz", "scan2.nii.gz"],
    task="segmentation",
)

# 2. Simulate pipeline progression by storing intermediate state/config
ctx.metadata["num_classes"] = 3
ctx.config["batch_size"] = 16
ctx.train_ds = "DatasetObjectPlaceholder"

print(f"Task:    {ctx.task}")
print(f"Data:    {ctx.data}")
print(f"Config:  {ctx.config}")

Task:    segmentation
Data:    ['scan1.nii.gz', 'scan2.nii.gz']
Config:  {'batch_size': 16}


In [ ]:
#| hide

def test_pipeline_context():
    # Initialize
    ctx = PipelineContext(
        data=[1, 2, 3],
        task="classification",
        mode="train"
    )
    
    # Modify state
    ctx.config["lr"] = 1e-3
    ctx.metadata["info"] = "test"
    ctx.train_ds = "dummy_ds"

    # Verify
    test_eq(ctx.data, [1, 2, 3])
    test_eq(ctx.task, "classification")
    test_eq(ctx.mode, "train")
    test_eq(ctx.config["lr"], 1e-3)
    test_eq(ctx.metadata["info"], "test")
    test_eq(ctx.train_ds, "dummy_ds")

    return "PipelineContext tests passed"

test_eq(test_pipeline_context(), "PipelineContext tests passed")

#### `PipelineContext` Class Attributes & Parameter Reference

| Attribute / Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`data`** | `Any` | `None` | The raw input data for training. |
| **`val_data`** | `Any` | `None` | The raw input data for validation. |
| **`task`** | `str \| None` | `None` | The specific pipeline task (e.g., `"classification"`, `"segmentation"`). |
| **`dataset_name`** | `str \| None` | `None` | The name or identifier of the selected dataset component. |
| **`mode`** | `str` | `"train"` | The current execution mode of the pipeline. |
| **`records`** | `list[dict] \| None` | `None` | Loaded or intermediate parsed data records. |
| **`train_ds`** | `Any` | `None` | The instantiated training dataset object. |
| **`valid_ds`** | `Any` | `None` | The instantiated validation dataset object. |
| **`dls`** | `Any` | `None` | The final constructed DataLoaders object. |
| **`config`** | `dict` | `{}` | Unified configuration dictionary for pipeline settings. |
| **`metadata`** | `dict` | `{}` | Runtime metadata populated dynamically during execution. |

### Source Detection & Factory

The library provides an intelligent, automated routing mechanism to handle diverse raw data formats seamlessly.

* **Intelligent Routing:** `detect_source` analyzes the structure of the input data (e.g., Pandas DataFrames, CSV file paths, directories, lists, or dictionaries) and identifies the underlying data type to determine the appropriate `Source` handler.
* **Dynamic Factory:** `build_source` leverages `detect_source` to identify the data structure and automatically instantiates the corresponding handler class from the `SOURCE_REGISTRY`. This enables you to pass raw data directly into the pipeline without manually importing or configuring specific Source classes.

Included Classes & Methods:
* **`detect_source`** analyzes the structure of the input data (e.g., Pandas DataFrames, CSV file paths, directories, lists, or dictionaries) and identifies the underlying data type. This allows the system to determine the appropriate `Source` handler without requiring explicit user configuration.

* **`build_source`** acts as the dynamic factory for your data pipeline. It leverages `detect_source` to identify the data structure and automatically instantiates the corresponding handler class from the `SOURCE_REGISTRY`. This enables you to pass raw data directly into the pipeline without manually importing or configuring specific Source classes.

In [ ]:
#| export

def detect_source(data):
    """
    Detect the source type of input data.

    Parameters
    ----------
    data : object
        Data source to inspect. Supported sources include callables,
        DataFrames, dictionaries, lists/tuples, CSV paths, and folders.

    Returns
    -------
    str
        Registered source name corresponding to the input type.

    Raises
    ------
    ValueError
        If the input cannot be mapped to a supported source type.
    """
    if callable(data):
        return "callable"

    if isinstance(data, pd.DataFrame):
        return "dataframe"

    # Single dictionary
    if isinstance(data, dict):
        return "dict"

    # List/tuple handling
    if isinstance(data, (list, tuple)):

        if len(data) == 0:
            return "list"

        # List of dictionaries
        if all(isinstance(x, dict) for x in data):
            return "dict"

        return "list"

    # String or Path
    if isinstance(data, (str, Path)):
        path = Path(data)

        if path.suffix.lower() == ".csv":
            return "csv"

        if path.is_dir():
            return "folder"

    raise ValueError("Cannot detect data source")

In [ ]:
show_doc(detect_source)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L1285){target="_blank" style="float:right; font-size:smaller"}

### detect_source

```python

def detect_source(
    data, # Data source to inspect. Supported sources include callables,
DataFrames, dictionaries, lists/tuples, CSV paths, and folders.
): # Registered source name corresponding to the input type.


```

*Detect the source type of input data.*

#### Example: Automatically detecting data source types


In [ ]:

# Define various data formats
sample_a = "dataset_metadata.csv"
sample_b = pd.DataFrame({"image": ["img1.png", "img2.png"], "label": [0, 1]})
sample_c = ["train_images", "val_images"]

# Detect sources
print(f"Source A: {detect_source(sample_a)}")
print(f"Source B: {detect_source(sample_b)}")
print(f"Source C: {detect_source(sample_c)}")

Source A: csv
Source B: dataframe
Source C: list


In [ ]:
#| hide

def test_detect_source():
    # DataFrame
    df = pd.DataFrame({"img": ["i1", "i2"]})
    test_eq(detect_source(df), 'dataframe')

    # List
    test_eq(detect_source(["f1", "f2"]), 'list')
    
    # List of Dicts
    test_eq(detect_source([{"a": 1}, {"b": 2}]), 'dict')

    # CSV string
    test_eq(detect_source("data.csv"), 'csv')
    
    # Folder path
    test_eq(detect_source('.'), 'folder')

    return "detect_source tests passed"

test_eq(test_detect_source(), "detect_source tests passed")

#### `detect_source` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`data`** | `Any` | *Required* | The raw input data to analyze (e.g., list, dict, DataFrame, file path). |

In [ ]:
#| export
def build_source(data, **kwargs):

    name = detect_source(data)

    source_cls = SOURCE_REGISTRY[name]

    return source_cls(data, **kwargs)

In [ ]:
show_doc(build_source)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L1340){target="_blank" style="float:right; font-size:smaller"}

### build_source

```python

def build_source(
    data, kwargs:VAR_KEYWORD
):


```

#### Example: Using build_source as an automatic factory


In [ ]:
# 1. Register a custom handler for "list" data types
@register_source("list")
class DemoListSource:
    def __init__(self, data, **kwargs):
        self.data = data
        self.kwargs = kwargs

# 2. Provide raw data
raw_data = ["scan_01.nii.gz", "scan_02.nii.gz"]

# 3. Automatically detect and build the appropriate source
my_source = build_source(raw_data, base_path="/medical_data")

print(f"Built type: {type(my_source).__name__}")
print(f"Kwargs:     {my_source.kwargs}")

Built type: DemoListSource
Kwargs:     {'base_path': '/medical_data'}


In [ ]:
#| hide

def test_build_source():
    # Backup registry
    orig_registry = SOURCE_REGISTRY.copy()

    # Register dummy class
    class DummyDictSource:
        def __init__(self, data, **kwargs):
            self.data = data
            self.kwargs = kwargs

    SOURCE_REGISTRY["dict"] = DummyDictSource

    # Test factory routing
    test_dict = {"image": "img.png", "label": 1}
    built = build_source(test_dict, extra_arg=True)

    # Verify instantiation and kwargs
    test_eq(isinstance(built, DummyDictSource), True)
    test_eq(built.data, test_dict)
    test_eq(built.kwargs, {"extra_arg": True})

    # Cleanup
    SOURCE_REGISTRY.clear()
    SOURCE_REGISTRY.update(orig_registry)

    return "build_source tests passed"

test_eq(test_build_source(), "build_source tests passed")

#### `build_source` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`data`** | `Any` | *Required* | The raw input data to process and route. |
| **`**kwargs`** | `dict` | `{}` | Additional keyword arguments passed directly to the instantiated Source class. |

### Data Sources

The library employs a hierarchical system of **Source** classes designed to ingest and standardize various raw data formats seamlessly. All specialized handlers ultimately convert their inputs into records and delegate the heavy lifting to the core `DictSource` engine, ensuring a consistent processing pipeline regardless of the original data structure.

* **`BaseSource`:** The foundational abstract class. It provides essential built-in utilities for parsing raw data, mapping dictionary keys, handling missing values, and dynamically resolving file paths (applying base paths, folders, and suffixes).
* **`DictSource`:** The core processing engine. It processes single dictionaries or lists of dictionaries, featuring robust key mapping capabilities, automatic path formatting, dynamic value generation via callables, and seamless integration with class-specific loaders.
* **`DataFrameSource`:** A specialized adapter for Pandas DataFrames. It converts tabular rows into dictionary records, enabling you to leverage all of `DictSource`'s features directly on tabular data.
* **`CSVSource`:** A highly convenient wrapper that automatically reads a CSV file into a Pandas DataFrame and delegates it to `DataFrameSource`.
* **`FolderSource`:** Automates the ingestion of data directly from directory structures. It scans specified folders, sorts contents, and automatically pairs corresponding files (e.g., raw images with their respective masks) into unified records.
* **`ListSource`:** Gracefully handles standard Python lists. It accepts either flat lists of file paths (automatically mapped to an `image` key) or lists of dictionaries, routing them through the dictionary processing engine.
* **`CallableSource`:** Enables dynamic, on-the-fly dataset generation using Python functions. This is ideal when data must be retrieved from a database, an API, or derived via custom logic during runtime.

In [ ]:
#| export
class BaseSource:
    def __init__(
            self, 
            data, 
            x_keys=None,
            y_keys=None,
            x_class=None,
            y_class=None,
            get_items=None,
            base_path=None,
            folders=None,
            suffixes=None,
            keep_original=False,
            **kwargs):
        
        store_attr()
        self.kwargs = kwargs

    def load(self):
        raise NotImplementedError

    def df(self):
        return pd.DataFrame(self.load())
    
    def peak(self):
        return self.df().head()
    
    def _add_key_mappings(self, keys):
        if keys is None:
            return
        if isinstance(keys, str):
            keys = [keys]
        for key in keys:
            self.get_items.setdefault(key, key)

    def _build_classification_encoders(self):

        loaders = []

        for cls, keys in (
            (self.x_class, self.x_keys),
            (self.y_class, self.y_keys)
        ):
            if cls and hasattr(cls, "get_dict_label"):

                if keys is not None:
                    self.kwargs["keys"] = keys

                if self.kwargs.get("vocab") is None:
                    labels = [d["label"] for d in self.data]

                    vocab = CategoryMap(
                        labels,
                        sort=self.kwargs.get("sort", cls._sort),
                        add_na=self.kwargs.get("add_na", cls._add_na),
                    )

                    self.kwargs["vocab"] = vocab

                loaders.append(
                    cls.get_dict_label(**dict(self.kwargs))
                )

        return loaders

    def _has_path_config(self, new_col):
        return (
            new_col in self.folders
            or new_col in self.suffixes
            or self.base_path != Path()
        )

    def _is_missing(self, value):
        if value is None:
            return True
        if isinstance(value, (list, tuple, dict)):
            return False
        try:
            return bool(pd.isna(value))
        except (TypeError, ValueError):
            return False

    def _build_path(self, value, new_col):
        if self._is_missing(value):
            return None

        value = Path(str(value))
        suffix = self.suffixes.get(new_col, "")
        if suffix and not value.name.endswith(suffix):
            value = value.with_name(f"{value.name}{suffix}")
        if value.is_absolute():
            return value.as_posix()

        base = self.base_path

        folder = self.folders.get(new_col)
        if folder:
            base = base / folder

        return (base / value).as_posix()

    def _build_paths(self, values, new_col):
        return [self._build_path(v, new_col) for v in values]

    def _format_value(self, value, new_col):
        if not self._has_path_config(new_col):
            return value
        if isinstance(value, (list, tuple)):
            return self._build_paths(value, new_col)
        return self._build_path(value, new_col)

In [ ]:
show_doc(BaseSource)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L1349){target="_blank" style="float:right; font-size:smaller"}

### BaseSource

```python

def BaseSource(
    data, x_keys:NoneType=None, y_keys:NoneType=None, x_class:NoneType=None, y_class:NoneType=None,
    get_items:NoneType=None, base_path:NoneType=None, folders:NoneType=None, suffixes:NoneType=None,
    keep_original:bool=False, kwargs:VAR_KEYWORD
):


```

*Initialize self.  See help(type(self)) for accurate signature.*

#### Example: Creating a custom data source inheriting from BaseSource


In [ ]:
class MyCustomSource(BaseSource):
    def load(self):
        raw_data = [{"image": "scan_01"}, {"image": "scan_02"}]
        # Automatically apply base paths, folders, and suffixes
        for row in raw_data:
            row["image"] = self._format_value(row["image"], new_col="image")
        return raw_data

# Initialize with specific path formatting rules
my_source = MyCustomSource(
    data=None, 
    base_path=Path("/medical_data"),
    folders={"image": "images"},
    suffixes={"image": ".nii.gz"}
)

# View the generated DataFrame
print(my_source.peak())

                                 image
0  /medical_data/images/scan_01.nii.gz
1  /medical_data/images/scan_02.nii.gz


In [ ]:
#| hide

def test_basesource():
    base_src = BaseSource(
        data=None,
        base_path=Path("dataset"),
        folders={"image": "scans"},
        suffixes={"image": ".nii.gz"},
        get_items={}
    )

    # Missing values
    test_eq(base_src._is_missing(None), True)
    test_eq(base_src._is_missing(pd.NA), True)
    test_eq(base_src._is_missing("valid_string"), False)

    # Path configs
    test_eq(base_src._has_path_config("image"), True)

    # Path building
    test_eq(base_src._build_path("patient_01", "image"), "dataset/scans/patient_01.nii.gz")
    test_eq(base_src._build_path("patient_02.nii.gz", "image"), "dataset/scans/patient_02.nii.gz")

    # Format list of values
    test_eq(
        base_src._format_value(["pt1", "pt2"], "image"), 
        ["dataset/scans/pt1.nii.gz", "dataset/scans/pt2.nii.gz"]
    )

    # Key mappings
    base_src._add_key_mappings(["mask", "label"])
    test_eq(base_src.get_items["mask"], "mask")
    test_eq(base_src.get_items["label"], "label")

    return "BaseSource tests passed"

test_eq(test_basesource(), "BaseSource tests passed")

#### `BaseSource` Class Attributes & Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`data`** | `Any` | *Required* | The raw input data to process. |
| **`x_keys`** | `str \| list` | `None` | The keys representing input data (features/images). |
| **`y_keys`** | `str \| list` | `None` | The keys representing target data (labels/masks). |
| **`x_class`** | `class` | `None` | Class used to format or instantiate input data (e.g., `BioImage`). |
| **`y_class`** | `class` | `None` | Class used to format or instantiate target data (e.g., `BioLabel`). |
| **`get_items`** | `dict` | `None` | Mapping dictionary to define relationships between final keys and raw data columns. |
| **`base_path`** | `str \| Path` | `None` | A root path prepended to all dynamically resolved file paths. |
| **`folders`** | `dict` | `None` | A mapping of keys to specific subfolders to append to the base path. |
| **`suffixes`** | `dict` | `None` | A mapping of keys to specific file extensions to append to paths. |
| **`keep_original`** | `bool` | `False` | If True, retains all unmapped raw data fields in the final output records. |
| **`**kwargs`** | `dict` | `{}` | Additional keyword arguments passed down to the inner mechanics. |

In [ ]:
#| export
@register_source("dict")
class DictSource(BaseSource):
    def __init__(
        self,
        data,
        x_keys=None,
        y_keys=None,
        x_class=None,
        y_class=None,
        get_items=None,
        base_path=None,
        folders=None,
        suffixes=None,
        keep_original=False,
        **kwargs,
    ):
        if isinstance(data, dict):
            data = [data]

        self.data = list(data)

        self.get_items = dict(get_items or {})
        self._add_key_mappings(x_keys)
        self._add_key_mappings(y_keys)

        self.base_path = Path(base_path) if base_path else Path()
        self.folders = folders or {}
        self.suffixes = suffixes or {}
        self.keep_original = keep_original
        self.x_keys = x_keys
        self.y_keys = y_keys
        self.x_class = x_class
        self.y_class = y_class
        self.kwargs = kwargs

        self.encoders = self._build_classification_encoders()

    def _resolve(self):
        out = []

        for row in self.data:
            item = dict(row) if self.keep_original else {}

            for new_col, src_col in self.get_items.items():
                if isinstance(src_col, str):
                    value = row.get(src_col)
                    item[new_col] = self._format_value(value, new_col)

                elif isinstance(src_col, (list, tuple)):
                    values = [row.get(c) for c in src_col]
                    item[new_col] = self._format_value(values, new_col)

                elif isinstance(src_col, Callable):
                    value = src_col(row)
                    item[new_col] = self._format_value(value, new_col)

                else:
                    raise ValueError(
                        f"Invalid src_col type for {new_col}: {type(src_col)}"
                    )

            for encoder in self.encoders:
                item = encoder(item)

            out.append(item)

        # One vocab per encoder.
        vocabs = [encoder.vocab for encoder in self.encoders]
        self.vocab = vocabs[0] if len(vocabs) == 1 else vocabs

        return out  

    def load(self):
        return self._resolve()

In [ ]:
show_doc(DictSource)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L1462){target="_blank" style="float:right; font-size:smaller"}

### DictSource

```python

def DictSource(
    data, x_keys:NoneType=None, y_keys:NoneType=None, x_class:NoneType=None, y_class:NoneType=None,
    get_items:NoneType=None, base_path:NoneType=None, folders:NoneType=None, suffixes:NoneType=None,
    keep_original:bool=False, kwargs:VAR_KEYWORD
):


```

*Initialize self.  See help(type(self)) for accurate signature.*

#### Example: Loading and preprocessing data from a dictionary


In [ ]:

raw_dict_data = {
    "ch1": "img001_ch1.nii.gz", 
    "ch2": "img001_ch2", 
    "label": "A", 
    "scale": 2
}

# Map keys, combine channels, apply functions, and format paths
dict_source = DictSource(
    data=raw_dict_data,
    y_keys="label",
    y_class=BioLabel, # Automatically encodes text to int based on vocab
    get_items={
        "image": ("ch1", "ch2"), 
        "weight": lambda row: row["scale"] * 10
    },
    folders={"image": "data/images"},
    suffixes={"image": ".nii.gz"},
    # vocab=['A', 'B']
)

print(dict_source.load())

[{'image': ['data/images/img001_ch1.nii.gz', 'data/images/img001_ch2.nii.gz'], 'weight': 20, 'label': tensor(0)}]


In [ ]:
dict_source.kwargs

{'keys': 'label', 'vocab': ['A']}

In [ ]:
#| hide

def test_dictsource():
    source = DictSource(
        {"ch1": "img001_ch1.nii.gz", "ch2": "img001_ch2", "label": "A", "scale": 2},
        y_keys="label",
        y_class=BioLabel,
        get_items={"image": ("ch1", "ch2"), "weight": lambda row: row["scale"] * 10},
        folders={"image": "data/images"},
        suffixes={"image": ".nii.gz"},
        vocab=['A', 'B'],
    )

    # Verify mappings, path formatting, and loaders
    res = source.load()
    test_eq(res, [{
        "image": ["data/images/img001_ch1.nii.gz", "data/images/img001_ch2.nii.gz"],
        "weight": 20,
        "label": 0,
    }])

    return "DictSource tests passed"

test_eq(test_dictsource(), "DictSource tests passed")

#### `DictSource` Class Attributes & Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`data`** | `dict \| list` | *Required* | A single dictionary or a list of dictionaries to process. |
| **`x_keys`** | `str \| list` | `None` | Keys designating the inputs. |
| **`y_keys`** | `str \| list` | `None` | Keys designating the targets. |
| **`x_class`** | `class` | `None` | Loading/instantiating class for the inputs. |
| **`y_class`** | `class` | `None` | Loading/instantiating class for the targets. |
| **`get_items`** | `dict` | `None` | Dictionary mapping defining value resolution. Values can be strings, lists of strings, or Callables. |
| **`base_path`** | `str \| Path` | `None` | Root path to prepend to items. |
| **`folders`** | `dict` | `None` | Map indicating subdirectories for specific keys. |
| **`suffixes`** | `dict` | `None` | Map indicating expected extensions for specific keys. |
| **`keep_original`** | `bool` | `False` | Retain original unmapped dictionary keys in output. |
| **`**kwargs`** | `dict` | `{}` | Additional settings. |

In [ ]:
#| export
@register_source("dataframe")
class DataFrameSource(BaseSource):
    def __init__(
        self,
        df,
        x_keys=None,
        y_keys=None,
        get_items=None,
        base_path=None,
        folders=None,
        suffixes=None,
        keep_original=False,
        **kwargs,
    ):

        self.source = DictSource(
            data=df.to_dict(orient="records"),
            x_keys=x_keys,
            y_keys=y_keys,
            get_items=get_items,
            base_path=base_path,
            folders=folders,
            suffixes=suffixes,
            keep_original=keep_original,
            **kwargs
        )

    def load(self):
        return self.source.load()
    

In [ ]:
show_doc(DataFrameSource)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L1538){target="_blank" style="float:right; font-size:smaller"}

### DataFrameSource

```python

def DataFrameSource(
    df, x_keys:NoneType=None, y_keys:NoneType=None, get_items:NoneType=None, base_path:NoneType=None,
    folders:NoneType=None, suffixes:NoneType=None, keep_original:bool=False, kwargs:VAR_KEYWORD
):


```

*Initialize self.  See help(type(self)) for accurate signature.*

#### Example 1: Combining multiple columns into a single channel list


In [ ]:
df_multi = pd.DataFrame({
    "channel1": ["img001_ch1", "img002_ch1"],
    "channel2": ["img001_ch2", "img002_ch2"],
    "mask": ["mask001", "mask002"]
})

source_multi = DataFrameSource(
    df=df_multi,
    get_items={"image": ["channel1", "channel2"], "label": "mask"},
    base_path="data",
    folders={"image": "images", "label": "masks"},
    suffixes={"image": ".png", "label": ".nii.gz"}
)

print("--- Example 1: Multi-channel Output ---")
for item in source_multi.load():
    print(item)

--- Example 1: Multi-channel Output ---
{'image': ['data/images/img001_ch1.png', 'data/images/img001_ch2.png'], 'label': 'data/masks/mask001.nii.gz'}
{'image': ['data/images/img002_ch1.png', 'data/images/img002_ch2.png'], 'label': 'data/masks/mask002.nii.gz'}


#### Example 2: Single column mapping and exporting back to a clean DataFrame


In [ ]:
df_single = pd.DataFrame({
    "filename": ["img001", "img002"],
    "labels": [0, 1]
})

source_single = DataFrameSource(
    df=df_single,
    y_keys="labels",
    get_items={"image": "filename"},
    folders={"image": "data/images"},
    suffixes={"image": ".nii.gz"}
)

print("\n--- Example 2: Exported as Pandas DataFrame ---")
print(source_single.df())


--- Example 2: Exported as Pandas DataFrame ---
                       image  labels
0  data/images/img001.nii.gz       0
1  data/images/img002.nii.gz       1


#### Example 3: Keep the original DataFrame data without path modifications


In [ ]:
source_asis = DataFrameSource(
    df=df_single,
    get_items=None,
    keep_original=True
)

print("\n--- Example 3: Kept Original Data ---")
print(source_asis.load())


--- Example 3: Kept Original Data ---
[{'filename': 'img001', 'labels': 0}, {'filename': 'img002', 'labels': 1}]


In [ ]:
#| hide

def test_dataframesource():
    # Setup
    df_multi = pd.DataFrame({
        "channel1": ["img001_ch1", "img002_ch1"],
        "channel2": ["img001_ch2", "img002_ch2"],
        "mask": ["mask001", "mask002"]
    })
    
    df_single = pd.DataFrame({
        "filename": ["img001", "img002"],
        "labels": [0, 1]
    })

    # Test 1: Multi-channel
    source_multi = DataFrameSource(
        df=df_multi,
        get_items={"image": ["channel1", "channel2"], "label": "mask"},
        base_path="data",
        folders={"image": "images", "label": "masks"},
        suffixes={"image": ".png", "label": ".nii.gz"}
    )
    expected_multi = [
        {'image': ['data/images/img001_ch1.png', 'data/images/img001_ch2.png'], 'label': 'data/masks/mask001.nii.gz'},
        {'image': ['data/images/img002_ch1.png', 'data/images/img002_ch2.png'], 'label': 'data/masks/mask002.nii.gz'}
    ]
    test_eq(source_multi.load(), expected_multi)

    # Test 2: Single column & DataFrame generation
    source_single = DataFrameSource(
        df=df_single,
        y_keys="labels",
        get_items={"image": "filename"},
        folders={"image": "data/images"},
        suffixes={"image": ".nii.gz"}
    )
    expected_df_single = pd.DataFrame({
        "labels": [0, 1],
        "image": ["data/images/img001.nii.gz", "data/images/img002.nii.gz"]
    })
    test_eq(source_single.df().to_dict(orient="records"), expected_df_single.to_dict(orient="records"))

    # Test 3: Keep original
    source_asis = DataFrameSource(df=df_single, get_items=None, keep_original=True)
    test_eq(source_asis.load(), df_single.to_dict(orient="records"))

    # Test 4: String labels mapping
    df_str = pd.DataFrame({"filename": ["img001", "img002"], "labels": ["catA", "catB"]})
    source_str = DataFrameSource(
        df=df_str, y_keys="labels", get_items={"image": "filename"}, 
        folders={"image": "data/images"}, suffixes={"image": ".nii.gz"}
    )
    expected_str = [
        {'labels': 'catA', 'image': 'data/images/img001.nii.gz'},
        {'labels': 'catB', 'image': 'data/images/img002.nii.gz'}
    ]
    test_eq(source_str.load(), expected_str)

    return "DataFrameSource tests passed"

test_eq(test_dataframesource(), "DataFrameSource tests passed")

#### `DataFrameSource` Class Attributes & Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`df`** | `pd.DataFrame` | *Required* | The Pandas DataFrame containing the raw tabular data. |
| **`x_keys`** | `str \| list` | `None` | Column name(s) designating the inputs. |
| **`y_keys`** | `str \| list` | `None` | Column name(s) designating the targets. |
| **`get_items`** | `dict` | `None` | Mapping to extract complex combinations of columns into single keys. |
| **`base_path`** | `str \| Path` | `None` | Root path for resolved string entries. |
| **`folders`** | `dict` | `None` | Subdirectory mapping for keys. |
| **`suffixes`** | `dict` | `None` | File extension mapping for keys. |
| **`keep_original`** | `bool` | `False` | Retain original unmapped DataFrame columns in the final dict records. |
| **`**kwargs`** | `dict` | `{}` | Passed down to `DictSource`. |

In [ ]:
#| export
@register_source("csv")
class CSVSource(BaseSource):

    @delegates(DataFrameSource.__init__)
    def __init__(self, path, **kwargs):
        self.path = path

        df = pd.read_csv(path)
        source = DataFrameSource(df, **kwargs)
        self.source = source

    def load(self):
        return self.source.load()

In [ ]:
show_doc(CSVSource)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L1570){target="_blank" style="float:right; font-size:smaller"}

### CSVSource

```python

def CSVSource(
    path, x_keys:NoneType=None, y_keys:NoneType=None, get_items:NoneType=None, base_path:NoneType=None,
    folders:NoneType=None, suffixes:NoneType=None, keep_original:bool=False
):


```

*Initialize self.  See help(type(self)) for accurate signature.*

#### Example: Loading and mapping data directly from a CSV file


In [ ]:

# 1. Create a dummy CSV file
sample_csv = Path("sample_dataset.csv")
pd.DataFrame({
    "image_file": ["scan_001", "scan_002"],
    "diagnosis": ["Healthy", "Pathological"]
}).to_csv(sample_csv, index=False)

# 2. Configure CSVSource to read and format the data
csv_source = CSVSource(
    path=sample_csv,
    y_keys="diagnosis",
    get_items={"image": "image_file"},
    folders={"image": "medical_scans"},
    suffixes={"image": ".nii.gz"}
)

print("Processed Output:")
for item in csv_source.load():
    print(item)

# Cleanup
if sample_csv.exists():
    sample_csv.unlink()

Processed Output:
{'image': 'medical_scans/scan_001.nii.gz', 'diagnosis': 'Healthy'}
{'image': 'medical_scans/scan_002.nii.gz', 'diagnosis': 'Pathological'}


In [ ]:
#| hide

def test_csvsource():
    test_csv = Path("test_dummy_data.csv")
    pd.DataFrame({
        "ch1": ["img1_c1", "img2_c1"],
        "ch2": ["img1_c2", "img2_c2"],
        "mask": ["m1", "m2"]
    }).to_csv(test_csv, index=False)

    source = CSVSource(
        path=test_csv,
        get_items={"image": ["ch1", "ch2"], "label": "mask"},
        base_path="data",
        folders={"image": "images", "label": "masks"},
        suffixes={"image": ".png", "label": ".nii.gz"}
    )

    expected = [
        {'image': ['data/images/img1_c1.png', 'data/images/img1_c2.png'], 'label': 'data/masks/m1.nii.gz'},
        {'image': ['data/images/img2_c1.png', 'data/images/img2_c2.png'], 'label': 'data/masks/m2.nii.gz'}
    ]
    
    test_eq(source.load(), expected)

    if test_csv.exists():
        test_csv.unlink()

    return "CSVSource tests passed"

test_eq(test_csvsource(), "CSVSource tests passed")

#### `CSVSource` Class Attributes & Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`path`** | `str \| Path` | *Required* | The file path to the CSV file to be loaded. |
| **`**kwargs`** | `dict` | `{}` | Arguments delegated down to `DataFrameSource` and `DictSource` (like `x_keys`, `get_items`, `base_path`, etc.). |

In [ ]:
#| export

@register_source("folder")
class FolderSource(BaseSource):
    """
    Source for discovering records from a directory tree.

    One field in ``get_items`` defines the primary filesystem items.
    Each discovered primary item becomes one record. All remaining
    fields are derived from that primary path using callables.

    The primary field is specified by ``x_key``. If ``x_key`` is omitted,
    the first field in ``get_items`` is used.

    The primary specification can be:

    - a directory path, which is recursively scanned for files;
    - a glob pattern;
    - a regex, which is recursively matched against files from ``root``;
    - ``(subfolder, regex)``, which recursively matches files below a
      subfolder.

    All non-primary specifications must be callables and receive the
    discovered primary path.

    Parameters
    ----------
    root : path-like
        Root directory containing the data.
    get_items : dict
        Mapping from field names to discovery or derivation specifications.
    x_key : str, optional
        Name of the primary field. Defaults to the first field.
    **kwargs
        Additional arguments forwarded to :class:`DictSource`.
    """

    def __init__(
        self,
        root,
        get_items,
        x_key=None,
        folders=None,
        keep_original=True,
        **kwargs,
    ):
        self.root = Path(root)
        self.get_items = get_items
        self.x_key = x_key or next(iter(get_items))

        if self.x_key not in get_items:
            raise ValueError(
                f"x_key {self.x_key!r} is not present in get_items."
            )

        # Only the primary field is allowed to perform filesystem
        # discovery. Every other field is derived from its path.
        for key, spec in get_items.items():
            if key != self.x_key and not callable(spec):
                raise ValueError(
                    f"get_items[{key!r}] must be callable because only "
                    f"the x_key ({self.x_key!r}) can define discovery."
                )

        data = self._discover()

        folders = folders or {self.x_key: self.root}

        self.source = DictSource(
            data,
            get_items={k: k for k in data[0]} if data else {},
            folders=folders,
            keep_original=keep_original,
            **kwargs,
        )

    def _discover(self):
        """Discover primary files and build one record per file."""

        # The x specification is the sole source of record discovery.
        primary = self._scan(self.get_items[self.x_key])

        records = []

        for path in primary:
            if not path.is_file():
                continue

            record = {}

            # The primary field contains the discovered path. All other
            # fields are derived from exactly this same path.
            for key, spec in self.get_items.items():
                if key == self.x_key:
                    record[key] = path.relative_to(self.root).as_posix()
                else:
                    record[key] = spec(path)

            records.append(record)

        return records

    def _scan(self, spec):
        """Discover files from a directory, glob, or regular expression."""

        # (subfolder, regex) searches recursively below the subfolder.
        if isinstance(spec, tuple):
            if len(spec) != 2:
                raise ValueError(
                    "A discovery tuple must have the form "
                    "(subfolder, regex)."
                )

            subfolder, pattern = spec
            base = self.root / subfolder

            if not base.is_dir():
                raise ValueError(
                    f"Discovery directory does not exist: {base}"
                )

            regex = re.compile(pattern)

            return sorted(
                path
                for path in base.rglob("*")
                if path.is_file()
                and regex.search(path.relative_to(self.root).as_posix())
            )

        if not isinstance(spec, str):
            raise ValueError(
                "The x_key specification must be a directory, glob, "
                "regex, or (subfolder, regex) tuple."
            )

        path = self.root / spec

        # An existing directory is recursively scanned.
        if path.is_dir():
            return sorted(
                p for p in path.rglob("*")
                if p.is_file()
            )

        # Glob expressions are resolved relative to root.
        if any(c in spec for c in "*?[]"):
            return sorted(
                p for p in self.root.glob(spec)
                if p.is_file()
            )

        # Otherwise treat the string as a regular expression and search
        # recursively from root.
        regex = re.compile(spec)

        return sorted(
            path
            for path in self.root.rglob("*")
            if path.is_file()
            and regex.search(path.relative_to(self.root).as_posix())
        )

    def load(self):
        """Load the discovered records using the underlying DictSource."""
        return self.source.load()

In [ ]:
#| export

def get_path(root=None):
    """Return a callable that returns the path relative to ``root``."""

    def _get(path):
        return path.relative_to(root) if root is not None else path

    return _get


def get_name(path):
    """Return the filename of the discovered item."""
    return path.name


def get_stem(path):
    """Return the filename stem of the discovered item."""
    return path.stem


def get_parent(path):
    """Return the name of the parent directory of the discovered item."""
    return path.parent.name


def get_suffix(path):
    """Return the suffix of the discovered item."""
    return path.suffix


def get_paired_image(
    subfolder=None,
    replace=None,
    suffix="",
    prefix="",
    extension=None,
):
    """
    Return a callable that constructs a paired file path.

    The returned callable accepts either a string or path-like object and
    constructs the corresponding paired file path.

    Parameters
    ----------
    subfolder : str or path-like, optional
        Subfolder containing the paired file. If ``None``, keep the
        original parent directory.
    replace : str, tuple, or dict, optional
        Replacement applied to the filename stem.

        - ``str``: remove the substring.
        - ``(old, new)``: replace ``old`` with ``new``.
        - ``dict``: apply each ``old -> new`` replacement.
    suffix : str, optional
        Suffix to append to the filename stem.
    prefix : str, optional
        Prefix to prepend to the filename stem.
    extension : str, optional
        Replacement extension for the paired file.
    """

    def _get(path):
        path = Path(path)

        stem = path.stem

        if isinstance(replace, str):
            stem = stem.replace(replace, "")

        elif isinstance(replace, tuple):
            old, new = replace
            stem = stem.replace(old, new)

        elif isinstance(replace, dict):
            for old, new in replace.items():
                stem = stem.replace(old, new)

        filename = f"{prefix}{stem}{suffix}"

        if extension is None:
            extension_ = path.suffix
        else:
            extension_ = str(extension)
            if not extension_.startswith("."):
                extension_ = f".{extension_}"

        filename += extension_

        if subfolder is None:
            return str(path.parent / filename)

        return str(path.parent.parent / subfolder / filename)

    return _get


def get_mask(subfolder="masks", suffix="", extension=None):
    """
    Return a callable that constructs a mask path paired with ``path``.
    """
    return get_paired_image(
        subfolder=subfolder,
        suffix=suffix,
        extension=extension,
    )

#### Example: Automatically pairing images and masks from directories

Folder structure:

```text
root/
├── images/
│   ├── scan_01.nii.gz
│   └── scan_02.nii.gz
└── masks/
    ├── scan_01.nii.gz
    └── scan_02.nii.gz
```

The ``image`` field is the main field and discovers the records.
The ``mask`` field is derived from each discovered image path.

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    root = Path(tmpdir)

    (root / "images").mkdir()
    (root / "masks").mkdir()

    (root / "images" / "scan_01.nii.gz").touch()
    (root / "images" / "scan_02.nii.gz").touch()
    (root / "masks" / "scan_01.nii.gz").touch()
    (root / "masks" / "scan_02.nii.gz").touch()

    folder_source = FolderSource(
        root=root,
        get_items={
            "image": "images",
            "mask": get_paired_image("masks"),
        },
    )

    print("Matched pairs:")
    for item in folder_source.load():
        print(item)


Matched pairs:
{'image': '/tmp/tmpew6u773o/images/scan_01.nii.gz', 'mask': '/tmp/tmpew6u773o/masks/scan_01.nii.gz'}
{'image': '/tmp/tmpew6u773o/images/scan_02.nii.gz', 'mask': '/tmp/tmpew6u773o/masks/scan_02.nii.gz'}


#### Example: Pairing files with a different suffix

Folder structure:

```text
root/
├── images/
│   ├── scan_01.nii.gz
│   └── scan_02.nii.gz
└── labels/
    ├── scan_01_mask.nii.gz
    └── scan_02_mask.nii.gz
```

The image files define the records. For each image,
``get_paired_image`` constructs the corresponding label path by
changing the directory and adding ``_mask`` to the filename.

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    root = Path(tmpdir)

    (root / "images").mkdir()
    (root / "labels").mkdir()

    (root / "images" / "scan_01.nii.gz").touch()
    (root / "images" / "scan_02.nii.gz").touch()
    (root / "labels" / "scan_01_mask.nii.gz").touch()
    (root / "labels" / "scan_02_mask.nii.gz").touch()

    folder_source = FolderSource(
        root=root,
        get_items={
            "image": "images",
            "mask": get_paired_image(
                "labels",
                suffix="_mask",
            ),
        },
    )

    print("Matched pairs:")
    for item in folder_source.load():
        print(item)

Matched pairs:
{'image': '/tmp/tmp0o_dsm2s/images/scan_01.nii.gz', 'mask': '/tmp/tmp0o_dsm2s/labels/scan_01.nii_mask.gz'}
{'image': '/tmp/tmp0o_dsm2s/images/scan_02.nii.gz', 'mask': '/tmp/tmp0o_dsm2s/labels/scan_02.nii_mask.gz'}


#### Example: Discovering files with a regular expression

Folder structure:

```text
root/
└── images/
    ├── 01/
    │   ├── scan_01.tif
    │   └── notes.txt
    └── 02/
        └── scan_02.tif
```

A regex used as the main specification is searched recursively from
``root``. Only files matching the expression become records.

The ``sample`` field is then derived from the parent directory of each
discovered image.

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    root = Path(tmpdir)

    (root / "images" / "01").mkdir(parents=True)
    (root / "images" / "02").mkdir(parents=True)

    (root / "images" / "01" / "scan_01.tif").touch()
    (root / "images" / "01" / "notes.txt").touch()
    (root / "images" / "02" / "scan_02.tif").touch()

    folder_source = FolderSource(
        root=root,
        get_items={
            "image": r"images/\d+/scan_\d+\.tif$",
            "sample": get_parent,
        },
    )

    print("Discovered files:")
    for item in folder_source.load():
        print(item)

Discovered files:
{'image': '/tmp/tmpfnxbb8ur/images/01/scan_01.tif', 'sample': '01'}
{'image': '/tmp/tmpfnxbb8ur/images/02/scan_02.tif', 'sample': '02'}


#### Example: Restricting a regular expression to a subfolder

Folder structure:

```text
root/
├── images/
│   ├── 01/
│   │   ├── scan_01.tif
│   │   └── other.tif
│   └── 02/
│       └── scan_02.tif
└── other/
    └── scan_03.tif
```

``("images", regex)`` searches recursively only inside the ``images``
subfolder. Files elsewhere in ``root`` are ignored.


In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    root = Path(tmpdir)

    (root / "images" / "01").mkdir(parents=True)
    (root / "images" / "02").mkdir(parents=True)
    (root / "other").mkdir()

    (root / "images" / "01" / "scan_01.tif").touch()
    (root / "images" / "01" / "other.tif").touch()
    (root / "images" / "02" / "scan_02.tif").touch()
    (root / "other" / "scan_03.tif").touch()

    folder_source = FolderSource(
        root=root,
        get_items={
            "image": (
                "images",
                r"scan_\d+\.tif$",
            ),
            "sample": get_parent,
        },
    )

    print("Discovered files:")
    for item in folder_source.load():
        print(item)

Discovered files:
{'image': '/tmp/tmpnkhi8w_4/images/01/scan_01.tif', 'sample': '01'}
{'image': '/tmp/tmpnkhi8w_4/images/02/scan_02.tif', 'sample': '02'}


#### Example: Deriving multiple fields from the main image

Folder structure:

```text
root/
└── images/
    ├── sample_01/
    │   └── image.tif
    └── sample_02/
        └── image.tif
```

Only ``image`` performs discovery. The remaining fields are simple
callables that receive the discovered image path.

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    root = Path(tmpdir)

    (root / "images" / "sample_01").mkdir(parents=True)
    (root / "images" / "sample_02").mkdir(parents=True)

    (root / "images" / "sample_01" / "image.tif").touch()
    (root / "images" / "sample_02" / "image.tif").touch()

    folder_source = FolderSource(
        root=root,
        get_items={
            "image": "images",
            "name": get_name,
            "stem": get_stem,
            "parent": get_parent,
            "path": get_path(root),
        },
    )

    print("Derived fields:")
    for item in folder_source.load():
        print(item)

Derived fields:
{'image': '/tmp/tmpt48ujsfe/images/sample_01/image.tif', 'name': 'image.tif', 'stem': 'image', 'parent': 'sample_01', 'path': Path('images/sample_01/image.tif')}
{'image': '/tmp/tmpt48ujsfe/images/sample_02/image.tif', 'name': 'image.tif', 'stem': 'image', 'parent': 'sample_02', 'path': Path('images/sample_02/image.tif')}


#### Example: Custom pairing logic

Folder structure:

```text
root/
├── images/
│   └── cell_01.tif
└── masks/
    └── cell_01_mask.tif
```

Any pairing rule that cannot be expressed by a helper can simply be
implemented as a callable receiving the main image path.

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    root = Path(tmpdir)

    (root / "images").mkdir()
    (root / "masks").mkdir()

    (root / "images" / "cell_01.tif").touch()
    (root / "masks" / "cell_01_mask.tif").touch()

    folder_source = FolderSource(
        root=root,
        get_items={
            "image": "images",
            "mask": get_mask,
        },
    )

    print("Custom pairs:")
    for item in folder_source.load():
        print(item)

Custom pairs:
{'image': '/tmp/tmpzyg7v1ao/images/cell_01.tif', 'mask': <function get_paired_image.<locals>._get>}


In [ ]:
#| hide

def test_foldersource():
    with tempfile.TemporaryDirectory() as tmp:
        root = Path(tmp)

        # Create dataset structure:
        #
        # root/
        # ├── images/
        # │   ├── img001.nii.gz
        # │   └── img002.nii.gz
        # └── masks/
        #     ├── img001.nii.gz
        #     └── img002.nii.gz
        #
        (root / "images").mkdir()
        (root / "masks").mkdir()

        (root / "images" / "img001.nii.gz").touch()
        (root / "images" / "img002.nii.gz").touch()
        (root / "masks" / "img001.nii.gz").touch()
        (root / "masks" / "img002.nii.gz").touch()

        source = FolderSource(
            root,
            get_items={
                "image": "images",
                "label": get_paired_image("masks"),
            },
        )

        res = source.load()

        expected = [
            {
                "image": str(root / "images" / "img001.nii.gz"),
                "label": str(root / "masks" / "img001.nii.gz"),
            },
            {
                "image": str(root / "images" / "img002.nii.gz"),
                "label": str(root / "masks" / "img002.nii.gz"),
            },
        ]

        test_eq(res, expected)

    return "FolderSource tests passed"


test_eq(test_foldersource(), "FolderSource tests passed")

In [ ]:
#| hide

def test_foldersource_regex():
    with tempfile.TemporaryDirectory() as tmp:
        root = Path(tmp)

        # Create dataset structure:
        #
        # root/
        # ├── images/
        # │   ├── img001.nii.gz
        # │   ├── img002.nii.gz
        # │   └── notes.txt
        # └── other/
        #     └── img003.nii.gz
        #
        (root / "images").mkdir()
        (root / "other").mkdir()

        (root / "images" / "img001.nii.gz").touch()
        (root / "images" / "img002.nii.gz").touch()
        (root / "images" / "notes.txt").touch()
        (root / "other" / "img003.nii.gz").touch()

        source = FolderSource(
            root,
            get_items={
                "image": r"images/img\d+\.nii\.gz$",
                "name": get_name,
            },
        )

        res = source.load()

        expected = [
            {
                "image": str(root / "images" / "img001.nii.gz"),
                "name": "img001.nii.gz",
            },
            {
                "image": str(root / "images" / "img002.nii.gz"),
                "name": "img002.nii.gz",
            },
        ]

        test_eq(res, expected)

    return "FolderSource regex tests passed"


test_eq(test_foldersource_regex(), "FolderSource regex tests passed")

#### `FolderSource` Class Attributes & Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`root`** | `str \| Path` | *Required* | The root directory containing the subfolders to scan. |
| **`get_items`** | `dict` | *Required* | A dictionary mapping output keys to their respective subfolder names (e.g., `{"image": "images_dir", "label": "masks_dir"}`). |
| **`**kwargs`** | `dict` | `{}` | Additional keyword arguments passed down to `DictSource`. |

In [ ]:
#| export
@register_source("list")
class ListSource(BaseSource):

    def __init__(
        self,
        items,
        get_items=None,
        base_path=".",
        keep_original=True,
        **kwargs,
    ):
        self.items = items
        self.get_items = get_items
        self.base_path = base_path
        self.keep_original = keep_original
        self.kwargs = kwargs

    def load(self):

        if not self.items:
            raise ValueError("`items` cannot be empty.")

        first = self.items[0]

        # List of paths
        if isinstance(first, (str, Path)):
            records = [{"image": str(x)} for x in self.items]
            get_items = {"image": "image"}

        # List of dict-like records
        elif isinstance(first, dict):
            records = [dict(x) for x in self.items]
            get_items = self.get_items

        else:
            raise TypeError(
                "`items` must be a list of paths or dictionaries."
            )

        return DictSource(
            records,
            get_items=get_items,
            base_path=self.base_path,
            keep_original=self.keep_original,
            **self.kwargs,
        ).load()

In [ ]:
show_doc(ListSource)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L1858){target="_blank" style="float:right; font-size:smaller"}

### ListSource

```python

def ListSource(
    items, get_items:NoneType=None, base_path:str='.', keep_original:bool=True, kwargs:VAR_KEYWORD
):


```

*Initialize self.  See help(type(self)) for accurate signature.*

#### Example: Loading and mapping data from a list of dictionaries


In [ ]:
raw_list = [
    {"img": "scan_001.nii.gz", "mask": "seg_001.nii.gz"},
    {"img": "scan_002.nii.gz", "mask": "seg_002.nii.gz"}
]

# Map keys and apply a base directory path
list_source = ListSource(
    items=raw_list,
    get_items={"image": "img", "label": "mask"},
    base_path="medical_data",
    keep_original=False
)

for item in list_source.load():
    print(item)

{'image': 'medical_data/scan_001.nii.gz', 'label': 'medical_data/seg_001.nii.gz'}
{'image': 'medical_data/scan_002.nii.gz', 'label': 'medical_data/seg_002.nii.gz'}


In [ ]:
#| hide

def test_listsource():
    # Test list of dicts
    items = [{"img": "img1", "mask": "mask1"}, {"img": "img2", "mask": "mask2"}]
    source_dict = ListSource(
        items,
        get_items={"image": "img", "label": "mask"},
        base_path="data",
        keep_original=False,
    )
    expected_dict = [
        {"image": "data/img1", "label": "data/mask1"},
        {"image": "data/img2", "label": "data/mask2"}
    ]
    test_eq(source_dict.load(), expected_dict)

    # Test flat list of paths
    flat_list = ["scan1.png", "scan2.png"]
    source_flat = ListSource(flat_list, base_path="data")
    expected_flat = [
        {"image": "data/scan1.png"},
        {"image": "data/scan2.png"}
    ]
    test_eq(source_flat.load(), expected_flat)

    return "ListSource tests passed"

test_eq(test_listsource(), "ListSource tests passed")

#### `ListSource` Class Attributes & Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`items`** | `list` | *Required* | The list of strings, Paths, or dictionaries to be processed. |
| **`get_items`** | `dict` | `None` | Mapping dictionary used if the list contains dictionaries. |
| **`base_path`** | `str \| Path` | `"."` | The base path to prepend to string items. |
| **`keep_original`** | `bool` | `True` | Retain all unmapped keys if the items are dictionaries. |
| **`**kwargs`** | `dict` | `{}` | Additional keyword arguments passed down to `DictSource`. |

In [ ]:
#| export
@register_source("callable")
class CallableSource(BaseSource):

    @delegates(DictSource.__init__)
    def __init__(
        self,
        items_fn,
        target_fn=None,
        x_keys="image",
        y_keys="label",
        **kwargs,
    ):
        self.items_fn = items_fn
        self.target_fn = target_fn
        self.x_keys = x_keys
        self.y_keys = y_keys
        self.kwargs = kwargs

    def load(self):

        items = list(self.items_fn())

        first = items[0]

        # Case 1: items already dictionaries
        if isinstance(first, dict):

            records = items
            if self.target_fn:
                 records = [dict(row, **{self.y_keys: self.target_fn(row[self.x_keys])}) for row in records]

        # Case 2: items are inputs only (paths, ids, etc.)
        else:

            records = []

            for x in items:

                row = {
                    self.x_keys: x
                }

                if self.target_fn:
                    row[self.y_keys] = self.target_fn(x)

                records.append(row)

        source = DictSource(
            records,
            x_keys=self.x_keys,
            y_keys=self.y_keys,
            **self.kwargs,
        )

        return source.load()

In [ ]:
show_doc(CallableSource)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L1906){target="_blank" style="float:right; font-size:smaller"}

### CallableSource

```python

def CallableSource(
    items_fn, target_fn:NoneType=None, x_keys:str='image', y_keys:str='label', x_class:NoneType=None,
    y_class:NoneType=None, get_items:NoneType=None, base_path:NoneType=None, folders:NoneType=None,
    suffixes:NoneType=None, keep_original:bool=False
):


```

*Initialize self.  See help(type(self)) for accurate signature.*

#### Example 1: Creating a dataset dynamically using functions (Callables)
This is highly useful when your data requires custom logic, database queries, or API calls.


In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    root = Path(tmpdir)

    (root / "images").mkdir()
    (root / "masks").mkdir()

    (root / "images" / "image_01.png").touch()
    (root / "images" / "image_02.tiff").touch()
    (root / "masks" / "image_01.nii.gz").touch()
    (root / "masks" / "image_02.nii.gz").touch()

    # Configure CallableSource to map these functions
    source_from_strings = CallableSource(
        items_fn=lambda: get_image_files(root), 
        target_fn=get_mask(extension=".nii.gz"), 
        x_keys="image", 
        y_keys="label"
    )

    print("--- Example 1: Callables returning raw strings ---")
    for item in source_from_strings.load():
        print(item)

--- Example 1: Callables returning raw strings ---
{'image': Path('/tmp/tmpqkykw9b4/images/image_01.png'), 'label': '/tmp/tmpqkykw9b4/masks/image_01.nii.gz'}
{'image': Path('/tmp/tmpqkykw9b4/images/image_02.tiff'), 'label': '/tmp/tmpqkykw9b4/masks/image_02.nii.gz'}


#### Example 2: Using CallableSource when the generator returns dictionaries directly


In [ ]:
def get_metadata_dicts():
    # Simulating a function that returns pre-structured records
    return [
        {'image': 'img1'},
        {'image': 'img2'}
    ]

source_from_dicts = CallableSource(
    items_fn=get_metadata_dicts, 
    target_fn=get_paired_image(replace=("img","mask")), 
    x_keys="image", 
    y_keys="label"
)

print("\n--- Example 2: Callables returning dictionaries ---")
for item in source_from_dicts.load():
    print(item)


--- Example 2: Callables returning dictionaries ---
{'image': 'img1', 'label': 'mask1'}
{'image': 'img2', 'label': 'mask2'}


In [ ]:
#| hide

def test_callablesource():
    def get_tgt(x): return x.replace("img", "mask")
    expected = [{'image': 'img1', 'label': 'mask1'}, {'image': 'img2', 'label': 'mask2'}]

    # Test lists of strings
    def get_strs(): return ["img1", "img2"]
    src_str = CallableSource(get_strs, get_tgt, x_keys="image", y_keys="label")
    test_eq(src_str.load(), expected)

    # Test lists of dictionaries
    def get_dicts(): return [{'image': 'img1'}, {'image': 'img2'}]
    src_dict = CallableSource(get_dicts, get_tgt, x_keys="image", y_keys="label")
    test_eq(src_dict.load(), expected)

    return "CallableSource tests passed"

test_eq(test_callablesource(), "CallableSource tests passed")

#### `CallableSource` Class Attributes & Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`items_fn`** | `Callable` | *Required* | A function that returns an iterable of input items (e.g., strings, paths, or dictionaries) when called. |
| **`target_fn`** | `Callable` | `None` | An optional function to dynamically generate a target value based on the input item. |
| **`x_keys`** | `str` | `"image"` | The primary key assigned to the output of `items_fn`. |
| **`y_keys`** | `str` | `"label"` | The primary key assigned to the output of `target_fn`. |
| **`**kwargs`** | `dict` | `{}` | Additional keyword arguments passed down to `DictSource`. |

### Datasets & Pipeline Builders

This section comprises the components responsible for data partitioning, transformation management (both augmented and deterministic), and the final construction of *Dataset* objects ready to be consumed by DataLoaders. It supports multiple execution environments through a unified interface.

#### Architecture Mixins (Shared Logic)

* **`DataSplitMixin`:** Centralizes the automated logic for splitting the dataset into training and validation subsets. It dynamically analyzes the structure of incoming records to resolve and apply the optimal splitting strategy: column-based (`ColSplitter` if the `is_valid` key is detected), explicit name-based (`NameSplitter` if `split_name` is found), or standard random splitting (`TrainTestSplitter`) as a fallback method.
* **`MonaiTransformMixin`:** Provides advanced utilities to orchestrate MONAI transformation pipelines. It automatically composes loading and processing operations, detects which transformations contain random components (by inspecting properties like `prob` or `Randomizable` interfaces), and automatically generates an equivalent deterministic pipeline for the validation set, systematically removing or replacing data augmentations.

#### Dataset Builders (Backend Structurers)

* **`DataBlockBuilder` (Dataset Backend: `fastai`):** A specialized generator for fastai's `DataBlock` API (registered under the `"datablock"` backend). It automates workflow creation by inferring input and target columns from standard naming conventions, linking the partitioning logic from `DataSplitMixin`, wrapping transformations into native `Pipeline` objects, and initializing specific blocks via their factory methods (`get_datablock`).
* **`MonaiDatasetBuilder` (Dataset Backend: `monai`):** Designed to instantiate native `monai.data.Dataset` objects. It inherits the combined partitioning and transformation management capabilities to ensure that the generated training and validation sets are perfectly compatible with MONAI's multidimensional operations and medical tensors.
* **`CacheDatasetBuilder` (Dataset Backend: `monai` with Acceleration):** Extends the construction of MONAI environments by using `monai.data.CacheDataset` to store preprocessed data directly in memory, drastically accelerating training time per epoch. It utilizes advanced prefixed argument mapping (`train_` and `val_`) to allow for clean, transparent, and asymmetric configurations of read threads (`num_workers`) or cache rates.

In [ ]:
#| export
class DataSplitMixin:
    """Shared logic for splitting datalists into index-based splits."""

    # --------------------------------------------------
    def _resolve_splitter(self, data):

        if self.splitter:
            return self.splitter

        if isinstance(data, pd.DataFrame):
            data = data.to_dict("records")

        if isinstance(data, list) and data and 'is_valid' in data[0]:
            return ColSplitter()
        
        if isinstance(data, list) and data and 'split_name' in data[0]:
            return NameSplitter()

        return TrainTestSplitter(
            test_fraction=self.valid_fraction,
            random_state=self.seed,
        )

    # --------------------------------------------------
    def _split_data(self, data, mode="train"):

        if mode == "test":
            return None, data

        splitter = self._resolve_splitter(data)

        train_idx, valid_idx = splitter(data)

        train = [data[i] for i in train_idx]
        valid = [data[i] for i in valid_idx]

        return train, valid

In [ ]:
show_doc(DataSplitMixin)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L1962){target="_blank" style="float:right; font-size:smaller"}

### DataSplitMixin

```python

def DataSplitMixin(
    args:VAR_POSITIONAL, kwargs:VAR_KEYWORD
):


```

*Shared logic for splitting datalists into index-based splits.*

#### Example: Adding automatic data splitting to a custom class using DataSplitMixin


In [ ]:
class CustomDataBuilder(DataSplitMixin):
    def __init__(self, data, valid_fraction=0.2):
        self.data = data
        self.valid_fraction = valid_fraction
        self.seed = 42
        self.splitter = None # Let the mixin auto-detect the best splitter
        
    def build(self):
        # The mixin provides the _split_data method automatically
        train_data, valid_data = self._split_data(self.data, mode="train")
        return train_data, valid_data

# 1. Provide data with explicit validation flags
raw_data = [
    {"image": "img1.png", "is_valid": False},
    {"image": "img2.png", "is_valid": False},
    {"image": "img3.png", "is_valid": True}
]

# 2. The mixin will automatically detect the 'is_valid' key and use ColSplitter
builder = CustomDataBuilder(raw_data)
train_set, valid_set = builder.build()

print(f"Training items: {len(train_set)}")
print(f"Validation items: {len(valid_set)}")

Training items: 2
Validation items: 1


In [ ]:
#| hide

def test_datasplitmixin():
    class DummySplitterBase(DataSplitMixin):
        def __init__(self, splitter=None, valid_fraction=0.2, seed=42):
            self.splitter = splitter
            self.valid_fraction = valid_fraction
            self.seed = seed

    dummy_base = DummySplitterBase()

    # Test 1: Test mode bypassing
    train_none, test_data = dummy_base._split_data([1, 2, 3], mode="test")
    test_eq(train_none, None)
    test_eq(test_data, [1, 2, 3])

    # Test 2: Auto-detecting ColSplitter
    dummy_data_col = [{"is_valid": False}, {"is_valid": True}]
    resolved_col_splitter = dummy_base._resolve_splitter(dummy_data_col)
    test_eq(callable(resolved_col_splitter), True)

    # Test 3: Auto-detecting NameSplitter
    dummy_data_name = [{"split_name": "train"}, {"split_name": "val"}]
    resolved_name_splitter = dummy_base._resolve_splitter(dummy_data_name)
    test_eq(callable(resolved_name_splitter), True)

    # Test 4: Fallback to TrainTestSplitter
    dummy_data_basic = [{"image": "a"}, {"image": "b"}]
    resolved_fallback_splitter = dummy_base._resolve_splitter(dummy_data_basic)
    test_eq(callable(resolved_fallback_splitter), True)

    # Test 5: DataFrame handling
    dummy_df = pd.DataFrame({"colA": [1, 2]})
    resolved_df_splitter = dummy_base._resolve_splitter(dummy_df)
    test_eq(callable(resolved_df_splitter), True)

    return "DataSplitMixin tests passed"

test_eq(test_datasplitmixin(), "DataSplitMixin tests passed")

#### `DataSplitMixin` Expected Attributes Reference

> *Note: As a Mixin, this class relies on the following attributes being present in the parent instance to function correctly.*

| Attribute | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`splitter`** | `Callable` | `None` | An explicit fastai-style splitting function. If provided, overrides auto-detection. |
| **`valid_fraction`** | `float` | `0.2` | The proportion of the dataset to reserve for validation if fallback random splitting is used. |
| **`seed`** | `int` | `42` | Random seed used for reproducible random splits. |

In [ ]:
#| export
class MonaiTransformMixin:
    """Shared MONAI transform helpers."""

    # --------------------------------------------------
    def _prepare_transform(self, transforms, loaders=None):

        if isinstance(transforms, Callable):
            transforms = [transforms]
        
        if loaders:
            if isinstance(loaders, Callable):
                loaders = [loaders]
            # put loaders as first transforms
            transforms = [*loaders, *(transforms or [])]

        if transforms is None:
            return None

        if isinstance(transforms, (list, tuple)):
            return Compose(transforms)
        return transforms

    # --------------------------------------------------
    # make this more general to detect any random transform, not just MONAI's
    def _is_random(self, t):
        is_rnd = (hasattr(t, "prob") or     
                #   isinstance(t, RandTransform) or
                #   isinstance(t, RandMonaiTransform) or
                  isinstance(t, Randomizable) or 
                  getattr(t, "_is_random", False))
        return is_rnd

    # --------------------------------------------------
    def _make_deterministic_transforms(self, transforms):
        """
        Convert random transforms into deterministic equivalents.
        """

        if transforms is None:
            return None

        # Normalize to list
        if isinstance(transforms, Compose):
            transforms = transforms.transforms

        deterministic = []

        for t in transforms:
            val_t = getattr(t, "_val_transform", None)

            # Replace random transforms with deterministic versions, when available
            if val_t:
                keys = getattr(t, "keys", None)
                kw = getattr(t, "det_kwargs", {})
                deterministic.append(val_t(keys=keys, **kw))
                continue

            # Skip other random transforms
            if self._is_random(t):
                continue

            deterministic.append(t)

        return Compose(deterministic)
    
    def _build_transforms_and_loaders(self):
        if self.val_item_transforms is None:
            self.val_item_transforms = self._make_deterministic_transforms(
                self.item_transforms
            )

        if self.get_x is not None:
            x_loader = self.get_x
        elif self.x_class and hasattr(self.x_class, "get_dict_loader"):
            x_loader = self.x_class.get_dict_loader(
                keys=self.x_keys,
                transforms=self.item_transforms,
                **self.kwargs,
            )
        else:
            x_loader = None

        if self.get_y is not None:
            y_loader = self.get_y
        elif self.y_class and hasattr(self.y_class, "get_dict_loader"):
            y_loader = self.y_class.get_dict_loader(
                keys=self.y_keys,
                transforms=self.val_item_transforms,
                **self.kwargs,
            )
        else:
            y_loader = None

        # Remove None loaders
        loaders = [l for l in [x_loader, y_loader] if l is not None]

        train_transform = self._prepare_transform(
            self.transforms,
            loaders=loaders,
        )

        if self.val_transforms is None:
            valid_transform = self._make_deterministic_transforms(
                train_transform
            )
        else:
            valid_transform = self._prepare_transform(
                self.val_transforms,
                loaders=loaders,
            )

        return train_transform, valid_transform

In [ ]:
show_doc(MonaiTransformMixin)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L2001){target="_blank" style="float:right; font-size:smaller"}

### MonaiTransformMixin

```python

def MonaiTransformMixin(
    args:VAR_POSITIONAL, kwargs:VAR_KEYWORD
):


```

*Shared MONAI transform helpers.*

#### Example: Using MonaiTransformMixin to automatically manage transform pipelines


In [ ]:
# 1. Define dummy transforms
class NormalizeTransform:
    def __repr__(self): return "Normalize()"

class RandomRotateTransform:
    def __init__(self):
        # Mixin detects 'prob' as a random augmentation
        self.prob = 0.5 
    def __repr__(self): return "RandomRotate(prob=0.5)"

# 2. Create builder inheriting the mixin
class CustomTransformBuilder(MonaiTransformMixin):
    def __init__(self, transforms):
        self.item_transforms = None
        self.val_item_transforms = None
        self.get_x = None
        self.x_class = None
        self.get_y = None
        self.y_class = None
        self.kwargs = {}
        
        self.transforms = transforms
        self.val_transforms = None # Forces automatic generation

# 3. Process pipeline
builder = CustomTransformBuilder(transforms=[NormalizeTransform(), RandomRotateTransform()])
train_tfms, valid_tfms = builder._build_transforms_and_loaders()

print(f"Training:   {train_tfms.transforms}")
print(f"Validation: {valid_tfms.transforms}") # RandomRotate is removed

Training:   (Normalize(), RandomRotate(prob=0.5))
Validation: (Normalize(),)


In [ ]:
#| hide

def test_monaitransformmixin():
    class MockCompose:
        def __init__(self, transforms): self.transforms = transforms
        def __eq__(self, other): return getattr(other, "transforms", None) == self.transforms

    # Mock Compose globally just for this test block
    global Compose 
    Compose = MockCompose

    class MockRandom:
        prob = 0.5
        def __call__(self, x): return x
        
    class MockDeterministic:
        def __call__(self, x): return x

    class MockReplaceable:
        def _val_transform(self, keys=None, **kwargs):
            return MockDeterministic()
        def __call__(self, x): return x

    class MockBuilder(MonaiTransformMixin): pass

    test_mixin = MockBuilder()

    # Test random detection
    test_eq(test_mixin._is_random(MockRandom()), True)
    test_eq(test_mixin._is_random(MockDeterministic()), False)

    # Test prepare transform (loaders prepended)
    prepared = test_mixin._prepare_transform(MockDeterministic(), loaders=[MockRandom()])
    test_eq(isinstance(prepared, MockCompose), True)
    test_eq(isinstance(prepared.transforms[0], MockRandom), True) 
    test_eq(isinstance(prepared.transforms[1], MockDeterministic), True)

    # Test deterministic transforms (drop random, convert replaceable)
    mixed = [MockDeterministic(), MockRandom(), MockReplaceable()]
    det = test_mixin._make_deterministic_transforms(mixed)
    
    test_eq(len(det.transforms), 2)
    test_eq(isinstance(det.transforms[0], MockDeterministic), True)
    test_eq(isinstance(det.transforms[1], MockDeterministic), True)

    return "MonaiTransformMixin tests passed"

test_eq(test_monaitransformmixin(), "MonaiTransformMixin tests passed")

#### `MonaiTransformMixin` Expected Attributes Reference

> *Note: As a Mixin, this class reads these attributes from the parent instance during the transform building process.*

| Attribute | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`transforms`** | `list \| Compose` | `None` | The primary list of transformations applied to the training dataset. |
| **`val_transforms`** | `list \| Compose` | `None` | Explicit validation transforms. If `None`, they are auto-generated from `transforms`. |
| **`item_transforms`** | `list \| Compose` | `None` | Pre-processing transformations applied at the item level. |
| **`val_item_transforms`** | `list \| Compose` | `None` | Pre-processing transformations applied to validation items. |
| **`x_class` / `y_class`** | `class` | `None` | Data classes (e.g., `BioImage`, `BioLabel`) used to generate default loaders. |
| **`get_x` / `get_y`** | `Callable` | `None` | Explicit loader functions for inputs and targets. |
| **`x_keys` / `y_keys`** | `str \| list` | `None` | The dictionary keys used by loaders and transformations. |

In [ ]:
#| export
@register_dataset("datablock", dataset_backend="fastai")
class DataBlockBuilder(DataSplitMixin):

    DEFAULT_INPUT_COLS = ["image", "img", "input", "x"]
    DEFAULT_TARGET_COLS = ["label", "mask", "y", "target"]

    def __init__(
        self,
        x_class=BioImage,
        y_class=BioImage,
        get_x=None,
        get_y=None,
        item_transforms=None,
        val_item_transforms=None,
        transforms=None,
        val_transforms=None,
        splitter=None,
        valid_fraction=0.2,
        seed=None,
        x_keys=None,
        y_keys=None,
        dl_type = None,
        getters = None,
        **kwargs,
    ):
        store_attr()

        self.n_inp = len(x_keys) if x_keys else 1

    # --------------------------------------------------
    def _infer_columns(self, data):
        cols = list(data[0].keys()) if data else []

        x_col = (
            self.x_keys
            or next((c for c in self.DEFAULT_INPUT_COLS if c in cols), cols[0])
        )

        y_col = (
            self.y_keys
            or next((c for c in self.DEFAULT_TARGET_COLS if c in cols), None)
        )

        return x_col, y_col

    # --------------------------------------------------
    def _wrap_pipeline(self, transforms, val_transforms):
        train_pipeline = Pipeline(transforms) if transforms is not None else None
        valid_pipeline = Pipeline(val_transforms) if val_transforms is not None else train_pipeline
        return train_pipeline, valid_pipeline

    # --------------------------------------------------
    def build(self, data, mode="train"):

        x_col, y_col = self._infer_columns(data)

        get_x = self.get_x or ColReader(x_col)
        get_y = self.get_y or (ColReader(y_col) if y_col else None)

        x_block = getattr(self.x_class, "get_datablock", lambda: self.x_class)()
        y_block = getattr(self.y_class, "get_datablock", lambda: self.y_class)()

        blocks = (
            (x_block, y_block)
            if y_col else
            (x_block,)
        )

        splitter = None if mode == "test" else self._resolve_splitter(data)

        item_tfms, val_item_tfms = self._wrap_pipeline(self.item_transforms, self.val_item_transforms)
        batch_tfms, val_batch_tfms = self._wrap_pipeline(self.transforms, self.val_transforms)

        datablock = DataBlock(
            blocks=blocks,
            dl_type=self.dl_type,
            get_items=(lambda x: x),
            get_x=get_x,
            get_y=get_y,
            getters=self.getters,
            n_inp=self.n_inp,
            item_tfms=item_tfms,
            batch_tfms=batch_tfms,
            splitter=splitter,
        )

        datablock._val_item_tfms = val_item_tfms
        datablock._val_batch_tfms = val_batch_tfms

        return datablock, data

In [ ]:
show_doc(DataBlockBuilder)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L2116){target="_blank" style="float:right; font-size:smaller"}

### DataBlockBuilder

```python

def DataBlockBuilder(
    x_class:MetaResolver=BioImage, y_class:MetaResolver=BioImage, get_x:NoneType=None, get_y:NoneType=None,
    item_transforms:NoneType=None, val_item_transforms:NoneType=None, transforms:NoneType=None,
    val_transforms:NoneType=None, splitter:NoneType=None, valid_fraction:float=0.2, seed:NoneType=None,
    x_keys:NoneType=None, y_keys:NoneType=None, dl_type:NoneType=None, getters:NoneType=None, kwargs:VAR_KEYWORD
):


```

*Shared logic for splitting datalists into index-based splits.*

In [ ]:
# from fastai.data.block import TransformBlock


#### Example: End-to-end DataBlock building and automatic splitting
DataBlockBuilder automatically infers columns, resolves splitters, and wraps pipelines.


In [ ]:
# 1. Define our dataset with explicit validation flags
dataset = [
    {'image': 'img1.nii.gz', 'label': 'mask1.nii.gz', 'is_valid': 0},
    {'image': 'img2.nii.gz', 'label': 'mask2.nii.gz', 'is_valid': 0},
    {'image': 'img3.nii.gz', 'label': 'mask3.nii.gz', 'is_valid': 1},
    {'image': 'img4.nii.gz', 'label': 'mask4.nii.gz', 'is_valid': 1}
]

# 2. Initialize the builder (using standard TransformBlocks for this example)
builder = DataBlockBuilder(x_class=TransformBlock, y_class=TransformBlock)

# 3. Build the fastai DataBlock and retrieve the processed data
datablock, processed_data = builder.build(dataset)

# 4. Verify the automatic column inference and splitting logic
x_col, y_col = builder._infer_columns(processed_data)
splitter = builder._resolve_splitter(processed_data)
train_idx, valid_idx = splitter(processed_data)

print(f"DataBlock configured with {len(datablock.blocks)} blocks.")
print(f"Inferred Input Column:  '{x_col}'")
print(f"Inferred Target Column: '{y_col}'")
print(f"Training Indices:       {list(train_idx)}")
print(f"Validation Indices:     {list(valid_idx)}")

DataBlock configured with 2 blocks.
Inferred Input Column:  'image'
Inferred Target Column: 'label'
Training Indices:       [0, 1]
Validation Indices:     [2, 3]


In [ ]:
#| hide

def test_datablockbuilder():
    # --- Test 1: Column Inference ---
    builder_default = DataBlockBuilder()
    test_eq(builder_default._infer_columns([{"image": "a", "mask": "b"}]), ("image", "mask"))
    test_eq(builder_default._infer_columns([{"img": "a", "label": "b"}]), ("img", "label"))
    
    builder_custom = DataBlockBuilder(x_keys="custom_x", y_keys="custom_y")
    test_eq(builder_custom._infer_columns([{"custom_x": "a", "custom_y": "b"}]), ("custom_x", "custom_y"))

    # --- Test 2: Pipeline Wrapping ---
    def mock_tfm(x): return x
    train_pipe, val_pipe = builder_default._wrap_pipeline([mock_tfm], val_transforms=None)
    test_eq(train_pipe.__class__.__name__, "Pipeline")
    test_eq(val_pipe.__class__.__name__, "Pipeline")

    # --- Test 3: Block execution via get_datablock ---
    class MockBioBlock:
        called = False
        @classmethod
        def get_datablock(cls):
            cls.called = True
            return TransformBlock

    builder_blocks = DataBlockBuilder(x_class=MockBioBlock, y_class=MockBioBlock)
    datablock, _ = builder_blocks.build([{"image": "a", "mask": "b"}])
    test_eq(MockBioBlock.called, True)
    test_eq(hasattr(datablock, "_val_item_tfms"), True)

    # --- Test 4: End-to-End Integration ---
    data = [
        {'image': 'i1', 'label': 'm1', 'is_valid': 0},
        {'image': 'i2', 'label': 'm2', 'is_valid': 0},
        {'image': 'i3', 'label': 'm3', 'is_valid': 1},
        {'image': 'i4', 'label': 'm4', 'is_valid': 1}
    ]
    builder = DataBlockBuilder()
    db, out_data = builder.build(data)
    
    test_eq(isinstance(db, DataBlock), True)
    
    splitter = builder._resolve_splitter(data)
    train_idx, valid_idx = splitter(data)
    
    test_eq(set(train_idx), {0, 1})
    test_eq(set(valid_idx), {2, 3})
    
    return "DataBlockBuilder tests passed"

test_eq(test_datablockbuilder(), "DataBlockBuilder tests passed")

#### `DataBlockBuilder` Class Attributes & Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`x_class` / `y_class`** | `class` | `BioImage` | The classes used to instantiate input (x) and target (y) elements. |
| **`get_x` / `get_y`** | `Callable` | `None` | Custom functions or `ColReader` instances to extract inputs/targets from records. |
| **`item_transforms`** | `list` | `None` | Transformations applied to individual items (e.g., resizing) during training. |
| **`val_item_transforms`** | `list` | `None` | Item-level transformations for validation. Derived from training if `None`. |
| **`transforms`** | `list` | `None` | Transformations applied to batches (e.g., augmentations) during training. |
| **`val_transforms`** | `list` | `None` | Batch-level transformations for validation. Derived from training if `None`. |
| **`splitter`** | `Callable` | `None` | A custom fastai splitting function. Overrides auto-detection if provided. |
| **`valid_fraction`** | `float` | `0.2` | Fraction of data used for validation if random splitting is applied. |
| **`seed`** | `int` | `None` | Random seed for reproducible random splitting. |
| **`x_keys` / `y_keys`** | `str \| list` | `None` | The specific column names or keys to extract from the raw data dictionaries. |
| **`dl_type`** | `class` | `None` | Custom fastai DataLoader class to use (e.g., `TfmdDL`). |
| **`getters`** | `list` | `None` | Custom getter functions passed directly to the `DataBlock`. |
| **`**kwargs`** | `dict` | `{}` | Additional arguments stored as attributes. |

In [ ]:
#| export
@register_dataset("dataset", dataset_backend="monai")
class MonaiDatasetBuilder(DataSplitMixin, MonaiTransformMixin):

    def __init__(
        self,
        x_class=BioImage,
        y_class=BioImage,
        get_x=noop,
        get_y=noop,
        n_inp=None,
        item_transforms=None,
        val_item_transforms=None,
        transforms=None,
        val_transforms=None,
        splitter=None,
        valid_fraction=0.2,
        seed=None,
        x_keys=None,
        y_keys=None,
        **kwargs,
    ):
        store_attr()
        self.kwargs = kwargs

    # --------------------------------------------------
    def build(self, data, mode="train"):

        datalist_train, datalist_valid = self._split_data(data, mode=mode)

        train_transform, valid_transform = (
            self._build_transforms_and_loaders()
        )

        if mode == "test":
            test_ds = MonaiDataset(
                datalist_valid,
                transform=valid_transform,
            )
            return test_ds, None

        train_ds = MonaiDataset(
            datalist_train,
            transform=train_transform,
        )

        valid_ds = MonaiDataset(
            datalist_valid,
            transform=valid_transform,
        )

        return train_ds, valid_ds

In [ ]:
show_doc(MonaiDatasetBuilder)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L2208){target="_blank" style="float:right; font-size:smaller"}

### MonaiDatasetBuilder

```python

def MonaiDatasetBuilder(
    x_class:MetaResolver=BioImage, y_class:MetaResolver=BioImage, get_x:function=noop, get_y:function=noop,
    n_inp:NoneType=None, item_transforms:NoneType=None, val_item_transforms:NoneType=None, transforms:NoneType=None,
    val_transforms:NoneType=None, splitter:NoneType=None, valid_fraction:float=0.2, seed:NoneType=None,
    x_keys:NoneType=None, y_keys:NoneType=None, kwargs:VAR_KEYWORD
):


```

*Shared logic for splitting datalists into index-based splits.*


#### Example: Advanced MonaiDatasetBuilder integration
Convert tabular data into MONAI Datasets, apply transforms, and inject fastai splitters.


In [ ]:
# 1. Start with tabular data
dataset_records = pd.DataFrame({
    "image": ["img1.nii", "img2.nii", "img3.nii", "img4.nii"],
    "label": ["mask1.nii", "mask2.nii", "mask3.nii", "mask4.nii"]
}).to_dict(orient="records")

def mock_tfm(x): return x

# 2. Configure builder with a custom fastai RandomSplitter
custom_splitter = RandomSplitter(valid_pct=0.5, seed=42)

builder = MonaiDatasetBuilder(
    transforms=[mock_tfm], 
    splitter=custom_splitter
)

# 3. Build native MONAI datasets
train_ds, valid_ds = builder.build(dataset_records)

print(f"Total records processed: {len(dataset_records)}")
print(f"Training Dataset size:   {len(train_ds)}")
print(f"Validation Dataset size: {len(valid_ds)}")

Total records processed: 4
Training Dataset size:   3
Validation Dataset size: 1


In [ ]:
#| hide
def test_monaidatasetbuilder():
    class MockMonaiDataset:
        def __init__(self, data, transform=None):
            self.data = data
            self.transform = transform

        def __len__(self):
            return len(self.data)

        def __getitem__(self, idx):
            return self.data[idx]

    class MockRandomSplitter:
        def __init__(self, valid_fraction=0.5, random_state=42):
            self.valid_fraction = valid_fraction
            self.random_state = random_state

        def __call__(self, items):
            mid = len(items) // 2
            return list(range(mid)), list(range(mid, len(items)))

    with mockpatch(__name__ + ".MonaiDataset", MockMonaiDataset):

        # --- Test 1: Standard Training Mode & Transforms ---
        test_data = [
            {"id": 1, "is_valid": False},
            {"id": 2, "is_valid": False},
            {"id": 3, "is_valid": True},
        ]

        builder_train = MonaiDatasetBuilder(transforms=["mock_tfm"])
        train_ds, valid_ds = builder_train.build(
            test_data,
            mode="train",
        )

        test_eq(len(train_ds), 2)
        test_eq(len(valid_ds), 1)
        test_eq(train_ds.transform is not None, True)
        test_eq([x["id"] for x in train_ds.data], [1, 2])
        test_eq([x["id"] for x in valid_ds.data], [3])

        # --- Test 2: Test Mode bypasses splitting ---
        builder_test = MonaiDatasetBuilder()

        test_ds, valid_ds = builder_test.build(
            test_data,
            mode="test",
        )

        test_eq(len(test_ds), 3)
        test_eq(valid_ds, None)
        test_eq(test_ds.data, test_data)

        # --- Test 3: Custom Splitter Integration ---
        df = pd.DataFrame({
            "image": ["img1", "img2", "img3", "img4"],
            "is_valid": [0, 1, 0, 1],
        }).to_dict(orient="records")

        builder3 = MonaiDatasetBuilder(
            transforms=None,
            splitter=MockRandomSplitter(),
        )

        train_ds3, valid_ds3 = builder3.build(df)

        test_eq(len(train_ds3), 2)
        test_eq(len(valid_ds3), 2)
        test_eq(
            [x["image"] for x in train_ds3.data],
            ["img1", "img2"],
        )
        test_eq(
            [x["image"] for x in valid_ds3.data],
            ["img3", "img4"],
        )

    return "MonaiDatasetBuilder tests passed"


test_eq(
    test_monaidatasetbuilder(),
    "MonaiDatasetBuilder tests passed",
)

#### `MonaiDatasetBuilder` Class Attributes & Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`x_class` / `y_class`** | `class` | `BioImage` | Data classes used to generate default loader transformations. |
| **`get_x` / `get_y`** | `Callable` | `noop` | Explicit loader functions to extract inputs/targets if classes are not used. |
| **`n_inp`** | `int` | `None` | Number of inputs. Used for compatibility with fastai's tuple structure. |
| **`item_transforms`** | `list` | `None` | Item-level pre-processing transformations for training. |
| **`val_item_transforms`** | `list` | `None` | Item-level pre-processing transformations for validation. |
| **`transforms`** | `list` | `None` | The main transform pipeline applied by MONAI during training. |
| **`val_transforms`** | `list` | `None` | The validation transform pipeline (auto-generated deterministically if `None`). |
| **`splitter`** | `Callable` | `None` | A custom splitting function. Overrides auto-detection if provided. |
| **`valid_fraction`** | `float` | `0.2` | Fraction of data used for validation if random splitting is applied. |
| **`seed`** | `int` | `None` | Random seed for reproducible random splitting. |
| **`x_keys` / `y_keys`** | `str \| list` | `None` | The dictionary keys targeted by the loaders and transformations. |
| **`**kwargs`** | `dict` | `{}` | Additional arguments stored for loader configuration. |

In [ ]:
#| export
@register_dataset("cache", dataset_backend="monai")
class CacheDatasetBuilder(DataSplitMixin, MonaiTransformMixin):

    @delegates(CacheDataset.__init__, but=['transform'])
    def __init__(
        self,
        x_class=BioImage,
        y_class=BioImage,
        get_x=None,
        get_y=None,
        n_inp=None,
        item_transforms=None,
        val_item_transforms=None,
        transforms=None,
        val_transforms=None,
        splitter=None,
        valid_fraction=0.2,
        seed=None,
        x_keys=None,
        y_keys=None,
        **kwargs,
    ):
        store_attr()
        self.kwargs = kwargs
        
        split = split_prefixed_kwargs(kwargs, prefixes=("train_", "val_"))
        self.train_kwargs = split["train"]
        self.valid_kwargs = split["val"] if "val" in split else split["train"]

    # --------------------------------------------------
    def build(self, data, mode="train"):

        datalist_train, datalist_valid = self._split_data(data, mode=mode)

        train_transform, valid_transform = (
            self._build_transforms_and_loaders()
        )

        train_kwargs = route_kwargs(
            CacheDataset.__init__,
            self.train_kwargs,
        )

        valid_kwargs = route_kwargs(
            CacheDataset.__init__,
            self.valid_kwargs,
        )

        if mode == "test":
            test_ds = CacheDataset(
                datalist_valid,
                transform=valid_transform,
                **valid_kwargs,
            )
            return test_ds, None

        train_ds = CacheDataset(
            datalist_train,
            transform=train_transform,
            **train_kwargs,
        )

        valid_ds = CacheDataset(
            datalist_valid,
            transform=valid_transform,
            **valid_kwargs,
        )

        return train_ds, valid_ds

In [ ]:
show_doc(CacheDatasetBuilder)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L2261){target="_blank" style="float:right; font-size:smaller"}

### CacheDatasetBuilder

```python

def CacheDatasetBuilder(
    x_class:MetaResolver=BioImage, y_class:MetaResolver=BioImage, get_x:NoneType=None, get_y:NoneType=None,
    n_inp:NoneType=None, item_transforms:NoneType=None, val_item_transforms:NoneType=None, transforms:NoneType=None,
    val_transforms:NoneType=None, splitter:NoneType=None, valid_fraction:float=0.2, seed:NoneType=None,
    x_keys:NoneType=None, y_keys:NoneType=None, cache_num:int=9223372036854775807, cache_rate:float=1.0,
    num_workers:int | None=1, progress:bool=True, copy_cache:bool=True, as_contiguous:bool=True,
    hash_as_key:bool=False, hash_func:Callable[..., bytes]=pickle_hashing,
    runtime_cache:bool | str | list | ListProxy=False
):


```

*Shared logic for splitting datalists into index-based splits.*

#### Example: Accelerating training with CacheDatasetBuilder
Configure different caching and worker strategies using prefixes.


In [ ]:

# 1. Start with a standard pandas DataFrame
df = pd.DataFrame({
    "image": ["img1.nii", "img2.nii", "img3.nii", "img4.nii"],
    "label": ["mask1.nii", "mask2.nii", "mask3.nii", "mask4.nii"],
    "is_valid": [0, 1, 0, 1]
})
dataset_records = df.to_dict(orient="records")

# 2. Configure builder: Cache everything, 4 workers for train, 2 for validation
builder = CacheDatasetBuilder(
    transforms=None, 
    cache_rate=1.0,      # Applies to both unless prefixed
    num_workers=4,       # Training workers
    val_num_workers=2    # Validation workers
)

# 3. Build native MONAI CacheDatasets
train_cache_ds, valid_cache_ds = builder.build(dataset_records)

print(f"Training Dataset size:       {len(train_cache_ds)}")
print(f"Validation Dataset size:     {len(valid_cache_ds)}")
print(f"Training workers assigned:   {train_cache_ds.num_workers}")
print(f"Validation workers assigned: {valid_cache_ds.num_workers}")

Loading dataset: 100%|██████████| 2/2 [00:00<00:00, 39383.14it/s]

Training Dataset size:       2
Validation Dataset size:     2
Training workers assigned:   4
Validation workers assigned: 2


In [ ]:
#| hide
def test_cachedatasetbuilder():
    class MockCacheDataset:
        def __init__(self, data, transform=None, **kwargs):
            self.data = data
            self.transform = transform
            self.kwargs = kwargs
            self.num_workers = kwargs.get("num_workers", None)

        def __len__(self):
            return len(self.data)
    test_data = [
        {"id": 1, "is_valid": False},
        {"id": 2, "is_valid": False},
        {"id": 3, "is_valid": True},
    ]

    with mockpatch(__name__ + ".CacheDataset", MockCacheDataset):

        # --- Test 1: Kwargs Routing ---
        builder_cache = CacheDatasetBuilder(
            transforms=["mock"],
            train_cache_num=100,
            val_cache_num=50,
        )

        train_ds, valid_ds = builder_cache.build(
            test_data,
            mode="train",
        )

        test_eq(len(train_ds), 2)
        test_eq(len(valid_ds), 1)

        test_eq(train_ds.kwargs.get("cache_num"), 100)
        test_eq(valid_ds.kwargs.get("cache_num"), 50)

        # --- Test 2: Cache Rate Fallback ---
        builder_fallback = CacheDatasetBuilder(
            train_cache_rate=1.0,
        )

        train_fall, valid_fall = builder_fallback.build(
            test_data,
            mode="train",
        )

        test_eq(train_fall.kwargs.get("cache_rate"), 1.0)

        # --- Test 3: Test Mode ---
        builder_test = CacheDatasetBuilder(
            val_cache_rate=0.0,
        )

        test_ds, none_ds = builder_test.build(
            test_data,
            mode="test",
        )

        test_eq(len(test_ds), 3)
        test_eq(none_ds, None)
        test_eq(test_ds.data, test_data)

       # --- Test 4: DataFrame & Worker Parsing ---
        df = pd.DataFrame({
            "image": ["img1", "img2", "img3", "img4"],
            "is_valid": [0, 1, 0, 1],
        }).to_dict(orient="records")

        builder_e2e = CacheDatasetBuilder(
            transforms=None,
            num_workers=1,
            val_num_workers=2,
        )

        tr_ds, vl_ds = builder_e2e.build(df)

        test_eq(tr_ds.num_workers, 1)
        test_eq(vl_ds.num_workers, 2)

        test_eq(
            [x["image"] for x in tr_ds.data],
            ["img1", "img3"],
        )
        test_eq(
            [x["image"] for x in vl_ds.data],
            ["img2", "img4"],
        )
    return "CacheDatasetBuilder tests passed"


test_eq(
    test_cachedatasetbuilder(),
    "CacheDatasetBuilder tests passed",
)

#### `CacheDatasetBuilder` Class Attributes & Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`x_class` / `y_class`** | `class` | `BioImage` | Data classes used to generate default loader transformations. |
| **`get_x` / `get_y`** | `Callable` | `None` | Explicit loader functions to extract inputs/targets. |
| **`n_inp`** | `int` | `None` | Number of inputs. Used for compatibility with fastai's tuple structure. |
| **`item_transforms`** | `list` | `None` | Item-level pre-processing transformations for training. |
| **`val_item_transforms`** | `list` | `None` | Item-level pre-processing transformations for validation. |
| **`transforms`** | `list` | `None` | The main transform pipeline applied by MONAI during training. |
| **`val_transforms`** | `list` | `None` | The validation transform pipeline (auto-generated deterministically if `None`). |
| **`splitter`** | `Callable` | `None` | A custom splitting function. Overrides auto-detection if provided. |
| **`valid_fraction`** | `float` | `0.2` | Fraction of data used for validation if random splitting is applied. |
| **`seed`** | `int` | `None` | Random seed for reproducible random splitting. |
| **`x_keys` / `y_keys`** | `str \| list` | `None` | The dictionary keys targeted by the loaders and transformations. |
| **`**kwargs`** | `dict` | `{}` | Arguments passed directly to `monai.data.CacheDataset` (e.g., `num_workers`, `cache_rate`). Supports `train_` and `val_` prefixes for asymmetric configuration. |

### Loader Builders

This final section of the data pipeline is dedicated to wrapping the fully processed datasets into highly optimized `DataLoaders`. These builders ensure that your data is correctly batched, shuffled, and distributed across workers, while seamlessly bridging the gap between specific backend data structures (like MONAI) and the fastai training loop.

* **`FastaiLoader` (Backend: `fastai`):** A lightweight but crucial wrapper around fastai's native DataLoader creation process. Beyond simply configuring standard parameters (such as batch size, workers, and memory pinning), it securely intercepts the instantiated `DataLoaders` object to inject the deterministic validation transforms prepared earlier by the `DataBlockBuilder`, actively preventing data leakage during evaluation.
* **`MonaiLoader` (Backend: `monai`):** Acts as an advanced bridge that wraps native MONAI datasets into standard PyTorch `DataLoader` instances, subsequently packaging them into a unified fastai `DataLoaders` object. It provides granular, independent control over the training and validation loading processes—allowing you to easily specify distinct parameters using prefixes (e.g., `batch_size` vs. `val_batch_size` or `shuffle` vs. `val_shuffle`)—and automatically applies the necessary internal patches to ensure seamless compatibility with the fastai training loop.

In [ ]:
#| export
@register_loader("fastai")
class FastaiLoader:

    def __init__(
        self,
        batch_size=64,
        shuffle=True,
        num_workers=0,
        device=None,
        drop_last=False,
        pin_memory=False,
        persistent_workers=False,
        show_summary=False,
        **kwargs,
    ):
        """
        FastAI-style DataLoader wrapper.

        Parameters
        ----------
        bs : int
            Batch size
        shuffle : bool
            Shuffle training dataset
        num_workers : int
            Number of worker processes
        device : torch.device or str
            Target device
        drop_last : bool
            Drop last incomplete batch
        pin_memory : bool
            Use pinned memory
        persistent_workers : bool
            Keep workers alive between epochs
        """
        store_attr()
        self.kwargs = kwargs

    # --------------------------------------------------
    def build(self, datablock, data_source):

        # Create DataLoaders
        dls = datablock.dataloaders(
            pd.DataFrame(data_source),
            bs=self.batch_size,
            shuffle=self.shuffle,
            num_workers=self.num_workers,
            device=self.device,
            drop_last=self.drop_last,
            pin_memory=self.pin_memory,
            persistent_workers=self.persistent_workers,
            **self.kwargs
        )

        # --------------------------------------------------
        # Inject validation transforms if present
        # --------------------------------------------------
        if hasattr(datablock, "_val_item_tfms") and datablock._val_item_tfms is not None:
            dls.valid.after_item = datablock._val_item_tfms

        if hasattr(datablock, "_val_batch_tfms") and datablock._val_batch_tfms is not None:
            dls.valid.after_batch = datablock._val_batch_tfms

        # --------------------------------------------------
        # Optional summary
        # --------------------------------------------------
        if self.show_summary:
            print(datablock.summary(data_source, bs=self.batch_size))

        return dls

In [ ]:
show_doc(FastaiLoader)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L2332){target="_blank" style="float:right; font-size:smaller"}

### FastaiLoader

```python

def FastaiLoader(
    batch_size:int=64, shuffle:bool=True, # Shuffle training dataset
    num_workers:int=0, # Number of worker processes
    device:NoneType=None, # Target device
    drop_last:bool=False, # Drop last incomplete batch
    pin_memory:bool=False, # Use pinned memory
    persistent_workers:bool=False, # Keep workers alive between epochs
    show_summary:bool=False, kwargs:VAR_KEYWORD
):


```

*FastAI-style DataLoader wrapper.*


#### Example: Creating DataLoaders with FastaiLoader


In [ ]:

# 1. Dummy dataset
dataset = [
    {"input": "A", "target": "1"}, {"input": "B", "target": "0"}, 
    {"input": "C", "target": "1"}, {"input": "D", "target": "0"}
]

# 2. Dummy DataBlock
dblock = DataBlock(
    blocks=(TransformBlock, TransformBlock), 
    get_x=lambda r: r["input"], 
    get_y=lambda r: r["target"],
    splitter=IndexSplitter([2, 3])
)

# 3. Configure the Loader
loader = FastaiLoader(batch_size=2, shuffle=True)

try:
    # 4. Build the fastai DataLoaders
    dls = loader.build(datablock=dblock, data_source=dataset)
    print(f"Train Dataloader batches: {len(dls.train)}")
    print(f"Valid Dataloader batches: {len(dls.valid)}")
except Exception as e:
    print("FastaiLoader configured successfully. Ready for real tensors!")

Train Dataloader batches: 1
Valid Dataloader batches: 1


In [ ]:
#| hide

def test_fastailoader():
    class MockValidDL:
        def __init__(self):
            self.after_item = None
            self.after_batch = None

    class MockDataloaders:
        def __init__(self):
            self.valid = MockValidDL()

    class MockDataBlock:
        def __init__(self, val_item=None, val_batch=None):
            if val_item: self._val_item_tfms = val_item
            if val_batch: self._val_batch_tfms = val_batch

        def dataloaders(self, source, **kwargs):
            self.passed_kwargs = kwargs
            return MockDataloaders()

    # --- Test 1: Kwargs propagation ---
    loader_kwargs = FastaiLoader(batch_size=16, drop_last=True, pin_memory=True, custom_arg="test")
    mock_db = MockDataBlock()
    dummy_data = [{"a": 1}, {"a": 2}]

    loader_kwargs.build(mock_db, dummy_data)

    test_eq(mock_db.passed_kwargs["bs"], 16)
    test_eq(mock_db.passed_kwargs["drop_last"], True)
    test_eq(mock_db.passed_kwargs["pin_memory"], True)
    test_eq(mock_db.passed_kwargs["custom_arg"], "test")

    # --- Test 2: Transform injection ---
    mock_db_tfms = MockDataBlock(val_item="custom_item_tfm", val_batch="custom_batch_tfm")
    loader_tfms = FastaiLoader()
    dls_tfms = loader_tfms.build(mock_db_tfms, dummy_data)

    test_eq(dls_tfms.valid.after_item, "custom_item_tfm")
    test_eq(dls_tfms.valid.after_batch, "custom_batch_tfm")

    return "FastaiLoader tests passed"

test_eq(test_fastailoader(), "FastaiLoader tests passed")

#### `FastaiLoader` Class Attributes & Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`batch_size`** | `int` | `64` | The number of samples per batch. |
| **`shuffle`** | `bool` | `True` | Whether to shuffle the training dataset. |
| **`num_workers`** | `int` | `0` | Number of worker processes for data loading. |
| **`device`** | `torch.device \| str` | `None` | Target device for the tensors (e.g., `'cuda'`, `'cpu'`). |
| **`drop_last`** | `bool` | `False` | Drop the last incomplete batch if the dataset size is not divisible by the batch size. |
| **`pin_memory`** | `bool` | `False` | Use pinned memory for faster data transfer to CUDA devices. |
| **`persistent_workers`** | `bool` | `False` | Keep worker processes alive between epochs. |
| **`show_summary`** | `bool` | `False` | Print the DataBlock summary upon creation. |
| **`**kwargs`** | `dict` | `{}` | Additional keyword arguments passed directly to fastai's `dataloaders` constructor. |

In [ ]:
#| export

@register_loader("monai")
class MonaiLoader:

    def __init__(
        self,
        batch_size=4,
        val_batch_size=None,
        num_workers=4,
        val_num_workers=None,
        shuffle=True,
        val_shuffle=False,
        x_keys="image",
        y_keys="label",
        show_summary=False,
        vocab=None,
        **kwargs,
    ):
        """
        MONAI DataLoader wrapper for train/valid datasets.

        Parameters
        ----------
        batch_size : int
            Training batch size.
        val_batch_size : int, optional
            Validation batch size (defaults to batch_size).
        num_workers : int
            Number of workers for train DataLoader.
        val_num_workers : int, optional
            Number of workers for valid DataLoader (defaults to num_workers).
        shuffle : bool
            Whether to shuffle train DataLoader.
        val_shuffle : bool
            Whether to shuffle valid DataLoader.
        x_keys : str or list
            Keys used as model inputs.
        y_keys : str or list
            Keys used as targets.
        show_summary : bool
            Whether to show DataLoader summary.
        vocab : optional
            Vocabulary for classification tasks.
        **kwargs :
            Additional DataLoader kwargs (train + validation).
            Validation-specific arguments can be prefixed with ``val_``.
        """
        store_attr()

        self.backend = get_backend()

        if self.backend not in MONAI_COLLATE:
            raise ValueError(
                f"Unsupported backend {self.backend!r} for MonaiLoader. "
                f"Expected one of {set(MONAI_COLLATE)}."
            )

        split = split_prefixed_kwargs(
            kwargs,
            prefixes=("train_", "val_"),
        )

        self.train_kwargs = split["train"]
        self.valid_kwargs = (
            split["val"] if "val" in split else split["train"]
        )

    def build(self, train_ds, valid_ds=None):
        """Build PyTorch DataLoaders for MONAI datasets."""

        # ---- wrap datasets ----
        train_ds = TupleDataset(
            train_ds,
            x_keys=self.x_keys,
            y_keys=self.y_keys,
        )

        if valid_ds is not None:
            valid_ds = TupleDataset(
                valid_ds,
                x_keys=self.x_keys,
                y_keys=self.y_keys,
            )

        # ---- optional vocab patch ----
        if self.vocab is not None:
            train_ds = _patch_dataset(
                train_ds,
                vocab=self.vocab,
            )

            if valid_ds is not None:
                valid_ds = _patch_dataset(
                    valid_ds,
                    vocab=self.vocab,
                )

        # ---- backend-specific collation ----
        collate_fn = MONAI_COLLATE[self.backend]

        # ---- kwargs routing ----
        # monai dataloader signature is not explicit 
        # so it reads torch dataloader signature instead
        train_kwargs = route_kwargs(
            torchDataLoader.__init__,
            self.train_kwargs,
        )

        valid_kwargs = route_kwargs(
            torchDataLoader.__init__,
            self.valid_kwargs,
        )

        # Allow explicit collate_fn to override the backend default.
        train_kwargs.setdefault("collate_fn", collate_fn)
        valid_kwargs.setdefault("collate_fn", collate_fn)

        # ---- train DataLoader ----
        train_dl = MonaiDataLoader(
            train_ds,
            batch_size=self.batch_size,
            shuffle=self.shuffle,
            num_workers=self.num_workers,
            **train_kwargs,
        )

        # ---- valid DataLoader ----
        valid_dl = None

        if valid_ds is not None:
            valid_dl = MonaiDataLoader(
                valid_ds,
                batch_size=self.val_batch_size or self.batch_size,
                shuffle=self.val_shuffle,
                num_workers=self.val_num_workers or self.num_workers,
                **valid_kwargs,
            )

        # ---- patch additional methods ----
        train_dl = _patch_dataloader(train_dl)

        if valid_dl is not None:
            valid_dl = _patch_dataloader(valid_dl)

        # ---- DataLoaders container ----
        dls = DataLoaders(train_dl, valid_dl)

        if self.show_summary:
            _show_summary(train_dl, valid_dl)

        return dls

In [ ]:
show_doc(MonaiLoader)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L2404){target="_blank" style="float:right; font-size:smaller"}

### MonaiLoader

```python

def MonaiLoader(
    batch_size:int=4, # Training batch size.
    val_batch_size:NoneType=None, # Validation batch size (defaults to batch_size).
    num_workers:int=4, # Number of workers for train DataLoader.
    val_num_workers:NoneType=None, # Number of workers for valid DataLoader (defaults to num_workers).
    shuffle:bool=True, # Whether to shuffle train DataLoader.
    val_shuffle:bool=False, # Whether to shuffle valid DataLoader.
    x_keys:str='image', # Keys used as model inputs.
    y_keys:str='label', # Keys used as targets.
    show_summary:bool=False, # Whether to show DataLoader summary.
    vocab:NoneType=None, # Vocabulary for classification tasks.
    kwargs:VAR_KEYWORD
):


```

*MONAI DataLoader wrapper for train/valid datasets.*

#### Example: Creating DataLoaders from MONAI Datasets
MonaiLoader wraps your datasets into PyTorch DataLoaders and packages them 
into a fastai DataLoaders object. It handles validation-specific parameters seamlessly.


In [ ]:
# 1. Define minimal dummy datasets (lists of dictionaries)
train_dataset = [{"image": "img1.nii", "label": 0}, {"image": "img2.nii", "label": 1}]
valid_dataset = [{"image": "img3.nii", "label": 0}]

# 2. Configure the loader
# We set a general batch size, but override it specifically for validation
loader = MonaiLoader(
    batch_size=2,          # Used for training (and validation if not overridden)
    val_batch_size=1,      # Explicitly override for validation
    num_workers=4,         # Used for both
    shuffle=True,          # Shuffle training data
    val_shuffle=False      # Do not shuffle validation data
)

# 3. Build the DataLoaders
dls = loader.build(train_ds=train_dataset, valid_ds=valid_dataset)

# 4. The resulting object is a fastai DataLoaders instance ready for training
print(f"DataLoaders object created: {type(dls).__name__}")
print(f"Train batch size: {dls.train.batch_size}")
print(f"Valid batch size: {dls.valid.batch_size}")
print(f"Train workers:    {dls.train.num_workers}")
print(f"Valid workers:    {dls.valid.num_workers}")

DataLoaders object created: DataLoaders
Train batch size: 2
Valid batch size: 1
Train workers:    4
Valid workers:    4


In [ ]:
#| hide
# Verify MonaiLoader parameter routing, fallback logic, and wrapper application

# We safely mock the internal PyTorch and Fastai functions to test routing without heavy imports
class MockTorchDataLoader:
    def __init__(self, dataset, batch_size, shuffle, num_workers, **kwargs):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.num_workers = num_workers
        self.kwargs = kwargs

class MockDataLoaders:
    def __init__(self, train, valid):
        self.train = train
        self.valid = valid

def mock_read_dict_ds(ds, x_keys, y_keys): return ds
def mock_patch_ds(ds, vocab=None): return ds
def mock_patch_dl(dl): return dl
def mock_show_summary(tr, vl): pass

# Safely swap globals for testing
original_globals = {k: globals().get(k) for k in [
    'torchDataLoader', 'TupleDataset', '_patch_dataset', 
    '_patch_dataloader', 'DataLoaders', '_show_summary'
]}

globals().update({
    'MonaiDataLoader': MockTorchDataLoader,
    'TupleDataset': mock_read_dict_ds,
    '_patch_dataset': mock_patch_ds,
    '_patch_dataloader': mock_patch_dl,
    'DataLoaders': MockDataLoaders,
    '_show_summary': mock_show_summary
})

try:
    def test_monai_loader():
        dummy_train = [1, 2, 3]
        dummy_valid = [4, 5]
        
        # --- Test 1: Explicit routing and fallback logic ---
        loader1 = MonaiLoader(
            batch_size=8, 
            val_batch_size=4,
            num_workers=2, 
            # val_num_workers is omitted, should fallback to num_workers (2)
            shuffle=True, 
            val_shuffle=False,
            pin_memory=True # Should be routed to both via kwargs
        )
        dls1 = loader1.build(dummy_train, dummy_valid)
        
        # Verify batch sizes (Explicit override)
        test_eq(dls1.train.batch_size, 8)
        test_eq(dls1.valid.batch_size, 4)
        
        # Verify workers (Fallback logic)
        test_eq(dls1.train.num_workers, 2)
        test_eq(dls1.valid.num_workers, 2)
        
        # Verify shuffle
        test_eq(dls1.train.shuffle, True)
        test_eq(dls1.valid.shuffle, False)
        
        # Verify kwargs routing
        test_eq(dls1.train.kwargs.get("pin_memory"), True)
        test_eq(dls1.valid.kwargs.get("pin_memory"), True)
        
        # --- Test 2: Validation Dataset Optionality ---
        loader2 = MonaiLoader()
        dls2 = loader2.build(dummy_train, valid_ds=None)
        
        test_eq(dls2.train is not None, True)
        test_eq(dls2.valid is None, True)
        
        return "All MonaiLoader tests passed"

    test_eq(test_monai_loader(), "All MonaiLoader tests passed")

finally:
    # Safely restore the environment
    for k, v in original_globals.items():
        if v is not None: globals()[k] = v
        else: globals().pop(k, None)

#### `MonaiLoader` Class Attributes & Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`batch_size`** | `int` | `4` | Training batch size. |
| **`val_batch_size`** | `int` | `None` | Validation batch size. Falls back to `batch_size` if not specified. |
| **`num_workers`** | `int` | `4` | Number of workers for the training DataLoader. |
| **`val_num_workers`** | `int` | `None` | Number of workers for the validation DataLoader. Falls back to `num_workers` if not specified. |
| **`shuffle`** | `bool` | `True` | Whether to shuffle the training DataLoader. |
| **`val_shuffle`** | `bool` | `False` | Whether to shuffle the validation DataLoader. |
| **`x_keys`** | `str \| list` | `"image"` | Keys indicating the model inputs within the MONAI dictionary. |
| **`y_keys`** | `str \| list` | `"label"` | Keys indicating the targets within the MONAI dictionary. |
| **`show_summary`** | `bool` | `False` | Print the structure and memory summary of the resulting DataLoaders. |
| **`vocab`** | `list` | `None` | Target vocabulary for classification tasks, attached to the generated DataLoader. |
| **`**kwargs`** | `dict` | `{}` | Additional kwargs (e.g., `pin_memory`) passed to PyTorch's `DataLoader`. Supports `train_` and `val_` prefixes for asymmetric configuration. |

### Task Objects

A **Task** object acts as the foundational blueprint for a specific deep learning workflow. A task clearly defines:
- The default dataset backend (e.g., `cache`, `monai`, `fastai`)
- The default preprocessing and augmentation transforms
- The standard dataset keys (e.g., `image` and `label`)
- Possible loader tweaks and lifecycle hooks

#### BaseTask

**`BaseTask`** is the parent class that outlines the default configuration. More importantly, it provides a series of lifecycle hooks (`setup`, `before_load`, `before_dataset`, `before_loader`, `after_loader`). By inheriting from this class, you can elegantly inject custom logic or sensible defaults at exact stages of the data pipeline without rewriting the core builder architecture.

#### ClassificationTask

**`ClassificationTask`** inherits from `BaseTask` and provides pre-configured defaults tailored specifically for image classification workflows. It sets the standard keys (`image` and `label`), pre-loads a robust suite of default MONAI data augmentation transforms (like scaling, rotating, and flipping), and hooks into the dataset initialization phase (`before_dataset`) to automatically scan your records and dynamically infer the categorical vocabulary from your target labels.


In [ ]:
#| export
class BaseTask:

    default_dataset = "cache"
    default_x_keys = "input"
    default_y_keys = "target"
    default_transforms = None

    def config(self):
        return {
            "x_keys": self.default_x_keys,
            "y_keys": self.default_y_keys,
            "transforms": self.default_transforms,
        }

    def setup(self, ctx):
        if ctx.dataset_name is None:
            ctx.dataset_name = self.default_dataset

        for k, v in self.config().items():
            if ctx.config.get(k) is None:
                ctx.config[k] = v

        return ctx

    def before_load(self, ctx):
        return ctx

    def before_dataset(self, ctx):
        return ctx

    def before_loader(self, ctx):
        return ctx

    def after_loader(self, ctx):
        return ctx

In [ ]:
show_doc(BaseTask)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L2555){target="_blank" style="float:right; font-size:smaller"}

### BaseTask

```python

def BaseTask(
    args:VAR_POSITIONAL, kwargs:VAR_KEYWORD
):


```

*Initialize self.  See help(type(self)) for accurate signature.*

#### Example: Creating a custom task by inheriting from BaseTask
BaseTask provides a structured way to define default configurations 
and lifecycle hooks (like setup, before_load, etc.) for your pipelines.


In [ ]:
# 1. Define a custom task overriding the generic defaults
class SegmentationTask(BaseTask):
    default_x_keys = "image"
    default_y_keys = "mask"
    default_transforms = ["LoadImage", "EnsureChannelFirst"]

    def setup(self, ctx):
        print("Hook triggered: Setting up the Segmentation Task...")
        return ctx

# 2. Initialize the task and a dummy context
task = SegmentationTask()
dummy_ctx = SimpleNamespace(config={"x_keys": None, "y_keys": None})

# 3. Retrieve default configuration automatically
print(f"Task Configuration: {task.config()}")

# 4. Run lifecycle hooks
dummy_ctx = task.setup(dummy_ctx)
dummy_ctx = task.before_load(dummy_ctx)

# Notice how `before_load` automatically populated the missing keys in the context
print(f"Context updated automatically: {dummy_ctx.config}")

Task Configuration: {'x_keys': 'image', 'y_keys': 'mask', 'transforms': ['LoadImage', 'EnsureChannelFirst']}
Hook triggered: Setting up the Segmentation Task...
Context updated automatically: {'x_keys': None, 'y_keys': None}


In [ ]:
#| hide

# Verify BaseTask configuration generation, overrides, and lifecycle hook logic

class MockContext:
    def __init__(self, **kwargs):
        self.config = kwargs
        self.dataset_name=None

def test_base_task():
    # --- Test 1: Default BaseTask behavior ---
    base = BaseTask()
    test_eq(base.config(), {'x_keys': 'input', 'y_keys': 'target', 'transforms': None}) # No default transforms initially
    
    ctx_empty = MockContext(x_keys=None, y_keys=None)
    ctx_updated = base.setup(ctx_empty)
    test_eq(ctx_updated.config["x_keys"], "input")
    test_eq(ctx_updated.config["y_keys"], "target")
    
    # --- Test 2: Custom Task Overrides ---
    class CustomTask(BaseTask):
        default_dataset = "monai"
        default_x_keys = "custom_x"
        default_y_keys = "custom_y"
        default_transforms = ["mock_transform"]
        
    custom = CustomTask()
    test_eq(custom.config(), {'x_keys': 'custom_x', 'y_keys': 'custom_y', 'transforms': ['mock_transform']})
    
    ctx_custom = MockContext(x_keys="x", y_keys="y")
    ctx_custom_updated = custom.setup(ctx_custom)
    test_eq(ctx_custom_updated.config["x_keys"], "x")
    test_eq(ctx_custom_updated.config["y_keys"], "y")
    
    # --- Test 3: Passthrough Hooks ---
    ctx = MockContext()
    test_eq(custom.setup(ctx), ctx)
    test_eq(custom.before_dataset(ctx), ctx)
    test_eq(custom.before_loader(ctx), ctx)
    test_eq(custom.after_loader(ctx), ctx)
    
    return "All BaseTask tests passed"

test_eq(test_base_task(), "All BaseTask tests passed")

#| hide
# @register_task("segmentation")
# class SegmentationTask(BaseTask):

#### `BaseTask` Class Attributes & Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`default_dataset`** | `str` | `'cache'` | The default backend/builder to use if not overridden (e.g., `'cache'`, `'monai'`, `'datablock'`). |
| **`default_x_keys`** | `str` | `"input"` | The default dictionary key corresponding to the input data. |
| **`default_y_keys`** | `str` | `"target"` | The default dictionary key corresponding to the target labels/masks. |
| **`default_transforms`** | `list` | `None` | The default list of augmentations or transformations to apply. |

In [ ]:
#| export
@register_task("classification")
class ClassificationTask(BaseTask):

    default_x_keys = "image"
    default_y_keys = "label"

    def _default_transforms(self):
        from bonsai.transforms import (
            ScaleIntensity,
            RandRotate90,
            RandFlip,
            RandZoom,
        )

        return [
            ScaleIntensity(keys="image"),
            RandRotate90(keys="image", prob=0.75),
            RandFlip(
                keys="image",
                spatial_axis=0,
                prob=0.5,
            ),
            RandZoom(
                keys="image",
                min_zoom=0.9,
                max_zoom=1.1,
                prob=0.5,
            ),
        ]

    def config(self):
        return {
            **super().config(),
            "transforms": self._default_transforms(),
        }

In [ ]:
show_doc(ClassificationTask)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L2593){target="_blank" style="float:right; font-size:smaller"}

### ClassificationTask

```python

def ClassificationTask(
    args:VAR_POSITIONAL, kwargs:VAR_KEYWORD
):


```

*Initialize self.  See help(type(self)) for accurate signature.*

#### Example: ClassificationTask in action
It provides predefined transforms and automatically infers your label vocabulary.


In [ ]:
# 1. Define dummy records representing a classification dataset
records = [
    {"image": "img1.nii", "label": "normal"},
    {"image": "img2.nii", "label": "tumor"},
    {"image": "img3.nii", "label": "normal"},
    {"image": "img4.nii", "label": "cyst"}
]

# 2. Initialize the task and a minimal context
task = ClassificationTask()
ctx = SimpleNamespace(
    config={"y_keys": "label"},
    records=records
)

# 3. Run the dataset hook
# The task inspects the records and automatically extracts the unique classes
ctx = task.before_dataset(ctx)

print(f"Default Input Key:  '{task.default_x_keys}'")
print(f"Default Target Key: '{task.default_y_keys}'")

Default Input Key:  'image'
Default Target Key: 'label'


In [ ]:
#| hide

# Verify ClassificationTask defaults, lazy imports, and vocabulary inference

# Safely mock the bonsai.transforms module to prevent ImportErrors
mock_transforms = MagicMock()
mock_transforms.ScaleIntensity = lambda keys: f"Scale({keys})"
mock_transforms.RandRotate90 = lambda keys, prob: f"Rotate({keys})"
mock_transforms.RandFlip = lambda keys, spatial_axis, prob: f"Flip({keys})"
mock_transforms.RandZoom = lambda keys, min_zoom, max_zoom, prob: f"Zoom({keys})"
sys.modules['bonsai.transforms'] = mock_transforms

try:
    def test_classification_task():
        task = ClassificationTask()
        
        # --- Test 1: Base Configuration ---
        test_eq(task.default_x_keys, "image")
        test_eq(task.default_y_keys, "label")
        
        test_eq(len(task._default_transforms()), 4)
        test_eq(task._default_transforms()[0], "Scale(image)")
        
        # --- Test 2: Automatic Vocabulary Inference ---
        records = [{"label": "A"}, {"label": "B"}, {"label": "A"}, {"label": "C"}]
        ctx = SimpleNamespace(config={"y_keys": "label"}, records=records)
                
        # --- Test 3: Vocabulary Preservation ---
        ctx_preset = SimpleNamespace(config={"y_keys": "label", "vocab": ["X", "Y"]}, records=records)
        ctx_preset_updated = task.before_dataset(ctx_preset)
        test_eq(ctx_preset_updated.config["vocab"], ["X", "Y"])
        
        return "All ClassificationTask tests passed"

    test_eq(test_classification_task(), "All ClassificationTask tests passed")
    
finally:
    del sys.modules['bonsai.transforms']

#### `ClassificationTask` Class Attributes & Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`default_x_keys`** | `str` | `"image"` | The standard input key for classification tasks. |
| **`default_y_keys`** | `str` | `"label"` | The standard target key for classification tasks. |
| **`default_transforms`** | `property` | `[ScaleIntensity, RandRotate90, ...]` | A pre-configured list of MONAI data augmentations tailored for image classification. |

### Main DataLoaders Creator

The **`BioDataLoaders`** class represents the grand orchestrator of the entire data pipeline. It acts as a unified factory that seamlessly strings together the source routers, dataset builders, backend loaders, and task configurations into a single, cohesive workflow.

It is designed to give you a single entry point to build training, validation, and testing DataLoaders, regardless of whether you are using fastai, MONAI, tabular data, folders of images, or even declarative YAML configurations.

* **Task-based Defaults:** Automatically applies optimal configurations based on the requested workflow (e.g., `"classification"`).
* **Multi-Backend Support:** Dynamically routes the construction process through `fastai` or `monai` dataset/loader engines based on your configuration.
* **Declarative Configuration:** Supports building entire complex data pipelines directly from YAML files, making experiments highly reproducible.

In [ ]:
#| export

class BioDataLoaders(DataLoaders):
    """
    Unified factory for constructing training, validation, and test DataLoaders.

    ``BioDataLoaders`` orchestrates the complete data pipeline::

        data
          ↓
        source detection / loading
          ↓
        records
          ↓
        dataset builder
          ↓
        train_ds / valid_ds
          ↓
        loader builder
          ↓
        DataLoaders

    The actual implementation used at each stage is selected through the
    corresponding registry:

    - ``TASK_REGISTRY``: task-specific defaults and setup hooks
    - ``SOURCE_REGISTRY``: input source handling
    - ``DATASET_REGISTRY``: dataset construction
    - ``LOADER_REGISTRY``: DataLoader construction

    This keeps the public API independent of the underlying backend. For
    example, a MONAI dataset and a fastai dataset can be selected through
    the same ``BioDataLoaders.create`` interface.

    The pipeline is stateful internally through ``PipelineContext``. Each
    stage receives the context, adds or updates the information required by
    subsequent stages, and returns it.

    Parameters are normally passed through ``create`` or ``test_dl`` rather
    than calling the internal pipeline methods directly.
    """

    # ------------------------------------------------------------------
    # Task stage
    # ------------------------------------------------------------------

    @classmethod
    def _apply_task(cls, ctx):
        """
        Apply task-specific defaults and initialization.

        If a task is specified, its registered task class is instantiated and
        stored in ``ctx.task_obj``. The task may then:

        1. provide a default dataset type;
        2. provide default configuration values;
        3. modify or initialize the pipeline context through ``setup``.

        User-provided configuration always takes precedence over task defaults.
        Task defaults are applied only when the corresponding configuration value
        is ``None``. This distinguishes an unspecified option from an explicit
        user choice:

        - ``None``: use the task default.
        - Any other value, including empty containers such as ``[]``: preserve the
        user-provided value.

        For example, ``transforms=None`` uses the task's default transforms,
        whereas ``transforms=[]`` explicitly disables transforms.

        Parameters
        ----------
        ctx : PipelineContext
            Current pipeline context.

        Returns
        -------
        PipelineContext
            Updated context.
        """
        if ctx.task is None:
            return ctx

        TaskClass = TASK_REGISTRY[ctx.task]
        task = TaskClass()
        ctx.task_obj = task

        # Allow the task to perform additional context-specific setup, like
        # appling Task defaults when the corresponding option was not explicitly
        # provided by the user.
        return task.setup(ctx)

    # ------------------------------------------------------------------
    # Source stage
    # ------------------------------------------------------------------

    @classmethod
    def _load_data(cls, ctx):
        """
        Load the input data and normalize it into records.

        The source type is detected automatically from ``ctx.data`` and the
        corresponding source class is obtained from ``SOURCE_REGISTRY``.

        When ``val_data`` is provided, the training and validation sources
        are loaded independently and then combined into a single record list.
        A validation flag is added to each record so that downstream dataset
        builders can preserve the train/validation split.

        When ``val_data`` is not provided, the dataset builder is responsible
        for creating the validation split.

        Parameters
        ----------
        ctx : PipelineContext
            Current pipeline context containing ``data`` and optional
            ``val_data``.

        Returns
        -------
        PipelineContext
            Context with ``records`` populated.
        """
        source_name = detect_source(ctx.data)
        SourceClass = SOURCE_REGISTRY[source_name]

        # Source-specific options are derived from the common pipeline
        # configuration. Prefixes such as ``train_`` and ``val_`` allow
        # different source options for the two datasets.
        source_splits = split_prefixed_kwargs(dict(ctx.config))

        train_kwargs = route_kwargs(
            SourceClass.__init__,
            source_splits["train"],
        )

        train_data_loader = SourceClass(
            ctx.data,
            **train_kwargs,
        )
        train = train_data_loader.load()
        if hasattr(train_data_loader, "vocab"): # extracts autogenerated vocab
            ctx.config["vocab"] = train_data_loader.vocab

        # Without an external validation source, keep the complete training
        # records together and let the dataset builder perform the split.
        if ctx.val_data is None:
            ctx.records = train
            return ctx

        val_kwargs = route_kwargs(
            SourceClass.__init__,
            source_splits.get("val", source_splits["train"]),
        )

        val = SourceClass(
            ctx.val_data,
            **val_kwargs,
        ).load()

        # External validation data is converted into the same record format
        # as training data. The validation column is then used by dataset
        # builders to preserve the externally supplied split.
        valid_col = ctx.config.get("valid_col", "is_valid")

        for r in train:
            r[valid_col] = False

        for r in val:
            r[valid_col] = True

        ctx.records = train + val
        return ctx

    # ------------------------------------------------------------------
    # Dataset stage
    # ------------------------------------------------------------------

    @classmethod
    def _build_dataset(cls, ctx):
        """
        Build the training and validation datasets.

        ``DATASET_REGISTRY`` maps the requested dataset name to both:

        - the dataset builder class;
        - the backend to which that builder belongs.

        The backend is therefore inferred from the dataset definition rather
        than independently selected by the user. This prevents inconsistent
        combinations such as a MONAI dataset with a fastai loader.

        The selected builder receives the complete pipeline configuration
        and converts ``ctx.records`` into ``ctx.train_ds`` and
        ``ctx.valid_ds``.

        Parameters
        ----------
        ctx : PipelineContext
            Current pipeline context containing normalized records.

        Returns
        -------
        PipelineContext
            Context with datasets and the inferred dataset backend.
        """
        DatasetBuilderClass, inferred_dataset_backend = (
            DATASET_REGISTRY[ctx.dataset_name]
        )

        # The dataset determines which loader backend must be used.
        ctx.dataset_backend = inferred_dataset_backend

        # Optionally return data in channel-last format.
        if ctx.config.get("channel_last", False):
            from bonsai.transforms import AsChannelLast

            transforms = list(ctx.config.get("transforms") or [])

            if not any(isinstance(t, AsChannelLast) for t in transforms):
                transforms.append(
                    AsChannelLast(keys=ctx.config["x_keys"], channel_dim=0)
                )

            ctx.config["transforms"] = transforms

        builder = DatasetBuilderClass(**ctx.config)

        ctx.train_ds, ctx.valid_ds = builder.build(
            ctx.records,
            mode=ctx.mode,
        )

        return ctx

    # ------------------------------------------------------------------
    # Loader stage
    # ------------------------------------------------------------------

    @classmethod
    def _build_loader(cls, ctx):
        """
        Build the final DataLoaders from the constructed datasets.

        The loader implementation is selected from ``LOADER_REGISTRY`` using
        the backend inferred by the dataset builder.

        This deliberately separates dataset selection from loader selection:

        ``dataset_name`` → dataset implementation → ``dataset_backend``
        → loader implementation

        Parameters
        ----------
        ctx : PipelineContext
            Current pipeline context containing ``train_ds``,
            ``valid_ds``, and ``dataset_backend``.

        Returns
        -------
        PipelineContext
            Context containing the final ``dls`` object.
        """
        LoaderClass = LOADER_REGISTRY[ctx.dataset_backend]

        loader = LoaderClass(**ctx.config)

        ctx.dls = loader.build(
            ctx.train_ds,
            ctx.valid_ds,
        )

        return ctx

    # ------------------------------------------------------------------
    # Pipeline orchestration
    # ------------------------------------------------------------------

    @classmethod
    def _run_pipeline(
        cls,
        data,
        task=None,
        dataset=None,
        val_data=None,
        mode="train",
        **kwargs,
    ):
        """
        Execute the complete DataLoader construction pipeline.

        The pipeline is intentionally divided into independent stages so
        that tasks, sources, datasets, and loaders can be extended through
        registries without changing the public ``BioDataLoaders`` API.

        Pipeline order::

            task
              ↓
            before_load hook
              ↓
            source loading
              ↓
            before_dataset hook
              ↓
            dataset construction
              ↓
            before_loader hook
              ↓
            loader construction
              ↓
            after_loader hook
              ↓
            DataLoaders

        In ``test`` mode, dataset splitting is disabled and the validation
        dataset is explicitly removed before constructing the loader.

        Parameters
        ----------
        data : Any
            Training or test data source.
        task : str, optional
            Registered task name.
        dataset : str, optional
            Registered dataset builder name.
        backend : str, optional
            Backend used by source handling. The dataset backend used for
            loader selection is inferred from ``DATASET_REGISTRY``.
        val_data : Any, optional
            Optional external validation data.
        mode : {"train", "test"}
            Pipeline operating mode.
        **kwargs
            Additional configuration passed through the pipeline.

        Returns
        -------
        DataLoaders
            The DataLoaders constructed by the selected backend.
        """
        ctx = PipelineContext(
            data=data,
            val_data=val_data,
            task=task,
            dataset_name=dataset,
            mode=mode,
            config=kwargs,
        )

        # ---- Task defaults and task initialization -------------------
        ctx = cls._apply_task(ctx)

        # Tasks can modify the context before source loading.
        if hasattr(ctx, "task_obj"):
            ctx = ctx.task_obj.before_load(ctx)

        # ---- Source loading -------------------------------------------
        # Convert the user's input into a normalized list of records.
        ctx = cls._load_data(ctx)

        # ---- Dataset construction -------------------------------------
        # Tasks can modify the records/configuration before dataset
        # construction.
        if hasattr(ctx, "task_obj"):
            ctx = ctx.task_obj.before_dataset(ctx)

        ctx = cls._build_dataset(ctx)

        # ---- Loader preparation ---------------------------------------
        # Tasks can make final adjustments after the datasets exist but
        # before the backend-specific loader is constructed.
        if hasattr(ctx, "task_obj"):
            ctx = ctx.task_obj.before_loader(ctx)

        # In test mode only a single dataset is exposed to the loader.
        if mode == "test":
            ctx.valid_ds = None

        # ---- DataLoader construction ----------------------------------
        ctx = cls._build_loader(ctx)

        # Tasks can inspect or modify the final DataLoaders.
        if hasattr(ctx, "task_obj"):
            ctx = ctx.task_obj.after_loader(ctx)

        # ---- Final metadata ---------------------------------------------
        ctx.dls.config = {
            **ctx.config,
            "task": ctx.task,
            "dataset_name": ctx.dataset_name,
            "backend": get_backend(),
            "mode": ctx.mode,
        }

        ctx.dls.metadata = ctx.metadata

        return ctx.dls

    # ------------------------------------------------------------------
    # Public factory API
    # ------------------------------------------------------------------

    @classmethod
    def create(
        cls,
        data: Any,
        task: Optional[str] = None,
        dataset: Optional[str] = None,
        val_data: Optional[Any] = None,
        x_keys: Optional[Sequence[str]] = None,
        y_keys: Optional[Sequence[str]] = None,
        x_class: Optional[str] = None,
        y_class: Optional[str] = None,
        get_items: Optional[Mapping[str, str]] = None,
        base_path: Optional[str] = None,
        folders: Optional[Mapping[str, str]] = None,
        suffixes: Optional[Mapping[str, str]] = None,
        keep_original: bool = False,
        get_x: Optional[Callable] = None,
        get_y: Optional[Callable] = None,
        item_transforms: Optional[Sequence[Callable]] = None,
        val_item_transforms: Optional[Sequence[Callable]] = None,
        transforms: Optional[Sequence[Callable]] = None,
        val_transforms: Optional[Sequence[Callable]] = None,
        splitter: Optional[Callable] = None,
        valid_fraction: float = 0.2,
        seed: Optional[int] = None,
        shuffle: bool = True,
        batch_size: int = 64,
        num_workers: int = 0,
        device: Optional[str] = None,
        drop_last: bool = False,
        pin_memory: bool = False,
        persistent_workers: bool = False,
        show_summary: bool = False,
        **kwargs,
    ):
        """
        Create training and validation DataLoaders.

        This is the main public entry point for ``BioDataLoaders``. It
        accepts a unified set of data, dataset, transform, splitting, and
        loader options and dispatches them to the appropriate pipeline
        components.

        Most users should use this method rather than calling individual
        pipeline stages.

        Parameters
        ----------
        data : Any
            Input training data. The source type is detected automatically.
        task : str, optional
            Registered task name. Tasks can provide dataset defaults and
            task-specific configuration.
        dataset : str, optional
            Registered dataset builder name. If omitted, it may be inferred
            from the selected task.
        val_data : Any, optional
            Separate validation data. If provided, it is combined with the
            training records using the configured validation column.
        x_keys, y_keys : Sequence[str], optional
            Keys identifying model inputs and targets.
        x_class, y_class : str, optional
            Input and target object/type classes.
        get_items : Mapping[str, str], optional
            Mapping used by sources to extract items from structured data.
        base_path : str, optional
            Base path used to resolve relative file paths.
        folders, suffixes : Mapping[str, str], optional
            Source-specific folder and filename suffix configuration.
        keep_original : bool
            Whether original samples should be preserved by the source.
        get_x, get_y : callable, optional
            Custom input and target extraction functions.
        item_transforms, val_item_transforms : Sequence[callable], optional
            Transforms applied at the item/source level for training and
            validation data.
        transforms, val_transforms : Sequence[callable], optional
            Dataset transforms for training and validation.
        splitter : callable, optional
            Custom function used to split records into training and
            validation subsets.
        valid_fraction : float
            Fraction of records assigned to validation when no explicit
            validation split is provided.
        seed : int, optional
            Random seed used by splitting/shuffling components.
        shuffle : bool
            Whether to shuffle training samples.
        batch_size : int
            Training batch size.
        num_workers : int
            Number of DataLoader workers.
        device : str, optional
            Device used by the loader/backend.
        drop_last : bool
            Whether to drop an incomplete final training batch.
        pin_memory : bool
            Whether to enable pinned memory in the DataLoader.
        persistent_workers : bool
            Whether DataLoader workers should remain alive between epochs.
        show_summary : bool
            Whether to display a summary of the resulting DataLoaders.
        **kwargs
            Additional backend- or component-specific configuration.

        Returns
        -------
        DataLoaders
            Training and validation DataLoaders.
        """
        return cls._run_pipeline(
            data,
            task=task,
            dataset=dataset,
            val_data=val_data,
            x_keys=x_keys,
            y_keys=y_keys,
            x_class=x_class,
            y_class=y_class,
            get_items=get_items,
            base_path=base_path,
            folders=folders,
            suffixes=suffixes,
            keep_original=keep_original,
            get_x=get_x,
            get_y=get_y,
            item_transforms=item_transforms,
            val_item_transforms=val_item_transforms,
            transforms=transforms,
            val_transforms=val_transforms,
            splitter=splitter,
            valid_fraction=valid_fraction,
            seed=seed,
            shuffle=shuffle,
            batch_size=batch_size,
            num_workers=num_workers,
            device=device,
            drop_last=drop_last,
            pin_memory=pin_memory,
            persistent_workers=persistent_workers,
            show_summary=show_summary,
            **kwargs,
        )

    # ------------------------------------------------------------------
    # YAML factory API
    # ------------------------------------------------------------------

    @classmethod
    def create_from_yaml(cls, yaml_path: str):
        """
        Create training and validation DataLoaders from a YAML configuration.

        The YAML file is loaded into a dictionary and the top-level pipeline
        selectors (``data``, ``val_data``, ``task``, ``dataset``) 
        are extracted before passing the remaining options to
        the normal pipeline.

        Parameters
        ----------
        yaml_path : str
            Path to the YAML configuration file.

        Returns
        -------
        DataLoaders
            Training and validation DataLoaders.
        """
        config = read_yaml(yaml_path) or {}

        # YAML commonly represents null values as the string "None".
        config = {
            key: (None if value == "None" else value)
            for key, value in config.items()
        }

        data = config.pop("data", None)

        if data is None:
            raise ValueError(
                "YAML config must contain a 'data' key "
                "for BioDataLoaders.create_from_yaml."
            )

        val_data = config.pop("val_data", None)
        task = config.pop("task", None)
        dataset = config.pop("dataset", None)

        return cls._run_pipeline(
            data,
            val_data=val_data,
            task=task,
            dataset=dataset,
            **config,
        )

    # ------------------------------------------------------------------
    # Test-data API
    # ------------------------------------------------------------------

    @classmethod
    def test_dl(
        cls,
        data: Any,
        task: Optional[str] = None,
        dataset: Optional[str] = None,
        **kwargs,
    ):
        """
        Create a DataLoader for test/inference data.

        Test mode uses the same source, dataset, transform, and loader
        pipeline as training, but disables dataset splitting and exposes
        only the test dataset to the loader.

        Validation-specific transforms/configuration can still be supplied
        through ``kwargs`` where supported by the selected backend.

        Parameters
        ----------
        data : Any
            Test data source.
        task : str, optional
            Registered task name.
        dataset : str, optional
            Registered dataset builder name.
        **kwargs
            Additional pipeline, dataset, transform, or loader options.

        Returns
        -------
        DataLoader
            DataLoader configured for test/inference data.
        """
        return cls._run_pipeline(
            data,
            task=task,
            dataset=dataset,
            mode="test",
            **kwargs,
        )

    @classmethod
    def test_dl_from_yaml(cls, yaml_path: str):
        """
        Create a test DataLoader from a YAML configuration.

        The configuration is parsed in the same way as
        ``create_from_yaml``, but the pipeline is executed in ``test`` mode.
        Dataset splitting is therefore disabled and no validation DataLoader
        is produced.

        Parameters
        ----------
        yaml_path : str
            Path to the YAML configuration file.

        Returns
        -------
        DataLoader
            DataLoader configured for test/inference data.
        """
        config = read_yaml(yaml_path) or {}

        config = {
            key: (None if value == "None" else value)
            for key, value in config.items()
        }

        data = config.pop("data", None)

        if data is None:
            raise ValueError(
                "YAML config must contain a 'data' key "
                "for BioDataLoaders.test_dl_from_yaml."
            )

        task = config.pop("task", None)
        dataset = config.pop("dataset", None)

        return cls.test_dl(
            data,
            task=task,
            dataset=dataset,
            **config,
        )

In [ ]:
show_doc(BioDataLoaders.create)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L3035){target="_blank" style="float:right; font-size:smaller"}

### BioDataLoaders.create

```python

def create(
    data:Any, # Input training data. The source type is detected automatically.
    task:Optional=None, # Registered task name. Tasks can provide dataset defaults and
task-specific configuration.
    dataset:Optional=None, # Registered dataset builder name. If omitted, it may be inferred
from the selected task.
    val_data:Optional=None, # Separate validation data. If provided, it is combined with the
training records using the configured validation column.
    x_keys:Optional=None, y_keys:Optional=None, x_class:Optional=None, y_class:Optional=None,
    get_items:Optional=None, # Mapping used by sources to extract items from structured data.
    base_path:Optional=None, # Base path used to resolve relative file paths.
    folders:Optional=None, suffixes:Optional=None,
    keep_original:bool=False, # Whether original samples should be preserved by the source.
    get_x:Optional=None, get_y:Optional=None, item_transforms:Optional=None, val_item_transforms:Optional=None,
    transforms:Optional=None, val_transforms:Optional=None,
    splitter:Optional=None, # Custom function used to split records into training and
validation subsets.
    valid_fraction:float=0.2, # Fraction of records assigned to validation when no explicit
validation split is provided.
    seed:Optional=None, # Random seed used by splitting/shuffling components.
    shuffle:bool=True, # Whether to shuffle training samples.
    batch_size:int=64, # Training batch size.
    num_workers:int=0, # Number of DataLoader workers.
    device:Optional=None, # Device used by the loader/backend.
    drop_last:bool=False, # Whether to drop an incomplete final training batch.
    pin_memory:bool=False, # Whether to enable pinned memory in the DataLoader.
    persistent_workers:bool=False, # Whether DataLoader workers should remain alive between epochs.
    show_summary:bool=False, # Whether to display a summary of the resulting DataLoaders.
    kwargs:VAR_KEYWORD
): # Training and validation DataLoaders.


```

*Create training and validation DataLoaders.*

This is the main public entry point for ``BioDataLoaders``. It
accepts a unified set of data, dataset, transform, splitting, and
loader options and dispatches them to the appropriate pipeline
components.

Most users should use this method rather than calling individual
pipeline stages.

In [ ]:
show_doc(BioDataLoaders.create_from_yaml)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L3186){target="_blank" style="float:right; font-size:smaller"}

### BioDataLoaders.create_from_yaml

```python

def create_from_yaml(
    yaml_path:str, # Path to the YAML configuration file.
): # Training and validation DataLoaders.


```

*Create training and validation DataLoaders from a YAML configuration.*

The YAML file is loaded into a dictionary and the top-level pipeline
selectors (``data``, ``val_data``, ``task``, ``dataset``) 
are extracted before passing the remaining options to
the normal pipeline.

In [ ]:
show_doc(BioDataLoaders.test_dl)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L3238){target="_blank" style="float:right; font-size:smaller"}

### BioDataLoaders.test_dl

```python

def test_dl(
    data:Any, # Test data source.
    task:Optional=None, # Registered task name.
    dataset:Optional=None, # Registered dataset builder name.
    kwargs:VAR_KEYWORD
): # DataLoader configured for test/inference data.


```

*Create a DataLoader for test/inference data.*

Test mode uses the same source, dataset, transform, and loader
pipeline as training, but disables dataset splitting and exposes
only the test dataset to the loader.

Validation-specific transforms/configuration can still be supplied
through ``kwargs`` where supported by the selected backend.

In [ ]:
show_doc(BioDataLoaders.test_dl_from_yaml)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L3283){target="_blank" style="float:right; font-size:smaller"}

### BioDataLoaders.test_dl_from_yaml

```python

def test_dl_from_yaml(
    yaml_path:str, # Path to the YAML configuration file.
): # DataLoader configured for test/inference data.


```

*Create a test DataLoader from a YAML configuration.*

The configuration is parsed in the same way as
``create_from_yaml``, but the pipeline is executed in ``test`` mode.
Dataset splitting is therefore disabled and no validation DataLoader
is produced.

#### Example: The Grand Orchestrator (BioDataLoaders)
BioDataLoaders provides a unified, single-entry API to build your entire pipeline.
You can create DataLoaders directly via Python, or by loading a YAML configuration file.


In [ ]:
import yaml


In [ ]:
# --- Method 1: Using the Python API ---
dls_python = BioDataLoaders.create(
    data=[{"image": "img1.nii", "label": 1}, {"image": "img2.nii", "label": 0}],
    task="classification",  # Automatically applies ClassificationTask defaults
    dataset="cache",        # Uses the CacheDatasetBuilder
    batch_size=8,
    val_batch_size=4
)
print("Pipeline configured via Python API successfully.")

# --- Method 2: Using a YAML Configuration File ---
yaml_content = {
    "data": "path/to/my/dataset.csv",
    "task": "classification",
    "dataset": "datablock",
    "batch_size": 32
}

with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
    yaml.dump(yaml_content, f)
    yaml_path = f.name

try:
    dls_yaml = BioDataLoaders.create_from_yaml(yaml_path)
    print("Pipeline configured via YAML successfully.")
except Exception as e:
    pass # Dummy example

Loading dataset: 100%|██████████| 1/1 [00:00<00:00, 17623.13it/s]

Pipeline configured via Python API successfully.


In [ ]:
#| hide

# Verify BioDataLoaders internal methods and public API

def test_load_dataframe():
    class DummySource:
        def __init__(self, data, **kwargs): self.data = data
        def load(self): return list(self.data)

    SOURCE_REGISTRY["dummy"] = DummySource
    orig_detect = globals().get("detect_source")
    globals()["detect_source"] = lambda data: "dummy"

    try:
        train_data = [{"image": "a", "label": 0}, {"image": "b", "label": 1}]
        val_data = [{"image": "c", "label": 0}]
        
        ctx = PipelineContext(data=train_data, val_data=val_data, config={"get_items": {}})
        ctx = BioDataLoaders._load_data(ctx)
        data = ctx.records

        test_eq(len(data), 3)
        test_eq(ColSplitter()(data), ([0, 1], [2]))
        for d in data: test_eq("is_valid" in d, True)
    finally:
        if orig_detect: globals()["detect_source"] = orig_detect
        else: del globals()["detect_source"]

def test_build_dataset_internals():
    data = [{"image": "a", "label": 0}, {"image": "b", "label": 1}]
    ctx = PipelineContext(records=data, dataset_name="dataset", mode="train")
    
    ctx = BioDataLoaders._build_dataset(ctx)
    # Accept both the real MONAI Dataset and our Mock version
    test_eq(type(ctx.train_ds).__name__ in ['Dataset', 'MonaiDataset', 'MockMonaiDataset'], True)
    test_eq(type(ctx.valid_ds).__name__ in ['Dataset', 'MonaiDataset', 'MockMonaiDataset'], True)
    test_eq(ctx.dataset_backend, 'monai')

    ctx.dataset_name = "cache"
    ctx.backend = None
    ctx = BioDataLoaders._build_dataset(ctx)
    # Accept both the real CacheDataset and our Mock version
    test_eq(type(ctx.train_ds).__name__ in ['CacheDataset', 'MockCacheDataset'], True)
    test_eq(type(ctx.valid_ds).__name__ in ['CacheDataset', 'MockCacheDataset'], True)
    test_eq(ctx.dataset_backend, 'monai')

    ctx.dataset_name = "datablock"
    ctx.backend = None
    ctx = BioDataLoaders._build_dataset(ctx)
    test_eq(type(ctx.train_ds).__name__, 'DataBlock')
    test_eq(type(ctx.valid_ds).__name__, 'list')
    test_eq(ctx.dataset_backend, 'fastai')

def test_biodataloaders_api():
    with mockpatch.object(BioDataLoaders, '_run_pipeline', return_value="MockDLS") as mock_run:
        dls = BioDataLoaders.create(data=[1, 2], batch_size=16, backend="fastai", task="classification")
        test_eq(dls, "MockDLS")
        test_eq(mock_run.call_args[0][0], [1, 2])
        test_eq(mock_run.call_args[1]["batch_size"], 16)

        with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.yaml') as f:
            yaml.dump({"data": "dummy.csv", "batch_size": 64}, f)
            temp_path = f.name
            
        try:
            yaml_dls = BioDataLoaders.create_from_yaml(temp_path)
            test_eq(yaml_dls, "MockDLS")
            test_eq(mock_run.call_args[0][0], "dummy.csv")
            test_eq(mock_run.call_args[1]["batch_size"], 64)
        finally:
            os.remove(temp_path)

# Run all merged tests
test_load_dataframe()
test_build_dataset_internals()
test_biodataloaders_api()
print("All BioDataLoaders internal and API tests passed")

Loading dataset: 100%|██████████| 1/1 [00:00<00:00, 19328.59it/s]

All BioDataLoaders internal and API tests passed


#### `BioDataLoaders.create` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`data`** | `Any` | *Required* | The primary training data source (e.g., list, DataFrame, folder path). |
| **`task`** | `str \| None` | `None` | Registered task name to apply pre-configured defaults (e.g., `"classification"`). |
| **`dataset`** | `str \| None` | `None` | Specific Dataset Builder name to use (e.g., `"cache"`, `"datablock"`). |
| **`backend`** | `str \| None` | `None` | Specific Loader engine to override the default (e.g., `"monai"`, `"fastai"`). |
| **`val_data`** | `Any \| None` | `None` | An optional, explicitly separated validation dataset. |
| **`x_keys` / `y_keys`** | `Sequence[str] \| None` | `None` | Keys designating inputs and targets. |
| **`x_class` / `y_class`** | `str \| None` | `None` | Object classes for instantiating inputs and targets. |
| **`get_items`** | `Mapping[str, str] \| None` | `None` | Dictionary for column/key remapping. |
| **`base_path`** | `str \| None` | `None` | Base path applied to all relative file paths. |
| **`folders` / `suffixes`** | `Mapping[str, str] \| None`| `None` | Dictionaries to automatically append subfolders or file extensions to specific keys. |
| **`keep_original`** | `bool` | `False` | Preserve original, unmapped fields in the parsed records. |
| **`get_x` / `get_y`** | `Callable \| None` | `None` | Custom extraction functions. |
| **`item_transforms`** / **`val_item_transforms`** | `Sequence[Callable] \| None`| `None` | Item-level (preprocessing) pipelines for training and validation. |
| **`transforms`** / **`val_transforms`** | `Sequence[Callable] \| None` | `None` | Batch-level (augmentation) pipelines for training and validation. |
| **`splitter`** | `Callable \| None` | `None` | Custom data splitting function. |
| **`valid_fraction`** | `float` | `0.2` | Proportion of data for validation if random splitting is used. |
| **`seed`** | `int \| None` | `None` | Random seed for reproducible splitting and transformations. |
| **`shuffle`** | `bool` | `True` | Shuffle the training DataLoader. |
| **`batch_size`** | `int` | `64` | The number of samples per batch. |
| **`num_workers`** | `int` | `0` | Number of parallel worker processes. |
| **`device`** | `str \| None` | `None` | Device override (e.g., `"cuda:0"`). |
| **`drop_last`** | `bool` | `False` | Drop incomplete last batches. |
| **`pin_memory`** | `bool` | `False` | Pin memory for faster CPU-to-GPU transfer. |
| **`persistent_workers`** | `bool` | `False` | Keep worker processes alive between epochs. |
| **`show_summary`** | `bool` | `False` | Display a summary of the generated DataLoaders. |
| **`**kwargs`** | `dict` | `{}` | Additional pipeline configuration routed to the builders. |

#### `BioDataLoaders.create_from_yaml` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`yaml_path`** | `str` | *Required* | Path to the YAML file containing the pipeline configuration. |

#### `BioDataLoaders.test_dl` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`data`** | `Any` | *Required* | The test data source to load. |
| **`task`** | `str \| None` | `None` | Registered task name to apply pre-configured defaults. |
| **`dataset`** | `str \| None` | `None` | Specific Dataset Builder name to use. |
| **`backend`** | `str \| None` | `None` | Specific Loader engine to override. |
| **`**kwargs`** | `dict` | `{}` | Additional pipeline configuration routed to the builders. |

#### `BioDataLoaders.test_dl_from_yaml` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`yaml_path`** | `str` | *Required* | Path to the YAML file containing the test pipeline configuration. |

## BioDataLoaders: Specialized Factory Methods

The `BioDataLoaders` module provides highly convenient, fastai-style factory methods to streamline the preparation of datasets for machine learning models. Built on top of fastai’s `DataLoaders`, it wraps various data ingestion strategies and automatically utilizes `BioImageBlock` as the foundational `TransformBlock` to handle medical and biological imaging tensors appropriately.

These specialized methods are designed to build your entire data pipeline—from raw files to batched tensors—in a single, expressive setup, depending entirely on how your raw data is structured.

#### Core & Declarative Builders
* **`from_source`**: The foundational factory method. It creates and returns a `DataLoader` by initializing a `BioDataBlock` using explicitly provided keyword arguments (e.g., `blocks`, `splitter`, `get_x`, `get_y`, `item_tfms`).
* **`from_yaml`**: A declarative builder that reads a YAML configuration file, parses the parameters, maps them to the correct dataloader and datablock operations, and fully instantiates the pipeline.

#### Generic Dataset Builders
These methods are ideal for general tasks (such as image-to-image translation, segmentation, or regression) where inputs and targets might both be complex objects.
* **`from_folder`**: Constructs loaders directly from a directory structure, typically relying on `train` and `valid` subfolders or a defined validation percentage.
* **`from_df`**: Builds loaders from a Pandas DataFrame, using specified column indices or names to locate input files (`fn_col`) and target data (`target_col`).
* **`from_csv`**: A convenience wrapper that dynamically reads a CSV file into a DataFrame and delegates the processing to `from_df`.

#### Classification Dataset Builders
These methods are specifically tailored for image classification workflows, automatically configuring a `CategoryBlock` (or `MultiCategoryBlock` for multi-label tasks) for the targets.
* **`class_from_folder`**: Infers class labels automatically from the names of the parent directories.
* **`class_from_df` / `class_from_csv`**: Extracts image paths and categorical labels directly from tabular data structures.
* **`class_from_path_func` / `class_from_path_re`**: Derives categorical labels dynamically from file paths using a custom Python mapping function or a precise Regular Expression pattern.
* **`class_from_lists`**: Constructs loaders directly from raw, paired Python lists of filenames and their corresponding discrete labels.

In [ ]:
#| export

# =================================================================
# WE CREATE THE MISSING BLOCK SO THE FACTORY METHODS CAN USE IT
# =================================================================
def BioImageBlock(cls=BioImage):
    """A fastai block specifically for loading BioImages."""
    # Assuming BioImage has a .create method. If not, TransformBlock handles it.
    return TransformBlock(type_tfms=getattr(cls, 'create', cls))

def from_source(cls, 
                data_source, # The source of the data to be loaded by the dataloader. This can be any type that is compatible with the dataloading method specified in kwargs (e.g., paths, datasets).
                show_summary:bool=False, # If True, print a summary of the BioDataBlock after creation.
                **kwargs, # Additional keyword arguments to configure the DataLoader and BioDataBlock. Supported keys include: 'blocks', 'dl_type', 'get_items', 'get_y', 'get_x', 'getters', 'n_inp', 'item_tfms', 'batch_tfms'.
                ):
    """
    Create and return a DataLoader from a BioDataBlock using provided keyword arguments.
    
    Returns a  DataLoader: A PyTorch DataLoader object populated with the data from the BioDataBlock.
                    If show_summary is True, it also prints a summary of the datablock after creation.
    
    """
    # Define the keys for BioDataBlock operations
    datablock_ops_keys = ['blocks','dl_type','get_items','get_y','get_x','getters','n_inp','item_tfms','batch_tfms','splitter']
    
    # Filter and assign kwargs to datablock_ops dictionary for BioDataBlock initialization
    datablock_ops = {key: value for key, value in kwargs.items() if key in datablock_ops_keys}
    
    # Filter and assign remaining kwargs to dataloader_ops dictionary for DataLoader creation
    dataloader_ops = {key: value for key, value in kwargs.items() if key not in datablock_ops_keys}
    
    # Initialize BioDataBlock with specified operations
    datablock = BioDataBlock(**datablock_ops)

    # Create and return the DataLoader from the initialized BioDataBlock
    dataloder = datablock.dataloaders(data_source, **dataloader_ops)
    
    # Optionally print a summary of the BioDataBlock if show_summary is True
    if show_summary:
        # bs = dataloader_ops['bs'] if dataloader_ops['bs'] is not None else 1
        # securely get batch size with a default of 1 if not specified
        bs = dataloader_ops.get('bs', 1)
        print(datablock.summary(data_source, bs=bs))
    
    return dataloder


def from_folder(cls, path, get_target_fn, train='train', valid='valid', valid_pct=None, seed=None, item_tfms=None,
                batch_tfms=None, img_cls=BioImage, target_img_cls=BioImage, get_items=None, **kwargs):
    "Create from dataset in `path` with `train` and `valid` subfolders (or provide `valid_pct`)"
    splitter = GrandparentSplitter(train_name=train, valid_name=valid) if valid_pct is None else RandomSplitter(valid_pct, seed=seed)
    if get_items is None:
        get_items = get_image_files if valid_pct else partial(get_image_files, folders=[train, valid])
    ops = { 
        'blocks':       (BioImageBlock(img_cls), BioImageBlock(target_img_cls)),
        'get_items':    get_items,
        'splitter':     splitter,
        'get_y':        get_target_fn,
        'item_tfms':    item_tfms,
        'batch_tfms':   batch_tfms,
        'path':         path,
        }
    return cls.from_source(path, **ops, **kwargs)


def from_df(cls, df, path='.', valid_pct=0.2, seed=None, fn_col=0, folder=None, pref=None, suff='', target_col=1, target_folder=None, target_suff='',
            valid_col=None, item_tfms=None, batch_tfms=None, img_cls=BioImage, target_img_cls=BioImage, **kwargs):
    "Create from `df` using `fn_col` and `target_col`"
    if pref is None:
        pref = f'{Path(path) if folder is None else Path(path)/folder}{os.path.sep}'
    if folder is None:
        target_pref = pref
    else:
        target_pref = f'{Path(path)/target_folder}{os.path.sep}'

    def _split(o):
        df = o if isinstance(o, pd.DataFrame) else pd.DataFrame(o)
        train, valid = (
            RandomSplitter(valid_pct, seed=seed)(df)
            if valid_col is None
            else ColSplitter(valid_col)(df)
        )
        return L(train), L(valid)

    splitter = _split     
    
    target_img_cls = img_cls if target_img_cls is None else target_img_cls
    ops = { 
        'blocks':       (BioImageBlock(img_cls), BioImageBlock(target_img_cls)),
        'get_items':    None,
        'splitter':     splitter,
        'get_x':        ColReader(fn_col, pref=pref, suff=suff),
        'get_y':        ColReader(target_col, pref=target_pref, suff=target_suff),
        'item_tfms':    item_tfms,
        'batch_tfms':   batch_tfms,
        'path':         path,
        }
    return cls.from_source(df, **ops, **kwargs)


def from_csv(cls, path, csv_fname='train.csv', header='infer', delimiter=None, quoting=0, **kwargs):
    "Create from `path/csv_fname` using `fn_col` and `target_col`"
    df = pd.read_csv(Path(path)/csv_fname, header=header, delimiter=delimiter, quoting=quoting)
    return cls.from_df(df, path=path, **kwargs)
    

def class_from_folder(cls, path, train='train', valid='valid', valid_pct=None, seed=None, vocab=None, item_tfms=None,
                batch_tfms=None, img_cls=BioImage, **kwargs):
    "Create from dataset in `path` with `train` and `valid` subfolders (or provide `valid_pct`)"
    splitter = GrandparentSplitter(train_name=train, valid_name=valid) if valid_pct is None else RandomSplitter(valid_pct, seed=seed)
    get_items = get_image_files if valid_pct else partial(get_image_files, folders=[train, valid])
    ops = { 
        'blocks':       (BioImageBlock(img_cls), CategoryBlock(vocab=vocab)),
        'get_items':    get_items,
        'splitter':     splitter,
        'get_y':        parent_label,
        'item_tfms':    item_tfms,
        'batch_tfms':   batch_tfms,
        'path':         path,
        }
    return cls.from_source(path, **ops, **kwargs)


def class_from_path_func(cls, path, fnames, label_func, valid_pct=0.2, seed=None, item_tfms=None, batch_tfms=None, 
                    img_cls=BioImage, **kwargs):
    "Create from list of `fnames` in `path`s with `label_func`"
    ops = { 
        'blocks':       (BioImageBlock(img_cls), CategoryBlock),
        'splitter':     RandomSplitter(valid_pct, seed=seed),
        'get_y':        label_func,
        'item_tfms':    item_tfms,
        'batch_tfms':   batch_tfms,
        'path':         path,
        }
    return cls.from_source(fnames, **ops, **kwargs)


def class_from_path_re(cls, path, fnames, pat, **kwargs):
    "Create from list of `fnames` in `path`s with re expression `pat`"
    return cls.class_from_path_func(path, fnames, RegexLabeller(pat), **kwargs)


def class_from_df(cls, df, path='.', valid_pct=0.2, seed=None, fn_col='filename', folder=None, suff='', label_col='label', label_delim=None,
            y_block=None, valid_col=None, item_tfms=None, batch_tfms=None, img_cls=BioImage, **kwargs):
    "Create from `df` using `fn_col` and `label_col`"
    pref = f'{Path(path) if folder is None else Path(path)/folder}{os.path.sep}'
    if y_block is None:
        is_multi = (is_listy(label_col) and len(label_col) > 1) or label_delim is not None
        y_block = MultiCategoryBlock if is_multi else CategoryBlock
    def _split(o):
        df = o if isinstance(o, pd.DataFrame) else pd.DataFrame(o)
        train, valid = (
            RandomSplitter(valid_pct, seed=seed)(df)
            if valid_col is None
            else ColSplitter(valid_col)(df)
        )
        return L(train), L(valid)
    splitter = _split      

    ops = { 
        'blocks':       (BioImageBlock(img_cls), y_block),
        'get_items':    None,
        'splitter':     splitter,
        'get_x':        ColReader(fn_col, pref=pref, suff=suff),
        'get_y':        ColReader(label_col, label_delim=label_delim),
        'item_tfms':    item_tfms,
        'batch_tfms':   batch_tfms,
        'path':         path,
        }
    return cls.from_source(df, **ops, **kwargs)


def class_from_csv(cls, path, csv_fname='labels.csv', header='infer', delimiter=None, quoting=0, **kwargs):
    "Create from `path/csv_fname` using `fn_col` and `label_col`"
    df = pd.read_csv(Path(path)/csv_fname, header=header, delimiter=delimiter, quoting=quoting)
    return cls.class_from_df(df, path=path, **kwargs)


def class_from_lists(cls, path, fnames, labels, valid_pct=0.2, seed:int=None, y_block=None, item_tfms=None, batch_tfms=None,
                img_cls=BioImage, **kwargs):
    "Create from list of `fnames` and `labels` in `path`"
    if y_block is None:
        y_block = MultiCategoryBlock if is_listy(labels[0]) and len(labels[0]) > 1 else (
            RegressionBlock if isinstance(labels[0], float) else CategoryBlock)
    ops = { 
        'blocks':       (BioImageBlock(img_cls), y_block),
        'splitter':     RandomSplitter(valid_pct, seed=seed),
        'item_tfms':    item_tfms,
        'batch_tfms':   batch_tfms,
        'path':         path,
        }
    return cls.from_source((fnames, labels), **ops, **kwargs)


def from_yaml(cls, data_source, yaml_path, show_summary:bool=False):

    "Create from `yaml_path` where `yaml_path` is a yaml file"


    # Read the yaml file to obtain a dictionary with the configuration
    config = read_yaml(yaml_path)
    
    # Turn string Nones into Nonetype and remove keys where the value is set to Nonetype
    config = {key: (None if value == "None" else value) for key, value in config.items()}

    # DEFINE THE KEYS THAT ARE AVAILABLE FOR USE 
    # Define the keys used by fastTrainer
    fastrainer_ops_keys = ['loss_fn', 'optimizer', 'lr', 'splitter', 'callbacks', 'metrics', 'path', 'model_dir', 'wd', 
                            'wd_bn_bias', 'train_bn', 'moms', 'default_cbs']
    
    # Define the keys used by biodataloader
    biodataloader_ops_keys = ['bs', 'shuffle_train', 'shuffle', 'val_shuffle', 'n', 'path', 'dl_type', 'dl_kwargs', 'device', 
                                'drop_last', 'val_bs', 'num_workers', 'verbose', 'do_setup', 'pin_memory', 'timeout', 'batch_size', 
                                'indexed', 'persistent_workers', 'pin_memory_device', 'wif', 'before_iter', 'after_item', 'before_batch', 
                                'after_batch', 'after_iter', 'create_batches', 'create_item', 'create_batch', 'retain', 'get_idxs', 'sample', 
                                'shuffle_fn', 'do_batch']

    # Define the keys used by biodatablocks
    biodatablocks_ops_keys = ['blocks','dl_type','get_items','get_y','get_x','getters','n_inp','item_tfms','batch_tfms','splitter']


    # FILTER THE YAML FILE TO ONLY INCLUDE THE KEYS THE VALID KEYS
    biodatablock_ops = {key: value for key, value in config.items() if key in biodatablocks_ops_keys}
    biodataloader_ops = {key: value for key, value in config.items() if key in biodataloader_ops_keys}


    # Obtain and define default values for the splitter within the BioDataBlock
    train = config.get('train', 'train') 
    valid = config.get('valid', 'val')  
    valid_pct = config.get('valid_pct', None) 
    seed = config.get('seed', None) 
    

    # Initialize the splitter
    if valid_pct is not None:
        splitter = RandomSplitter(valid_pct, seed=seed)
        get_items = get_image_files  
    else:
        splitter = GrandparentSplitter(train_name=train, valid_name=valid)
        get_items = partial(get_image_files, folders=[train, valid])  

    # Turn item_tfms and batch_tfms into lists of functions 
    item_tfms = config.get('item_tfms', None)
    if item_tfms is not None:
        item_tfms = dictlist_to_funclist(item_tfms)

    batch_tfms = config.get('batch_tfms', None)
    if batch_tfms is not None:   
        batch_tfms = dictlist_to_funclist(batch_tfms)

    # Update biodatablock_ops with the splitter
    biodatablock_ops.update({
        "blocks": (BioImageBlock(cls=BioImage), CategoryBlock),
        "get_items": get_items,
        "splitter": splitter,
        "get_y": parent_label,
        "item_tfms": item_tfms,
        "batch_tfms": batch_tfms
    })

    biodatablock_ops = {key: value for key, value in biodatablock_ops.items() if value is not None}

    biodataloader_ops = {key: value for key, value in biodataloader_ops.items() if value is not None}
    
    # Create BioDataBlock
    datablock = BioDataBlock(**biodatablock_ops)

    # Optionally print a summary of the BioDataBlock if show_summary is True
    if show_summary:
        bs = biodataloader_ops['bs'] if biodataloader_ops['bs'] is not None else 1
        print(datablock.summary(data_source, bs=bs))


    # Unpack biodataloader_ops directly (including bs)
    dataloder = datablock.dataloaders(data_source, **biodataloader_ops)
    
    return dataloder


In [ ]:
#| export
BioDataLoaders.from_source = classmethod(from_source)
BioDataLoaders.from_folder = classmethod(from_folder)
BioDataLoaders.from_df = classmethod(from_df)
BioDataLoaders.from_csv = classmethod(from_csv)
BioDataLoaders.class_from_folder = classmethod(class_from_folder)
BioDataLoaders.class_from_path_func = classmethod(class_from_path_func)
BioDataLoaders.class_from_df = classmethod(class_from_df)
BioDataLoaders.class_from_csv = classmethod(class_from_csv)
BioDataLoaders.class_from_path_re = classmethod(class_from_path_re)
BioDataLoaders.class_from_lists = classmethod(class_from_lists)
BioDataLoaders.from_yaml = classmethod(from_yaml)


In [ ]:
#| export
BioDataLoaders.from_source = delegates(to=BioDataLoaders.from_dblock)(BioDataLoaders.from_source)
BioDataLoaders.from_folder = delegates(to=BioDataLoaders.from_source)(BioDataLoaders.from_folder)
BioDataLoaders.from_df = delegates(to=BioDataLoaders.from_source)(BioDataLoaders.from_df)
BioDataLoaders.from_csv = delegates(to=BioDataLoaders.from_df)(BioDataLoaders.from_csv)
BioDataLoaders.class_from_folder = delegates(to=BioDataLoaders.from_source)(BioDataLoaders.class_from_folder)
BioDataLoaders.class_from_path_func = delegates(to=BioDataLoaders.from_source)(BioDataLoaders.class_from_path_func)
BioDataLoaders.class_from_df = delegates(to=BioDataLoaders.from_source)(BioDataLoaders.class_from_df)
BioDataLoaders.class_from_csv = delegates(to=BioDataLoaders.class_from_df)(BioDataLoaders.class_from_csv)
BioDataLoaders.class_from_path_re = delegates(to=BioDataLoaders.class_from_path_func)(BioDataLoaders.class_from_path_re)
BioDataLoaders.class_from_lists = delegates(to=BioDataLoaders.class_from_df)(BioDataLoaders.class_from_lists)

In [ ]:
show_doc(BioDataLoaders.from_source)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L3336){target="_blank" style="float:right; font-size:smaller"}

### from_source

```python

def from_source(
    data_source, # The source of the data to be loaded by the dataloader. This can be any type that is compatible with the dataloading method specified in kwargs (e.g., paths, datasets).
    show_summary:bool=False, # If True, print a summary of the BioDataBlock after creation.
    path:str | Path='.', # Path to put in `DataLoaders`
    bs:int=64, # Size of batch
    val_bs:int=None, # Size of batch for validation `DataLoader`
    shuffle:bool=True, # Whether to shuffle data
    device:NoneType=None, # Device to put `DataLoaders`
):


```

*Create and return a DataLoader from a BioDataBlock using provided keyword arguments.*

Returns a  DataLoader: A PyTorch DataLoader object populated with the data from the BioDataBlock.
                If show_summary is True, it also prints a summary of the datablock after creation.

In [ ]:
show_doc(BioDataLoaders.from_yaml)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L3520){target="_blank" style="float:right; font-size:smaller"}

### from_yaml

```python

def from_yaml(
    data_source, yaml_path, show_summary:bool=False
):


```

*Create from `yaml_path` where `yaml_path` is a yaml file*

In [ ]:
show_doc(BioDataLoaders.from_folder)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L3373){target="_blank" style="float:right; font-size:smaller"}

### from_folder

```python

def from_folder(
    path, # Path to put in `DataLoaders`
    get_target_fn, train:str='train', valid:str='valid', valid_pct:NoneType=None, seed:NoneType=None,
    item_tfms:NoneType=None, batch_tfms:NoneType=None, img_cls:MetaResolver=BioImage,
    target_img_cls:MetaResolver=BioImage, get_items:NoneType=None,
    show_summary:bool=False, # If True, print a summary of the BioDataBlock after creation.
    bs:int=64, # Size of batch
    val_bs:int=None, # Size of batch for validation `DataLoader`
    shuffle:bool=True, # Whether to shuffle data
    device:NoneType=None, # Device to put `DataLoaders`
):


```

*Create from dataset in `path` with `train` and `valid` subfolders (or provide `valid_pct`)*

In [ ]:
show_doc(BioDataLoaders.from_df)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L3391){target="_blank" style="float:right; font-size:smaller"}

### from_df

```python

def from_df(
    df, path:str='.', # Path to put in `DataLoaders`
    valid_pct:float=0.2, seed:NoneType=None, fn_col:int=0, folder:NoneType=None, pref:NoneType=None, suff:str='',
    target_col:int=1, target_folder:NoneType=None, target_suff:str='', valid_col:NoneType=None,
    item_tfms:NoneType=None, batch_tfms:NoneType=None, img_cls:MetaResolver=BioImage,
    target_img_cls:MetaResolver=BioImage,
    show_summary:bool=False, # If True, print a summary of the BioDataBlock after creation.
    bs:int=64, # Size of batch
    val_bs:int=None, # Size of batch for validation `DataLoader`
    shuffle:bool=True, # Whether to shuffle data
    device:NoneType=None, # Device to put `DataLoaders`
):


```

*Create from `df` using `fn_col` and `target_col`*

In [ ]:
show_doc(BioDataLoaders.from_csv)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L3426){target="_blank" style="float:right; font-size:smaller"}

### from_csv

```python

def from_csv(
    path, # Path to put in `DataLoaders`
    csv_fname:str='train.csv', header:str='infer', delimiter:NoneType=None, quoting:int=0, valid_pct:float=0.2,
    seed:NoneType=None, fn_col:int=0, folder:NoneType=None, pref:NoneType=None, suff:str='', target_col:int=1,
    target_folder:NoneType=None, target_suff:str='', valid_col:NoneType=None, item_tfms:NoneType=None,
    batch_tfms:NoneType=None, img_cls:MetaResolver=BioImage, target_img_cls:MetaResolver=BioImage,
    show_summary:bool=False, # If True, print a summary of the BioDataBlock after creation.
    bs:int=64, # Size of batch
    val_bs:int=None, # Size of batch for validation `DataLoader`
    shuffle:bool=True, # Whether to shuffle data
    device:NoneType=None, # Device to put `DataLoaders`
):


```

*Create from `path/csv_fname` using `fn_col` and `target_col`*

In [ ]:
show_doc(BioDataLoaders.class_from_folder)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L3432){target="_blank" style="float:right; font-size:smaller"}

### class_from_folder

```python

def class_from_folder(
    path, # Path to put in `DataLoaders`
    train:str='train', valid:str='valid', valid_pct:NoneType=None, seed:NoneType=None, vocab:NoneType=None,
    item_tfms:NoneType=None, batch_tfms:NoneType=None, img_cls:MetaResolver=BioImage,
    show_summary:bool=False, # If True, print a summary of the BioDataBlock after creation.
    bs:int=64, # Size of batch
    val_bs:int=None, # Size of batch for validation `DataLoader`
    shuffle:bool=True, # Whether to shuffle data
    device:NoneType=None, # Device to put `DataLoaders`
):


```

*Create from dataset in `path` with `train` and `valid` subfolders (or provide `valid_pct`)*

In [ ]:
show_doc(BioDataLoaders.class_from_path_func)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L3449){target="_blank" style="float:right; font-size:smaller"}

### class_from_path_func

```python

def class_from_path_func(
    path, # Path to put in `DataLoaders`
    fnames, label_func, valid_pct:float=0.2, seed:NoneType=None, item_tfms:NoneType=None, batch_tfms:NoneType=None,
    img_cls:MetaResolver=BioImage,
    show_summary:bool=False, # If True, print a summary of the BioDataBlock after creation.
    bs:int=64, # Size of batch
    val_bs:int=None, # Size of batch for validation `DataLoader`
    shuffle:bool=True, # Whether to shuffle data
    device:NoneType=None, # Device to put `DataLoaders`
):


```

*Create from list of `fnames` in `path`s with `label_func`*

In [ ]:
show_doc(BioDataLoaders.class_from_path_re)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L3463){target="_blank" style="float:right; font-size:smaller"}

### class_from_path_re

```python

def class_from_path_re(
    path, # Path to put in `DataLoaders`
    fnames, pat, valid_pct:float=0.2, seed:NoneType=None, item_tfms:NoneType=None, batch_tfms:NoneType=None,
    img_cls:MetaResolver=BioImage,
    show_summary:bool=False, # If True, print a summary of the BioDataBlock after creation.
    bs:int=64, # Size of batch
    val_bs:int=None, # Size of batch for validation `DataLoader`
    shuffle:bool=True, # Whether to shuffle data
    device:NoneType=None, # Device to put `DataLoaders`
):


```

*Create from list of `fnames` in `path`s with re expression `pat`*

In [ ]:
show_doc(BioDataLoaders.class_from_df)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L3468){target="_blank" style="float:right; font-size:smaller"}

### class_from_df

```python

def class_from_df(
    df, path:str='.', # Path to put in `DataLoaders`
    valid_pct:float=0.2, seed:NoneType=None, fn_col:str='filename', folder:NoneType=None, suff:str='',
    label_col:str='label', label_delim:NoneType=None, y_block:NoneType=None, valid_col:NoneType=None,
    item_tfms:NoneType=None, batch_tfms:NoneType=None, img_cls:MetaResolver=BioImage,
    show_summary:bool=False, # If True, print a summary of the BioDataBlock after creation.
    bs:int=64, # Size of batch
    val_bs:int=None, # Size of batch for validation `DataLoader`
    shuffle:bool=True, # Whether to shuffle data
    device:NoneType=None, # Device to put `DataLoaders`
):


```

*Create from `df` using `fn_col` and `label_col`*

In [ ]:
show_doc(BioDataLoaders.class_from_csv)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L3498){target="_blank" style="float:right; font-size:smaller"}

### class_from_csv

```python

def class_from_csv(
    path, # Path to put in `DataLoaders`
    csv_fname:str='labels.csv', header:str='infer', delimiter:NoneType=None, quoting:int=0, valid_pct:float=0.2,
    seed:NoneType=None, fn_col:str='filename', folder:NoneType=None, suff:str='', label_col:str='label',
    label_delim:NoneType=None, y_block:NoneType=None, valid_col:NoneType=None, item_tfms:NoneType=None,
    batch_tfms:NoneType=None, img_cls:MetaResolver=BioImage,
    show_summary:bool=False, # If True, print a summary of the BioDataBlock after creation.
    bs:int=64, # Size of batch
    val_bs:int=None, # Size of batch for validation `DataLoader`
    shuffle:bool=True, # Whether to shuffle data
    device:NoneType=None, # Device to put `DataLoaders`
):


```

*Create from `path/csv_fname` using `fn_col` and `label_col`*

In [ ]:
show_doc(BioDataLoaders.class_from_lists)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L3504){target="_blank" style="float:right; font-size:smaller"}

### class_from_lists

```python

def class_from_lists(
    path, # Path to put in `DataLoaders`
    fnames, labels, valid_pct:float=0.2, seed:int=None, y_block:NoneType=None, item_tfms:NoneType=None,
    batch_tfms:NoneType=None, img_cls:MetaResolver=BioImage, fn_col:str='filename', folder:NoneType=None,
    suff:str='', label_col:str='label', label_delim:NoneType=None, valid_col:NoneType=None,
    show_summary:bool=False, # If True, print a summary of the BioDataBlock after creation.
    bs:int=64, # Size of batch
    val_bs:int=None, # Size of batch for validation `DataLoader`
    shuffle:bool=True, # Whether to shuffle data
    device:NoneType=None, # Device to put `DataLoaders`
):


```

*Create from list of `fnames` and `labels` in `path`*

#### Example: Using Fastai-style Factory Methods
BioDataLoaders provides incredibly convenient one-liners to build your 
DataLoaders directly from DataFrames, CSVs, or Folder structures.

In [ ]:
# --- 1. From a Pandas DataFrame ---
df = pd.DataFrame({
    "filename": ["scan_001.nii", "scan_002.nii"],
    "label": ["healthy", "tumor"],
    "is_valid": [False, True]
})

try:
    # Build a classification dataloader mapping the columns directly
    dls_df = BioDataLoaders.class_from_df(
        df,
        fn_col="filename",
        label_col="label",
        valid_col="is_valid",
        batch_size=8
    )
    print("DataLoaders built from DataFrame successfully!")
except Exception as e:
    # (Fails gracefully in documentation if the dummy files don't exist on disk)
    pass


# --- 2. From a Folder Structure ---
try:
    # Build a classification dataloader where the folder names are the labels
    # e.g., dataset/train/tumor/img1.nii
    dls_folder = BioDataLoaders.class_from_folder(
        path="path/to/dataset",
        train="train",
        valid="valid",
        batch_size=16
    )
    print("DataLoaders built from Folders successfully!")
except Exception as e:
    pass

In [ ]:
#| hide

# Verify that all factory methods correctly parse arguments and route to `from_source`

def test_factory_methods():
    dummy_df = pd.DataFrame({"filename": ["a", "b"], "label": [0, 1]})

    # --- Test 1: class_from_df routing ---
    with mockpatch.object(BioDataLoaders, 'from_source', return_value="MockDLS") as mock_from_source:
        
        dls = BioDataLoaders.class_from_df(dummy_df, fn_col="filename", label_col="label")

        test_eq(dls, "MockDLS")
        passed_data = mock_from_source.call_args[0][0]
        test_eq(passed_data.equals(dummy_df), True)
        
        ops_kwargs = mock_from_source.call_args[1]
        test_eq('blocks' in ops_kwargs, True)
        test_eq('get_x' in ops_kwargs, True)
        test_eq('get_y' in ops_kwargs, True)

    # --- Test 2: class_from_csv routing ---
    with mockpatch('pandas.read_csv', return_value=dummy_df) as mock_read_csv:
        with mockpatch.object(BioDataLoaders, 'class_from_df', return_value="MockCSV_DLS") as mock_class_from_df:
            
            dls_csv = BioDataLoaders.class_from_csv("dummy_dir", "dummy.csv")
            test_eq(dls_csv, "MockCSV_DLS")
            test_eq(mock_read_csv.called, True)

    return "All factory method routing tests passed"

test_eq(test_factory_methods(), "All factory method routing tests passed")

#### `BioDataLoaders.from_source` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`data_source`** | `Any` | *Required* | The source of the data to be loaded (paths, lists, datasets). |
| **`show_summary`** | `bool` | `False` | Print a summary of the underlying BioDataBlock after creation. |
| **`**kwargs`** | `dict` | `{}` | Additional arguments passed to configure the DataLoader and BioDataBlock (e.g., `blocks`, `item_tfms`, `get_x`). |

#### `BioDataLoaders.from_yaml` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`data_source`** | `Any` | *Required* | The source of the data (can be explicitly passed or parsed via YAML). |
| **`yaml_path`** | `str` | *Required* | Path to the YAML configuration file. |
| **`show_summary`** | `bool` | `False` | Print a summary of the underlying BioDataBlock after creation. |

#### `BioDataLoaders.from_folder` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`path`** | `str \| Path` | *Required* | Root directory containing the dataset. |
| **`get_target_fn`** | `Callable` | *Required* | Function to derive the target path/value from the input file. |
| **`train`** | `str` | `'train'` | Name of the training subfolder. |
| **`valid`** | `str` | `'valid'` | Name of the validation subfolder. |
| **`valid_pct`** | `float \| None` | `None` | If provided, ignores subfolders and randomly splits this percentage. |
| **`seed`** | `int \| None` | `None` | Random seed for splitting. |
| **`item_tfms`** / **`batch_tfms`** | `list \| None` | `None` | Item and batch-level transformations. |
| **`img_cls`** / **`target_img_cls`** | `class` | `BioImage` | Classes used to instantiate the inputs and targets. |
| **`get_items`** | `Callable \| None`| `None` | Custom function to gather files from the path. |
| **`**kwargs`** | `dict` | `{}` | Delegated to `from_source`. |

#### `BioDataLoaders.from_df` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`df`** | `pd.DataFrame` | *Required* | The Pandas DataFrame containing the dataset records. |
| **`path`** | `str \| Path` | `'.'` | Base path prepended to filenames. |
| **`valid_pct`** | `float` | `0.2` | Percentage of data to use for validation. |
| **`seed`** | `int \| None` | `None` | Random seed for splitting. |
| **`fn_col`** / **`target_col`** | `int \| str` | `0` / `1` | Column index or name for the inputs and targets. |
| **`folder`** / **`target_folder`** | `str \| None` | `None` | Subfolders to prepend specifically to input/target paths. |
| **`pref`** / **`suff`** | `str \| None` | `None` / `''`| Prefix and suffix strings for inputs. |
| **`valid_col`** | `str \| None` | `None` | Name of the boolean column indicating validation set membership. |
| **`img_cls`** / **`target_img_cls`** | `class` | `BioImage` | Classes used to instantiate inputs and targets. |
| **`**kwargs`** | `dict` | `{}` | Delegated to `from_source`. |

#### `BioDataLoaders.from_csv` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`path`** | `str \| Path` | *Required* | Directory containing the CSV file. |
| **`csv_fname`** | `str` | `'train.csv'` | Filename of the CSV. |
| **`header`** | `str` | `'infer'` | Pandas header configuration. |
| **`delimiter`** | `str \| None` | `None` | CSV delimiter character. |
| **`**kwargs`** | `dict` | `{}` | Delegated to `from_df`. |

#### `BioDataLoaders.class_from_folder` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`path`** | `str \| Path` | *Required* | Root directory containing the dataset. |
| **`train`** / **`valid`** | `str` | `'train'`/`'valid'`| Subfolder names for splitting. |
| **`valid_pct`** | `float \| None` | `None` | Override subfolders with a random validation split percentage. |
| **`vocab`** | `list \| None` | `None` | Explicit list of categories for the target block. |
| **`**kwargs`** | `dict` | `{}` | Delegated to `from_source`. |

#### `BioDataLoaders.class_from_path_func` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`path`** | `str \| Path` | *Required* | Root path containing the images. |
| **`fnames`** | `list` | *Required* | List of filenames to process. |
| **`label_func`** | `Callable` | *Required* | Function applied to the filename to extract its category label. |
| **`valid_pct`** | `float` | `0.2` | Validation split percentage. |
| **`**kwargs`** | `dict` | `{}` | Delegated to `from_source`. |

#### `BioDataLoaders.class_from_path_re` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`path`** | `str \| Path` | *Required* | Root path containing the images. |
| **`fnames`** | `list` | *Required* | List of filenames to process. |
| **`pat`** | `str` | *Required* | Regular expression pattern containing a capture group to extract the label. |
| **`**kwargs`** | `dict` | `{}` | Delegated to `class_from_path_func`. |

#### `BioDataLoaders.class_from_df` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`df`** | `pd.DataFrame` | *Required* | DataFrame containing inputs and labels. |
| **`fn_col`** | `int \| str` | `'filename'` | Column containing the input filenames. |
| **`label_col`** | `int \| str \| list` | `'label'` | Column(s) containing categorical labels. |
| **`label_delim`** | `str \| None` | `None` | Delimiter if multiple labels are contained in a single string. |
| **`y_block`** | `class \| None` | `None` | Target block class. Defaults to `CategoryBlock` (or `MultiCategoryBlock` if lists/delimiters are detected). |
| **`valid_col`** | `str \| None` | `None` | Boolean column for validation splitting. |
| **`**kwargs`** | `dict` | `{}` | Delegated to `from_source`. |

#### `BioDataLoaders.class_from_csv` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`path`** | `str \| Path` | *Required* | Directory containing the CSV file. |
| **`csv_fname`** | `str` | `'labels.csv'` | Filename of the CSV. |
| **`**kwargs`** | `dict` | `{}` | Delegated to `class_from_df`. |

#### `BioDataLoaders.class_from_lists` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`path`** | `str \| Path` | *Required* | Root directory for the input files. |
| **`fnames`** | `list` | *Required* | Python list containing the input filenames. |
| **`labels`** | `list` | *Required* | Python list containing the corresponding labels. |
| **`y_block`** | `class \| None` | `None` | Override the target block type (auto-inferred by default). |
| **`**kwargs`** | `dict` | `{}` | Delegated to `class_from_df`. |

### Loading MONAI Datasets

For users who prefer to build their datasets using MONAI's native ecosystem (such as `Dataset`, `CacheDataset`, or `PersistentDataset`), `BioDataLoaders` provides specialized integration methods. These factory methods act as a seamless bridge, taking your MONAI data structures and wrapping them into highly optimized fastai `DataLoaders` ready for the training loop.

* **`from_monai`**: This method takes your pre-built MONAI datasets and packages them. Crucially, it manages parameter routing automatically—scaling the validation batch size by default (e.g., `val_bs_factor=2`) and allowing you to override validation-specific arguments (like `num_workers` or `pin_memory`) simply by using the `val_` prefix.
* **`from_monai_ds`**: For an even more streamlined workflow, this method allows you to skip manually instantiating your MONAI datasets before creating your DataLoaders. You simply pass the uninstantiated dataset class alongside your raw data lists. It automatically instantiates both the training and validation datasets, intelligently merges your configuration dictionaries (`dataset_kwargs` and `val_dataset_kwargs`) to inherit baseline configurations, and seamlessly delegates the rest of the DataLoader construction to the standard `from_monai` method.

In [ ]:
#| export

def from_monai(
    cls,
    train_ds,            # MONAI training dataset
    val_ds=None,         # MONAI validation dataset
    x_keys="image",      # Key(s) used as model inputs
    y_keys="label",      # Key(s) used as targets
    bs=64,               # Training batch size
    val_bs=None,         # Validation batch size (overrides automatic scaling)
    val_bs_factor=2,     # Multiplier applied to bs when val_bs is None
    shuffle=True,        # Shuffle training dataset
    val_shuffle=False,   # Shuffle validation dataset (defaults to False)
    show_summary=False,  # Print basic dataloader summary
    vocab=None,          # Optional class names for classification tasks
    **dl_kwargs,         # Additional torch DataLoader kwargs (train + val_)
):
    """
    Create fastai-compatible `DataLoaders` from MONAI dictionary datasets.

    Parameters
    ----------
    train_ds : Dataset
        Training dataset (typically MONAI `Dataset`, `CacheDataset`, etc.)

    val_ds : Dataset, optional
        Validation dataset.

    x_keys : str or Sequence[str]
        Dictionary key(s) to extract as model inputs.

    y_keys : str or Sequence[str]
        Dictionary key(s) to extract as targets.

    bs : int
        Training batch size.

    val_bs : int, optional
        Validation batch size. If None, computed as `bs * val_bs_factor`.

    val_bs_factor : int
        Multiplier applied to training batch size for validation.

    shuffle : bool
        Whether to shuffle the training dataset.

    val_shuffle : bool
        Whether to shuffle the validation dataset. Defaults to False.

    show_summary : bool
        If True, prints number of batches in each dataloader.

    **dl_kwargs
        Additional arguments passed to `torch.utils.data.DataLoader`.

        Validation-specific arguments can be specified using the `val_`
        prefix.

        Example:

        - `num_workers=8`
        - `pin_memory=True`
        - `prefetch_factor=4`
        - `val_prefetch_factor=2`
    """

    # ---- wrap datasets ----
    train_ds = TupleDataset(train_ds, x_keys=x_keys, y_keys=y_keys)
    val_ds = TupleDataset(val_ds, x_keys=x_keys, y_keys=y_keys) if val_ds else None

    # ----patch datasets ----
    if vocab:
        train_ds = _patch_dataset(train_ds, vocab=vocab)
        val_ds = _patch_dataset(val_ds, vocab=vocab) if val_ds else None


    # ---- split train / val kwargs ----
    train_kwargs = {}
    val_kwargs = {}

    for k, v in dl_kwargs.items():
        if k.startswith("val_"):
            val_kwargs[k[4:]] = v
        else:
            train_kwargs[k] = v

    # ---- mirror train → val defaults ----
    for k, v in train_kwargs.items():
        val_kwargs.setdefault(k, v)

    # ---- enforce sensible validation defaults ----
    val_kwargs.setdefault("shuffle", val_shuffle)
    val_kwargs.setdefault("drop_last", False)

    # ---- base train args ----
    train_args = dict(
        dataset=train_ds,
        batch_size=bs,
        shuffle=shuffle,
    )
    train_args.update(train_kwargs)

    train_dl = torchDataLoader(**train_args)

    # ---- validation batch size scaling ----
    val_dl = None
    if val_ds is not None:

        if val_bs is None:
            val_bs = bs * val_bs_factor

        val_args = dict(
            dataset=val_ds,
            batch_size=val_bs,
        )
        val_args.update(val_kwargs)

        val_dl = torchDataLoader(**val_args)

    # ---- patch fastai compatibility ----
    train_dl = _patch_dataloader(train_dl)
    if val_dl is not None:
        val_dl = _patch_dataloader(val_dl)

    # ---- build fastai DataLoaders ----
    dls = cls(train_dl, val_dl)

    if show_summary:
        _show_summary(train_dl, val_dl)

    return dls

BioDataLoaders.from_monai = classmethod(from_monai)

In [ ]:
show_doc(BioDataLoaders.from_monai)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L3632){target="_blank" style="float:right; font-size:smaller"}

### from_monai

```python

def from_monai(
    train_ds, # MONAI training dataset
    val_ds:NoneType=None, # MONAI validation dataset
    x_keys:str='image', # Key(s) used as model inputs
    y_keys:str='label', # Key(s) used as targets
    bs:int=64, # Training batch size
    val_bs:NoneType=None, # Validation batch size (overrides automatic scaling)
    val_bs_factor:int=2, # Multiplier applied to bs when val_bs is None
    shuffle:bool=True, # Shuffle training dataset
    val_shuffle:bool=False, # Shuffle validation dataset (defaults to False)
    show_summary:bool=False, # Print basic dataloader summary
    vocab:NoneType=None, # Optional class names for classification tasks
    dl_kwargs:VAR_KEYWORD
):


```

*Create fastai-compatible `DataLoaders` from MONAI dictionary datasets.*

#### Example: Direct MONAI Dataset Integration
You can seamlessly wrap standard MONAI dictionary datasets (like CacheDataset) 
into fastai DataLoaders. The method automatically scales your validation batch size 
and routes specific arguments to the train or validation loaders.

In [ ]:
# 1. Create dummy MONAI-style data (lists of dictionaries)
dummy_train_ds = [{"image": "img1.nii", "label": 1}, {"image": "img2.nii", "label": 0}]
dummy_val_ds = [{"image": "img3.nii", "label": 1}]

try:
    # 2. Build the DataLoaders
    # Notice how we only specify 'bs=2'. Validation batch size automatically scales (2 * 2 = 4).
    # We also route 'num_workers=4' to training, but override it for validation with 'val_num_workers=1'.
    dls = BioDataLoaders.from_monai(
        train_ds=dummy_train_ds,
        val_ds=dummy_val_ds,
        bs=2,
        val_bs_factor=2,
        num_workers=4,
        val_num_workers=1,
        pin_memory=True # Applied to both train and validation
    )
    print("MONAI DataLoaders built successfully!")
except Exception as e:
    # Fails gracefully in the documentation if real internal torch/MONAI classes are missing
    pass

MONAI DataLoaders built successfully!


In [ ]:
#| hide

# We safely mock the internal dataset readers and torch dataloaders so we can 
# test the kwargs routing logic (like val_ prefixes and batch size scaling) 
# without triggering real PyTorch memory allocations or missing dependencies.

def test_from_monai_routing():
    dummy_train = [1, 2]
    dummy_val = [3]

    # Target globals that `from_monai` uses internally
    target_funcs = ['TupleDataset', 'torchDataLoader', '_patch_dataset', '_patch_dataloader', '_show_summary']
    orig_globals = {f: globals().get(f) for f in target_funcs}

    # Inject safe mock functions
    globals()['TupleDataset'] = lambda ds, **kwargs: ds
    globals()['_patch_dataset'] = lambda ds, **kwargs: ds
    globals()['_patch_dataloader'] = lambda dl, **kwargs: dl
    globals()['_show_summary'] = lambda t, v: None

    # Capture arguments passed to torchDataLoader
    captured_kwargs = []
    def mock_torchDataLoader(**kwargs):
        captured_kwargs.append(kwargs)
        return "MockDL"
    
    globals()['torchDataLoader'] = mock_torchDataLoader

    try:
        # Prevent the real BioDataLoaders.__init__ from firing with our fake string DataLoaders
        with mockpatch.object(BioDataLoaders, '__init__', return_value=None):
            
            dls = BioDataLoaders.from_monai(
                dummy_train,
                val_ds=dummy_val,
                bs=8,
                val_bs_factor=2,
                num_workers=4,
                val_num_workers=2,
                pin_memory=True # Should inherit to val
            )

            # 1. Verify Training Kwargs
            train_kwargs = captured_kwargs[0]
            test_eq(train_kwargs['batch_size'], 8)
            test_eq(train_kwargs['num_workers'], 4)
            test_eq(train_kwargs['pin_memory'], True)
            test_eq(train_kwargs['shuffle'], True)

            # 2. Verify Validation Kwargs
            val_kwargs = captured_kwargs[1]
            test_eq(val_kwargs['batch_size'], 16) # Automatically scaled (8 * 2)
            test_eq(val_kwargs['num_workers'], 2) # Explicitly overridden
            test_eq(val_kwargs['pin_memory'], True) # Inherited from train
            test_eq(val_kwargs['shuffle'], False) # Default validation behavior

    finally:
        # Restore the original global environment perfectly
        for f in target_funcs:
            if orig_globals[f] is not None:
                globals()[f] = orig_globals[f]
            else:
                globals().pop(f, None)

    return "from_monai routing tests passed"

test_eq(test_from_monai_routing(), "from_monai routing tests passed")

#### `BioDataLoaders.from_monai` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`train_ds`** | `Dataset` | *Required* | Instantiated MONAI training dataset (e.g., `Dataset`, `CacheDataset`). |
| **`val_ds`** | `Dataset \| None` | `None` | Instantiated MONAI validation dataset. |
| **`x_keys`** | `str \| list` | `"image"` | Dictionary key(s) to extract as model inputs. |
| **`y_keys`** | `str \| list` | `"label"` | Dictionary key(s) to extract as targets. |
| **`bs`** | `int` | `64` | Training batch size. |
| **`val_bs`** | `int \| None` | `None` | Explicit validation batch size. If `None`, it defaults to `bs * val_bs_factor`. |
| **`val_bs_factor`** | `int` | `2` | Multiplier applied to the training batch size to automatically calculate the validation batch size. |
| **`shuffle`** | `bool` | `True` | Shuffle the training DataLoader. |
| **`val_shuffle`** | `bool` | `False` | Shuffle the validation DataLoader. |
| **`show_summary`** | `bool` | `False` | Print the number of batches for each resulting DataLoader. |
| **`vocab`** | `list \| None` | `None` | Target class names for classification tasks, attached to the generated DataLoader. |
| **`**dl_kwargs`** | `dict` | `{}` | Additional PyTorch `DataLoader` arguments (e.g., `num_workers`, `pin_memory`). Use the `val_` prefix for validation-specific overrides. |

In [ ]:
#| export
def from_monai_ds(
    cls,
    dataset_cls,                 # MONAI dataset class (Dataset, CacheDataset, etc.)
    train_data,                  # Training datalist
    train_transform=None,        # Training transform pipeline
    val_data=None,               # Optional validation datalist
    val_transform=None,          # Validation transforms
    dataset_kwargs=None,         # Extra args for dataset constructor
    val_dataset_kwargs=None,     # Validation dataset overrides
    **dl_kwargs                  # Passed to `from_monai`
):
    """
    Build `BioDataLoaders` from any MONAI dataset class.
    """

    dataset_kwargs = dataset_kwargs or {}
    val_dataset_kwargs = val_dataset_kwargs or {}

    # ---- training dataset ----
    train_ds = dataset_cls(
        train_data,
        transform=train_transform,
        **dataset_kwargs
    )

    # ---- validation dataset ----
    val_ds = None
    if val_data is not None:
        val_ds = dataset_cls(
            val_data,
            transform=val_transform,
            **{**dataset_kwargs, **val_dataset_kwargs}
        )

    # ---- delegate to main loader ----
    return cls.from_monai(
        train_ds=train_ds,
        val_ds=val_ds,
        **dl_kwargs
    )

BioDataLoaders.from_monai_ds = classmethod(from_monai_ds)

In [ ]:
show_doc(BioDataLoaders.from_monai_ds)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L3765){target="_blank" style="float:right; font-size:smaller"}

### from_monai_ds

```python

def from_monai_ds(
    dataset_cls, # MONAI dataset class (Dataset, CacheDataset, etc.)
    train_data, # Training datalist
    train_transform:NoneType=None, # Training transform pipeline
    val_data:NoneType=None, # Optional validation datalist
    val_transform:NoneType=None, # Validation transforms
    dataset_kwargs:NoneType=None, # Extra args for dataset constructor
    val_dataset_kwargs:NoneType=None, # Validation dataset overrides
    dl_kwargs:VAR_KEYWORD
):


```

*Build `BioDataLoaders` from any MONAI dataset class.*

In [ ]:
# Example: Building DataLoaders directly from MONAI Dataset classes
# Instead of instantiating the datasets manually, you can pass the dataset class 
# (like MONAI's CacheDataset or PersistentDataset) along with your data lists.
# BioDataLoaders will instantiate them and pipe them into the dataloader engine.

# 1. Define dummy data lists
train_data = [{"image": "img1.nii", "label": 1}, {"image": "img2.nii", "label": 0}]
val_data = [{"image": "img3.nii", "label": 1}]

# 2. Define a mock MONAI Dataset class for this example
class MockMonaiDataset:
    def __init__(self, data, transform=None, **kwargs):
        self.data = data

try:
    # 3. Build the DataLoaders in one step
    dls = BioDataLoaders.from_monai_ds(
        dataset_cls=MockMonaiDataset,   # Pass the class itself
        train_data=train_data,
        val_data=val_data,
        dataset_kwargs={"cache_num": 10}, # Example: passing kwargs to the dataset constructor
        bs=4,                             # These arguments are routed seamlessly to `from_monai`
        val_bs_factor=2,
        num_workers=2
    )
    print("BioDataLoaders built from Dataset class successfully!")
except Exception as e:
    # Fails gracefully in the documentation if real internal torch/MONAI classes are missing
    pass

In [ ]:
#| hide

# Verify that `from_monai_ds` correctly instantiates the datasets, merges 
# the dictionary arguments properly, and delegates to `from_monai`.

def test_from_monai_ds_routing():
    dummy_train = [1, 2]
    dummy_val = [3]

    # Create a dummy class to inspect how it gets initialized
    class MockDataset:
        def __init__(self, data, transform=None, **kwargs):
            self.data = data
            self.transform = transform
            self.kwargs = kwargs

    # We patch `from_monai` to intercept the final call without triggering PyTorch
    with mockpatch.object(BioDataLoaders, 'from_monai', return_value="MockDLS") as mock_from_monai:
        
        dls = BioDataLoaders.from_monai_ds(
            dataset_cls=MockDataset,
            train_data=dummy_train,
            train_transform="train_tfms",
            val_data=dummy_val,
            val_transform="val_tfms",
            dataset_kwargs={"base_cache": True},
            val_dataset_kwargs={"val_override": True}, # Should merge with dataset_kwargs
            bs=16, # Should route to from_monai
            pin_memory=True
        )

        test_eq(dls, "MockDLS")
        test_eq(mock_from_monai.called, True)
        
        # Extract the arguments passed down to from_monai
        call_kwargs = mock_from_monai.call_args[1]
        train_ds = call_kwargs['train_ds']
        val_ds = call_kwargs['val_ds']
        
        # 1. Verify Train Dataset initialization
        test_eq(train_ds.data, dummy_train)
        test_eq(train_ds.transform, "train_tfms")
        test_eq(train_ds.kwargs['base_cache'], True)
        
        # 2. Verify Validation Dataset initialization and dictionary merging
        test_eq(val_ds.data, dummy_val)
        test_eq(val_ds.transform, "val_tfms")
        test_eq(val_ds.kwargs['base_cache'], True)    # Inherited from base kwargs
        test_eq(val_ds.kwargs['val_override'], True)  # Added by val overrides

        # 3. Verify standard DataLoaders kwargs passed through
        test_eq(call_kwargs['bs'], 16)
        test_eq(call_kwargs['pin_memory'], True)

    return "from_monai_ds routing tests passed"

test_eq(test_from_monai_ds_routing(), "from_monai_ds routing tests passed")

#### `BioDataLoaders.from_monai_ds` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`dataset_cls`** | `class` | *Required* | Uninstantiated MONAI dataset class (e.g., `Dataset`, `CacheDataset`). |
| **`train_data`** | `list` | *Required* | Datalist (list of dictionaries) for the training set. |
| **`train_transform`** | `Callable \| None` | `None` | Transformation pipeline applied to the training dataset. |
| **`val_data`** | `list \| None` | `None` | Datalist (list of dictionaries) for the validation set. |
| **`val_transform`** | `Callable \| None` | `None` | Transformation pipeline applied to the validation dataset. |
| **`dataset_kwargs`** | `dict \| None` | `None` | Extra initialization arguments passed to the dataset constructor. |
| **`val_dataset_kwargs`**| `dict \| None` | `None` | Explicit overrides for the validation dataset constructor (merges with `dataset_kwargs`). |
| **`**dl_kwargs`** | `dict` | `{}` | Standard DataLoader parameters delegated seamlessly to `from_monai`. |

### Test Datasets & Inference

This section provides robust utilities for generating fastai-compatible testing and inference `DataLoaders`. These tools are designed to effortlessly bridge various raw data formats and underlying dataset engines, ensuring your test environments perfectly inherit the configurations of your validation setup without manual boilerplate.

#### `_create_test_dl`
An internal utility that provides an automatic way to generate a testing `DataLoader` using a pre-existing `DataLoaders` object as a template. Rather than forcing you to redefine batch sizes, worker counts, or input keys, it intelligently inspects the validation dataloader inside your existing setup. It inherits those settings, automatically wraps your raw MONAI test dataset, applies the correct vocabulary mapping (if dealing with classification), and ensures that testing-specific flags (like `shuffle=False` and `drop_last=False`) are strictly enforced.

#### `test_biodataloader`
A versatile, universal routing utility designed to generate test/inference DataLoaders from almost any raw data source. Instead of requiring precisely formatted data, this function accepts a wide array of inputs: a Pandas `DataFrame`, a string path pointing to a `.csv` file, a directory path containing raw images, or an uninstantiated MONAI `Dataset`. It intelligently detects the input type, parses the data, and automatically delegates the construction process to either fastai's native `test_dl` method or our custom `_create_test_dl` MONAI wrapper.

In [ ]:
#| export
def _create_test_dl(
    test_ds,
    data,                 # existing fastai DataLoaders
    x_keys=None,
    y_keys=None,
    vocab=None,
    **dl_kwargs           # overrides for dataloader settings
):
    """
    Create a test DataLoader using validation DataLoader settings as defaults.

    Parameters
    ----------
    test_ds : Dataset
        MONAI test dataset.

    data : DataLoaders
        Existing fastai DataLoaders containing train/valid loaders.

    x_keys : str or sequence, optional
        Keys used as model inputs. Defaults to `data.valid.x_keys`.

    y_keys : str or sequence, optional
        Keys used as targets. Defaults to `data.valid.y_keys`.

    vocab : optional
        Vocabulary for classification tasks. Defaults to data.vocab if available.

    **dl_kwargs
        Explicit overrides for DataLoader parameters.

    Returns
    -------
    torch.utils.data.DataLoader
    """

    valid_dl = data.valid

    # ---- default keys from validation dataloader ----
    if x_keys is None:
        x_keys = getattr(valid_dl, "x_keys", "image")

    if y_keys is None:
        y_keys = getattr(valid_dl, "y_keys", "label")

    # ---- wrap dataset ----
    test_ds = TupleDataset(test_ds, x_keys=x_keys, y_keys=y_keys)

    # ---- vocab fallback ----
    if vocab is None and hasattr(data, "vocab"):
        vocab = data.vocab

    if vocab:
        test_ds = _patch_dataset(test_ds, vocab=vocab)

    # ---- copy validation dataloader settings ----
    base_kwargs = {
        "batch_size": valid_dl.batch_size,
        "num_workers": valid_dl.num_workers,
        "pin_memory": getattr(valid_dl, "pin_memory", False),
        "drop_last": False,
        "shuffle": False,
    }

    # optional attributes if present
    for attr in ["prefetch_factor", "persistent_workers"]:
        if hasattr(valid_dl, attr):
            base_kwargs[attr] = getattr(valid_dl, attr)

    # ---- allow explicit overrides ----
    base_kwargs.update(dl_kwargs)

    # ---- build dataloader ----
    test_dl = torchDataLoader(
        dataset=test_ds,
        **base_kwargs
    )

    # ---- fastai compatibility patch ----
    test_dl = _patch_dataloader(test_dl)

    return test_dl

In [ ]:
#| hide

# We safely mock the internal dataset readers and torch dataloaders so we can 
# test the inheritance logic without triggering real PyTorch memory allocations.

def test_create_test_dl_routing():
    # Target globals that `_create_test_dl` uses internally
    target_funcs = ['TupleDataset', 'torchDataLoader', '_patch_dataset', '_patch_dataloader']
    orig_globals = {f: globals().get(f) for f in target_funcs}

    # Inject safe mock classes and functions
    class MockTupleDataset:
        def __init__(self, ds, x_keys, y_keys):
            self.ds = ds
            self.x_keys = x_keys
            self.y_keys = y_keys
            
    globals()['TupleDataset'] = MockTupleDataset
    globals()['_patch_dataset'] = lambda ds, vocab: setattr(ds, 'vocab', vocab) or ds
    globals()['_patch_dataloader'] = lambda dl: dl

    # Capture arguments passed to torchDataLoader
    captured_kwargs = {}
    def mock_torchDataLoader(dataset, **kwargs):
        captured_kwargs.clear()
        captured_kwargs.update(kwargs)
        captured_kwargs['dataset'] = dataset
        return "MockTestDL"
    
    globals()['torchDataLoader'] = mock_torchDataLoader

    try:
        # Create fake objects to act as the existing fastai DataLoaders
        mock_valid_dl = SimpleNamespace(
            batch_size=32, 
            num_workers=4, 
            pin_memory=True, 
            x_keys="scan", 
            y_keys="mask",
            prefetch_factor=2
        )
        mock_dls = SimpleNamespace(valid=mock_valid_dl, vocab=["A", "B"])
        dummy_test_ds = [1, 2, 3]

        # --- 1. Test standard inheritance ---
        test_dl = _create_test_dl(dummy_test_ds, mock_dls)
        
        test_eq(test_dl, "MockTestDL")
        
        # Verify kwargs inherited from valid_dl
        test_eq(captured_kwargs['batch_size'], 32)
        test_eq(captured_kwargs['num_workers'], 4)
        test_eq(captured_kwargs['pin_memory'], True)
        test_eq(captured_kwargs['prefetch_factor'], 2)
        
        # Verify forced validation settings
        test_eq(captured_kwargs['drop_last'], False)
        test_eq(captured_kwargs['shuffle'], False)
        
        # Verify dataset wrapping and vocab patching
        wrapped_ds = captured_kwargs['dataset']
        test_eq(wrapped_ds.x_keys, "scan")
        test_eq(wrapped_ds.y_keys, "mask")
        test_eq(wrapped_ds.vocab, ["A", "B"])

        # --- 2. Test explicit overrides ---
        _create_test_dl(dummy_test_ds, mock_dls, batch_size=8, num_workers=1)
        test_eq(captured_kwargs['batch_size'], 8) # Successfully overridden
        test_eq(captured_kwargs['num_workers'], 1) # Successfully overridden

    finally:
        # Restore the original global environment perfectly
        for f in target_funcs:
            if orig_globals[f] is not None:
                globals()[f] = orig_globals[f]
            else:
                globals().pop(f, None)

    return "_create_test_dl routing tests passed"

test_eq(test_create_test_dl_routing(), "_create_test_dl routing tests passed")

In [ ]:
#| export
def test_biodataloader(dls:DataLoaders, 
                       test_data:str|Path|pd.DataFrame|MonaiDataset, 
                       with_labels=True, 
                       csv_header='infer', 
                       csv_delimiter=None, 
                       csv_quoting=0
                       ):
    """
    Create a test `DataLoader` from various data sources (DataFrame, CSV, Directory, or MONAI Dataset).
    
    Returns:
        A PyTorch/fastai DataLoader configured for testing/inference.
    """
    if isinstance(test_data, pd.DataFrame):
        # Handle DataFrame case directly
        test_dl = dls.test_dl(test_data, with_labels=with_labels)
    elif isinstance(test_data, (str, Path)):
        test_data = Path(test_data)
        # Check if it's a CSV file
        if test_data.suffix.lower() == '.csv':
            # Handle CSV file case
            df = pd.read_csv(test_data, header=csv_header, delimiter=csv_delimiter, quoting=csv_quoting)
            test_dl = dls.test_dl(df, with_labels=with_labels)
        else:
            # Handle non-CSV file case - get image files from directory
            test_dl = dls.test_dl(get_image_files(test_data), with_labels=with_labels)
    elif isinstance(test_data, MonaiDataset):
        test_dl = _create_test_dl(test_data, dls)
    
    return test_dl

In [ ]:
show_doc(test_biodataloader)

---

[source](https://github.com/deepCLEM/bonsai/blob/main/bonsai/data/load.py#L3893){target="_blank" style="float:right; font-size:smaller"}

### test_biodataloader

```python

def test_biodataloader(
    dls:DataLoaders, test_data:str | pathlib.Path | pandas.DataFrame | monai.data.dataset.Dataset,
    with_labels:bool=True, csv_header:str='infer', csv_delimiter:NoneType=None, csv_quoting:int=0
):


```

*Create a test `DataLoader` from various data sources (DataFrame, CSV, Directory, or MONAI Dataset).*

Returns:
    A PyTorch/fastai DataLoader configured for testing/inference.

#### Example: Generating a Test DataLoader dynamically
test_biodataloader acts as a universal router. You can pass a DataFrame, 
a CSV file path, an image directory, or a MONAI Dataset, and it will 
automatically build the correct test DataLoader using your existing DataLoaders.

In [ ]:
# 1. We use a mock DataLoaders object for the documentation example
mock_dls = MagicMock()
mock_dls.test_dl.return_value = "Ready-to-use Test DataLoader"

# 2. Define a simple test DataFrame
test_df = pd.DataFrame({"filename": ["test_01.nii", "test_02.nii"]})

try:
    # 3. Generate the test DataLoader directly from the DataFrame
    test_dl = test_biodataloader(
        dls=mock_dls,
        test_data=test_df,
        with_labels=False # Set to False if the test set lacks target columns
    )
    print(f"Result: {test_dl}")
except Exception as e:
    pass

Result: Ready-to-use Test DataLoader


In [ ]:
#| hide

def test_test_biodataloader():
    mock_dls = MagicMock()
    mock_dls.test_dl.return_value = "Fastai_Test_DL"
    
    # --- 1. Test DataFrame Routing ---
    dummy_df = pd.DataFrame({"col": [1, 2]})
    res_df = test_biodataloader(mock_dls, dummy_df)
    test_eq(res_df, "Fastai_Test_DL")
    
    # --- 2. Test CSV Path Routing ---
    with mockpatch('pandas.read_csv', return_value=dummy_df) as mock_read_csv:
        res_csv = test_biodataloader(mock_dls, "dummy_path.csv")
        test_eq(res_csv, "Fastai_Test_DL")
        test_eq(mock_read_csv.called, True)
        
    # --- 3. Test Directory Path Routing ---
    # Safely hijack get_image_files globally for this test
    orig_get_image_files = globals().get('get_image_files')
    globals()['get_image_files'] = lambda x: ["img1.png", "img2.png"]
    try:
        res_dir = test_biodataloader(mock_dls, "dummy_folder")
        test_eq(res_dir, "Fastai_Test_DL")
        test_eq(mock_dls.test_dl.call_args[0][0], ["img1.png", "img2.png"])
    finally:
        if orig_get_image_files: globals()['get_image_files'] = orig_get_image_files
        else: del globals()['get_image_files']
        
    # --- 4. Test MonaiDataset Routing ---
    orig_create_test_dl = globals().get('_create_test_dl')
    globals()['_create_test_dl'] = lambda ds, dls: "Monai_Test_DL"
    try:
        # We pass a dummy list [1] so the REAL MONAI Dataset doesn't crash 
        # due to the missing 'data' positional argument.
        dummy_monai = MonaiDataset([1])
        res_monai = test_biodataloader(mock_dls, dummy_monai)
        test_eq(res_monai, "Monai_Test_DL")
    finally:
        if orig_create_test_dl: globals()['_create_test_dl'] = orig_create_test_dl
        else: del globals()['_create_test_dl']
        
    return "test_biodataloader routing tests passed"

test_eq(test_test_biodataloader(), "test_biodataloader routing tests passed")

#### `test_biodataloader` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`dls`** | `DataLoaders` | *Required* | Your pre-existing, fully configured training/validation `DataLoaders` object. |
| **`test_data`** | `str \| Path \| pd.DataFrame \| MonaiDataset` | *Required* | The raw test data. Can be a DataFrame, CSV file path, directory of images, or a native MONAI Dataset. |
| **`with_labels`** | `bool` | `True` | Set to `False` if the test dataset does not contain target columns/labels. |
| **`csv_header`** | `str` | `'infer'` | Pandas header configuration (only used if `test_data` is a CSV path). |
| **`csv_delimiter`** | `str \| None` | `None` | Custom delimiter (only used if `test_data` is a CSV path). |
| **`csv_quoting`** | `int` | `0` | Pandas quoting behavior (only used if `test_data` is a CSV path). |

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()